# Fact Table: fact_production

## Objetive

Create the `fact_production` table by integrating monthly production records with the corresponding dimension tables. The table stores quantitative production measures together with the surrogate keys of each dimension, enabling efficient analytical queries within the star schema.

## Source Data

The fact_production table is built from the monthly production dataset and the previously created dimension tables.

### Input datasets

|Dataset|Description|
|-------|-----------|
|production_df|Monthly oil and gas production records for each well.|
|dim_time|Time dimension containing surrogate keys for year and month.|
|dim_well|Well dimension containing the surrogate key for each well.|
|dim_geology|Geological information associated with each well.|
|dim_field_area|Field and production area dimension.|
|dim_location|Geographical location dimension.|
|dim_company|Operating company dimension.|

## Grain

The grain of the fact table is defined as:

- One record represents the production of one well during one month.

Therefore, the natural business key is:

```text
(idpozo, anio, mes)
```

This granularity guarantees that each well has at most one production record for a given month.

## Measures

The fact table stores the following quantitative production measures:

|Measure|Description|
|-------|-----------|
|prod_pet|Oil production.|
|prod_gas|Gas production.|
|prod_agua|Water production.|
|iny_agua|Injected water volume.|
|tef|Effective production time.|

In addition, the table stores operational descriptive attributes:

|Attribute|Description|
|---------|-----------|
|tipoextraccion|Extraction method.|
|tipoestado|Well operational status.|
|tipopozo|Well type.|

## Foreign Keys

Each production record references every analytical dimension of the warehouse.

|Foreign Key|References|
|-----------|----------|
|time_key|dim_time|
|well_key|dim_well|
|geology_key|dim_geology|
|field_area_key|dim_field_area|
|location_key|dim_location|
|company_key|dim_company|

These surrogate keys ensure referential integrity and support multidimensional analysis.

## Target Schema

The target table is fact_production, whose structure was previously created in MySQL and includes:

- Primary key (production_key)
- Six foreign keys
- Production measures
- Operational attributes
- A unique constraint on (time_key, well_key) to prevent duplicate monthly production records for the same well.

## ETL Workflow

The ETL process follows the same methodology applied to the dimension tables:

```text
Extract
    ↓
Transform
    ↓
Lookup surrogate keys
    ↓
Build fact table
    ↓
Load into MySQL
    ↓
Validate data quality
```

## Helper Functions

The following helper functions are defined to simplify database interaction throughout the ETL process. They encapsulate the most common SQL operations, improve code readability, and promote code reuse across the notebook.

|Function|Description|
|--------|-----------|
|execute_sql()|Executes SQL statements that modify the database (CREATE, INSERT, UPDATE, DELETE, etc.) and commits the transaction.|
|execute_query()|Executes SQL queries and returns all retrieved records.|
|describe_table()|Returns the structure of a table as a Pandas DataFrame using the SQL DESCRIBE command.|
|show_create_table()|Returns the complete SQL definition used to create a table.|
|get_row_count()|Returns the total number of records stored in a table.|
|load_to_table()|Loads multiple records into a MySQL table using executemany() for efficient batch insertion.|
|read_query| Execute a SQL query and return the result as a DataFrame.|

Execute SQL statements that modify the database

In [1]:
def execute_sql(sql, message=None):
    try:
        cursor.execute(sql)
        conn.commit()
        if message:
            print(message)
    except mysql.connector.Error as err:
        print(f"Error: {err}")

Execute a SQL query and return all results

In [2]:
def execute_query(sql):
    cursor.execute(sql)
    return cursor.fetchall()

Display the structure of a database table

In [3]:
def describe_table(table_name):
    return pd.DataFrame(
        execute_query(f"DESCRIBE {table_name}"),
        columns=[
            "Field",
            "Type",
            "Null",
            "Key",
            "Default",
            "Extra"
        ]
    )

Return the CREATE TABLE statement

In [4]:
def show_create_table(table_name):
    return execute_query(f"SHOW CREATE TABLE {table_name}")[0][1]

Return the number of records in a table

In [5]:
def get_row_count(table_name):
    return execute_query(
        f"SELECT COUNT(*) FROM {table_name}"
    )[0][0]

Helper Functions for MySQL ETL Loading

In [6]:
def load_to_table(insert_sql, dataframe, table_name):
    try:

        records = [
            tuple(
                None if pd.isna(value) else value
                for value in row
            )
            for row in dataframe.itertuples(index=False, name=None)
        ]

        cursor.executemany(insert_sql, records)
        conn.commit()

        print(f"✓ {cursor.rowcount} records inserted into '{table_name}'.")

    except mysql.connector.Error as err:
        conn.rollback()
        print(f"Error loading '{table_name}': {err}")

In [7]:
def read_query(sql):
    return pd.read_sql(sql, conn)

## Implementation

### 1. Extract

#### Objective

Load the monthly production dataset that will be used to populate the fact_production table.

#### Code

In [262]:
import pandas as pd
import mysql.connector

Connect to database

In [ ]:

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="123456789"
)

cursor = conn.cursor(buffered=True)

cursor.execute("USE oil_gas_dw")
print("Database selected.")

Database selected.


Load production dataset

In [ ]:
production_df = pd.read_csv(
    "../data/raw/produccin-de-pozos-de-gas-y-petrleo-no-convencional.csv",
    low_memory=False
)

production_df = production_df.rename(columns={
    "sub_tipo_recurso": "subtipo_recurso"
})

print(f"Shape: {production_df.shape}")

production_df.head()

Shape: (415903, 40)


,idempresa,anio,mes,idpozo,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,iny_co2,...,cuenca,provincia,coordenadax,coordenaday,tipo_de_recurso,proyecto,clasificacion,subclasificacion,subtipo_recurso,fecha_data
0,YSUR,2015,1,132772,15.600,274.010,22.94,0.0,0.0,0.0,...,NEUQUINA,Neuquén,-69.232012,-38.862477,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2015-01-31
1,YSUR,2017,1,130300,1.710,48.130,0.00,0.0,0.0,0.0,...,NEUQUINA,Neuquén,-69.193650,-39.085770,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2017-01-31
2,YSUR,2015,1,130518,14.040,272.420,9.88,0.0,0.0,0.0,...,NEUQUINA,Neuquén,-69.201125,-39.085068,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2015-01-31
3,YSUR,2018,1,133429,1.049,148.467,1.23,0.0,0.0,0.0,...,NEUQUINA,Neuquén,-69.224719,-38.867785,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2018-01-31
4,YSUR,2015,1,143846,73.390,1009.070,43.77,0.0,0.0,0.0,...,NEUQUINA,Rio Negro,-67.834228,-39.030246,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2015-01-31


In [11]:
production_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415903 entries, 0 to 415902
Data columns (total 40 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   idempresa               415903 non-null  object 
 1   anio                    415903 non-null  int64  
 2   mes                     415903 non-null  int64  
 3   idpozo                  415903 non-null  int64  
 4   prod_pet                415903 non-null  float64
 5   prod_gas                415903 non-null  float64
 6   prod_agua               415903 non-null  float64
 7   iny_agua                415903 non-null  float64
 8   iny_gas                 415903 non-null  float64
 9   iny_co2                 415903 non-null  float64
 10  iny_otro                415903 non-null  float64
 11  tef                     415903 non-null  float64
 12  vida_util               8837 non-null    float64
 13  tipoextraccion          415293 non-null  object 
 14  tipoestado          

In [12]:
fact_production_df = production_df.copy()

Load fracture dataset

In [ ]:
fracture_path = "../data/raw/datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv"
fact_fracturing_df = pd.read_csv(fracture_path)

#### Validation

The production dataset was successfully loaded and inspected.

Findings:

- Rows: 415,903
- Columns: 40
- No missing values were found in the columns required to establish relationships with the dimension tables (idpozo, anio, mes, empresa, formacion, cuenca, provincia, etc.).
- Missing values are present in some descriptive and operational fields, such as:
    - vida_util
    - observaciones
    - tipoextraccion
    - tipoestado
    - tipopozo
    - clasificacion
    - subclasificacion
    - sub_tipo_recurso
- Numeric production measures (prod_pet, prod_gas, prod_agua, iny_agua, tef) are complete and suitable for loading into the fact table.

### 2. Transform

#### 2.1 Select Required Columns

|Purpose|Columns|
|-------|-------|
|Time|anio, mes|
|Well|idpozo|
|Company|empresa|
|Geology|formacion, cuenca, tipo_de_recurso, clasificacion, subclasificacion, sub_tipo_recurso|
|Field Area|idareapermisoconcesion, areapermisoconcesion, idareayacimiento, areayacimiento|
|Location|provincia, coordenadax, coordenaday|
|Measures|prod_pet, prod_gas, prod_agua, iny_agua, tef|
|Operational Attributes|tipoextraccion, tipoestado, tipopozo|

#### 2.2 Lookup time_key

Objective

Match each production record with the corresponding surrogate key from the `dim_time` dimension by using the natural business key (`anio`, `mes`).

#### Implementation

Step 1. Load the time dimension

In [14]:
sql = """
SELECT
    time_key,
    full_date
FROM dim_time
"""

dim_time = read_query(sql)
dim_time.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,time_key,full_date
0,1,2006-01-31
1,2,2006-02-28
2,3,2006-03-31
3,4,2006-04-30
4,5,2006-05-19


Step 2. Ensure matching data types

In [15]:
fact_production_df["fecha_data"] = pd.to_datetime(
    fact_production_df["fecha_data"]
)

dim_time["full_date"] = pd.to_datetime(
    dim_time["full_date"]
)

Step 3. Merge with the time dimension

In [16]:
fact_production_df = fact_production_df.merge(
    dim_time,
    left_on="fecha_data",
    right_on="full_date",
    how="left"
)

In [17]:
print(f"Missing time_key: {fact_production_df['time_key'].isna().sum()}")

Missing time_key: 0


Step 4. Remove auxiliary column

In [18]:
fact_production_df.drop(
    columns=["full_date"], 
    inplace=True
)

In [19]:
fact_production_df[
    ["fecha_data", "time_key"]
].head()

,fecha_data,time_key
0,2015-01-31,1135
1,2017-01-31,1693
2,2015-01-31,1135
3,2018-01-31,1973
4,2015-01-31,1135


In [20]:
print(f"Rows: {fact_production_df.shape[0]}")

Rows: 415903


#### 2.3 Lookup well_key

Objective

Match each production record with the corresponding surrogate key from the dim_well dimension using the natural business key (idpozo).

#### Implementation

Step 1. Load the well dimension

In [21]:
sql = """
SELECT
    well_key,
    idpozo
FROM dim_well;
"""

dim_well = read_query(sql)

dim_well.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,well_key,idpozo
0,11080,458
1,11119,460
2,8727,3640
3,11242,5984
4,11292,5990


Step 2. Ensure matching data types

In [22]:
fact_production_df["idpozo"] = fact_production_df["idpozo"].astype(int)
dim_well["idpozo"] = dim_well["idpozo"].astype(int)

Step 3. Merge with the well dimension

In [23]:
fact_production_df = fact_production_df.merge(
    dim_well,
    on="idpozo",
    how="left"
)

Step 4. Validate the lookup

In [24]:
print(f"Missing well_key: {fact_production_df['well_key'].isna().sum()}")

Missing well_key: 0


#### 2.4 Lookup geology_key

Objective

Match each production record with the corresponding surrogate key from the dim_geology dimension by combining geological information from the production and fracturing datasets.

#### Implementation

Step 1. Extract geological information from production

In [25]:
production_geo = production_df[
    [
        "idpozo",
        "formacion",
        "subtipo_recurso"
    ]
].copy()

Step 2. Load dim_geology for lookup

In [26]:
dim_geology = read_query("""
    SELECT
        geology_key,
        formacion,
        subtipo_recurso,
        tipo_reservorio,
        subtipo_reservorio,
        formacion_productiva
    FROM dim_geology
""")

print(f"Rows in dim_geology: {dim_geology.shape[0]}")
print(f"Columns in dim_geology: {dim_geology.shape[1]}")

Rows in dim_geology: 113
Columns in dim_geology: 6


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


Step 3. Build one geological profile per production well

In [27]:
production_geo = production_df[
    [
        "idpozo",
        "formacion",
        "subtipo_recurso"
    ]
].copy()

production_geo = (
    production_geo
    .drop_duplicates(subset=["idpozo"])
    .reset_index(drop=True)
)


Step 4. Build one geological profile per fracturing well

In [28]:
fracture_geo = fact_fracturing_df[
    [
        "idpozo",
        "tipo_reservorio",
        "subtipo_reservorio",
        "formacion_productiva"
    ]
].copy()

fracture_geo = (
    fracture_geo
    .drop_duplicates(subset=["idpozo"])
    .reset_index(drop=True)
)

print(f"Rows: {fracture_geo.shape[0]}")
print(f"Unique wells: {fracture_geo['idpozo'].nunique()}")
print(f"Duplicated wells: {fracture_geo['idpozo'].duplicated().sum()}")

Rows: 4562
Unique wells: 4562
Duplicated wells: 0


Step 5. Combine production and fracturing geological information

In [29]:
geology_by_well = production_geo.merge(
    fracture_geo,
    on="idpozo",
    how="left"
)

print(f"Rows: {geology_by_well.shape[0]}")
print(f"Unique wells: {geology_by_well['idpozo'].nunique()}")
print(f"Columns: {geology_by_well.shape[1]}")

Rows: 5041
Unique wells: 5041
Columns: 6


Step 6. Build comparable geological profiles

In [30]:
geology_columns = [
    "formacion",
    "subtipo_recurso",
    "tipo_reservorio",
    "subtipo_reservorio",
    "formacion_productiva"
]

geology_by_well["geology_profile"] = (
    geology_by_well[geology_columns]
    .fillna("<NULL>")
    .astype(str)
    .agg("|".join, axis=1)
)

dim_geology["geology_profile"] = (
    dim_geology[geology_columns]
    .fillna("<NULL>")
    .astype(str)
    .agg("|".join, axis=1)
)

print(f"Production geological profiles: {geology_by_well['geology_profile'].nunique()}")
print(f"Dimension geological profiles: {dim_geology['geology_profile'].nunique()}")

geology_by_well[
    ["idpozo", "geology_profile"]
].head()

Production geological profiles: 58
Dimension geological profiles: 113


,idpozo,geology_profile
0,132772,basamento|TIGHT|<NULL>|<NULL>|<NULL>
1,130300,precuyo|TIGHT|<NULL>|<NULL>|<NULL>
2,130518,precuyo|TIGHT|NO CONVENCIONAL|TIGHT|precuyo
3,133429,precuyo|TIGHT|<NULL>|<NULL>|<NULL>
4,143846,lajas|TIGHT|<NULL>|<NULL>|<NULL>


Step 7. Validate production profiles against dim_geology

In [31]:
production_profiles = set(
    geology_by_well["geology_profile"]
)

dimension_profiles = set(
    dim_geology["geology_profile"]
)

unmatched_profiles = production_profiles - dimension_profiles

print(f"Production profiles: {len(production_profiles)}")
print(f"Dimension profiles: {len(dimension_profiles)}")
print(f"Unmatched profiles: {len(unmatched_profiles)}")

if unmatched_profiles:
    print("\nUnmatched geological profiles:")
    for profile in sorted(unmatched_profiles):
        print(profile)

Production profiles: 58
Dimension profiles: 113
Unmatched profiles: 0


Step 8. Lookup geology_key

In [32]:
geology_lookup = dim_geology[
    [
        "geology_profile",
        "geology_key"
    ]
].copy()

geology_by_well = geology_by_well.merge(
    geology_lookup,
    on="geology_profile",
    how="left"
)

print(f"Rows: {geology_by_well.shape[0]}")
print(f"Unique wells: {geology_by_well['idpozo'].nunique()}")
print(f"Missing geology_key: "f"{geology_by_well['geology_key'].isna().sum()}")

geology_by_well[
    [
        "idpozo",
        "geology_profile",
        "geology_key"
    ]
].head()

Rows: 5041
Unique wells: 5041
Missing geology_key: 0


,idpozo,geology_profile,geology_key
0,132772,basamento|TIGHT|<NULL>|<NULL>|<NULL>,1
1,130300,precuyo|TIGHT|<NULL>|<NULL>|<NULL>,2
2,130518,precuyo|TIGHT|NO CONVENCIONAL|TIGHT|precuyo,3
3,133429,precuyo|TIGHT|<NULL>|<NULL>|<NULL>,2
4,143846,lajas|TIGHT|<NULL>|<NULL>|<NULL>,4


Step 9. Add geology_key to fact_production_df

In [33]:
geology_key_lookup = geology_by_well[
    [
        "idpozo",
        "geology_key"
    ]
].copy()

fact_production_df = fact_production_df.merge(
    geology_key_lookup,
    on="idpozo",
    how="left"
)

print(f"Rows: {fact_production_df.shape[0]}")
print(f"Columns: {fact_production_df.shape[1]}")
print(f"Missing geology_key: "f"{fact_production_df['geology_key'].isna().sum()}")

fact_production_df[
    [
        "idpozo",
        "geology_key"
    ]
].head()

Rows: 415903
Columns: 43
Missing geology_key: 0


,idpozo,geology_key
0,132772,1
1,130300,2
2,130518,3
3,133429,2
4,143846,4


Step 10. Validate fact grain after geology lookup

In [34]:
production_rows_before = len(production_df)
production_rows_after = len(fact_production_df)

print(f"Production rows before lookup: {production_rows_before}")
print(f"Production rows after lookup:  {production_rows_after}")
print(f"Rows preserved: {production_rows_before == production_rows_after}")

print(f"Unique production wells: "f"{fact_production_df['idpozo'].nunique()}")

print(f"Missing geology_key: "f"{fact_production_df['geology_key'].isna().sum()}")

Production rows before lookup: 415903
Production rows after lookup:  415903
Rows preserved: True
Unique production wells: 5041
Missing geology_key: 0


#### 2.5 Lookup field_area_key

Objective

Retrieve the surrogate key (field_area_key) from dim_field_area by matching the field and concession area attributes associated with each production record. Since dim_field_area was built by integrating information from both the production and hydraulic fracturing datasets, the production data must first be enriched with the corresponding yacimiento attribute before performing the lookup.

#### Implementation

Step 1. Select required field area attributes

In [35]:
field_area_lookup_df = fact_fracturing_df[
    [
        "idpozo",
        "yacimiento"
    ]
].copy()

Step 2. Validate duplicated wells

In [36]:
field_area_lookup_df["idpozo"].duplicated().sum()

np.int64(244)

Step 3. Inspect duplicated wells

In [37]:
duplicated_wells = (
    field_area_lookup_df[
        field_area_lookup_df["idpozo"].duplicated(keep=False)
    ].sort_values("idpozo")
)

duplicated_wells.head(20)

,idpozo,yacimiento
1926,6051,EL TORDILLO
1927,6051,EL TORDILLO
1948,6187,EL TORDILLO
1949,6187,EL TORDILLO
1924,6316,EL TORDILLO
1925,6316,EL TORDILLO
1923,7010,EL TORDILLO
1922,7010,EL TORDILLO
1909,8322,PUESTO QUIROGA OESTE
1919,8322,PUESTO QUIROGA OESTE


Step 4. Validate consistency of duplicated wells

In [38]:
duplicated_wells.groupby("idpozo")["yacimiento"].nunique().max()

np.int64(1)

Step 5. Remove duplicated field area records

In [39]:
field_area_lookup_df = field_area_lookup_df.drop_duplicates(
    subset=["idpozo"]
)

In [40]:
field_area_lookup_df["idpozo"].duplicated().sum()

np.int64(0)

Step 6. Merge production with field area information

In [41]:
fact_production_df = fact_production_df.merge(
    field_area_lookup_df,
    on="idpozo",
    how="left"
)

In [42]:
fact_production_df["yacimiento"].isna().sum()

np.int64(142273)

Step 7. Load the field area dimension

In [43]:
sql = """
SELECT
    field_area_key,
    idareapermisoconcesion,
    areapermisoconcesion,
    idareayacimiento,
    areayacimiento,
    yacimiento
FROM dim_field_area;
"""

dim_field_area = read_query(sql)

dim_field_area.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,field_area_key,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,yacimiento
0,1,None,None,None,None,ESTANCIA LA MARIPOSA
1,2,ELO,ENTRE LOMAS,ECL,EL CARACOL,None
2,3,None,None,None,None,EL TORDILLO
3,4,FOR,FORTIN DE PIEDRA,FOR,FORTIN DE PIEDRA,None
4,5,None,None,None,None,LA TAPERA


Step 8. Validate duplicated natural keys

In [44]:
dim_field_area.duplicated(
    subset=[
        "idareapermisoconcesion",
        "areapermisoconcesion",
        "idareayacimiento",
        "areayacimiento",
        "yacimiento"
    ]
).sum()

np.int64(0)

In [45]:
dim_field_area.groupby(
    [
        "idareapermisoconcesion",
        "areapermisoconcesion",
        "idareayacimiento",
        "areayacimiento",
        "yacimiento"
    ]
).size().sort_values(ascending=False).head(10)

idareapermisoconcesion  areapermisoconcesion                 idareayacimiento  areayacimiento                   yacimiento                     
AFED                    AGUADA FEDERAL                       AGFD              AGUADA FEDERAL                   AGUADA FEDERAL                     1
AGA                     AGUADA DE LA ARENA                   AGA               AGUADA DE LA ARENA               AGUADA DE LA ARENA                 1
AGCA                    AGUADA DE CASTRO                     AGCA              AGUADA DE CASTRO                 AGUADA DE CASTRO                   1
AGCO                    AGUADA CÁNEPA                        ADCO              AGUADA CÁNEPA                    AGUADA CÁNEPA                      1
AGMO                    AGUILA MORA                          AMOR              AGUILA MORA                      AGUILA MORA                        1
AGUH                    AGUADA DEL CHAÑAR                    LCOL              LOMA COLORADA                   

Step 9. Ensure matching data types

In [46]:
lookup_columns = [
    "idareapermisoconcesion",
    "areapermisoconcesion",
    "idareayacimiento",
    "areayacimiento",
    "yacimiento"
]

Step 10. Merge with the field area dimension

In [47]:
fact_production_df = fact_production_df.merge(
    dim_field_area,
    on=[
        "idareapermisoconcesion",
        "areapermisoconcesion",
        "idareayacimiento",
        "areayacimiento",
        "yacimiento"
    ],
    how="left"
)

In [48]:
print("Rows before merge:", production_rows_before)
print("Rows after merge:", fact_production_df.shape[0])
print("Rows preserved:", production_rows_before == fact_production_df.shape[0])

Rows before merge: 415903
Rows after merge: 415903
Rows preserved: True


In [49]:
print(
    "Duplicated production records:",
    fact_production_df.duplicated(
        subset=["idpozo", "anio", "mes"]
    ).sum()
)

Duplicated production records: 0


Step 11. Validate field_area_key

In [50]:
print(
    "Missing field_area_key:",
    fact_production_df["field_area_key"].isna().sum()
)

Missing field_area_key: 0


#### 2.6 Lookup location_key

Objective

Retrieve the surrogate key (location_key) from dim_location by matching the geographical attributes associated with each production record.

#### Implementation

Step 1. Load the location dimension

In [51]:
sql = """
SELECT
    location_key,
    cuenca,
    provincia,
    coordenadax,
    coordenaday
FROM dim_location;
"""

dim_location = read_query(sql)
dim_location.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,location_key,cuenca,provincia,coordenadax,coordenaday
0,1,NEUQUINA,Neuquén,-69.232012,-38.862477
1,2,NEUQUINA,Neuquén,-69.193650,-39.085770
2,3,NEUQUINA,Neuquén,-69.201125,-39.085068
3,4,NEUQUINA,Neuquén,-69.224719,-38.867785
4,5,NEUQUINA,Rio Negro,-67.834228,-39.030246


Step 2. Validate duplicated natural keys

In [52]:
dim_location.duplicated(
    subset=[
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday"
    ]
).sum()

np.int64(0)

In [53]:
dim_location.groupby(
    [
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday"
    ]
).size().sort_values(ascending=False).head(10)

cuenca   provincia   coordenadax  coordenaday
AUSTRAL  Santa Cruz  -71.99831    -51.30810      1
                     -71.36139    -50.54743      1
                     -71.35721    -50.60085      1
                     -71.35280    -50.54612      1
                     -71.35187    -50.54975      1
                     -71.34767    -50.60875      1
                     -71.34421    -50.59942      1
                     -71.34095    -50.54320      1
                     -71.33861    -50.53196      1
                     -71.33747    -50.55538      1
dtype: int64

Step 3. Standardize coordinate precision

In [54]:
fact_production_df["coordenadax"] = (
    fact_production_df["coordenadax"]
    .round(8)
)

fact_production_df["coordenaday"] = (
    fact_production_df["coordenaday"]
    .round(8)
)

Step 4. Ensure matching data types

In [55]:
lookup_columns = [
    "cuenca",
    "provincia",
    "coordenadax",
    "coordenaday"
]

for col in lookup_columns:
    fact_production_df[col] = fact_production_df[col].astype(
        dim_location[col].dtype
    )

Step 5. Merge with the location dimension

In [56]:
fact_production_df = fact_production_df.merge(
    dim_location,
    on=[
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday"
    ],
    how="left"
)

Step 6. Validate location_key

In [57]:
print(
    f"Missing location_key: "
    f"{fact_production_df['location_key'].isna().sum()}"
)

print(
    f"Rows: "
    f"{fact_production_df.shape[0]}"
)

Missing location_key: 0
Rows: 415903


#### 2.7 Lookup company_key

Objective

Retrieve the surrogate key (company_key) from dim_company by matching the company attributes associated with each production record.

#### Implementation

Step 1. Load the company dimension

In [58]:
sql = """
SELECT
    company_key,
    idempresa,
    empresa
FROM dim_company;
"""

dim_company = read_query(sql)
dim_company.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,company_key,idempresa,empresa
0,1,YSUR,YSUR ENERGÍA ARGENTINA S.R.L.
1,2,YPF,YPF S.A.
2,3,WIN,WINTERSHALL ENERGIA S.A.
3,4,WDA,WINTERSHALL DEA ARGENTINA S.A
4,5,VST,VISTA ENERGY ARGENTINA SAU


Step 2. Validate company natural key uniqueness

In [59]:
print(
    f"Duplicated idempresa: "
    f"{dim_company['idempresa'].duplicated().sum()}"
)

print(
    f"Duplicated (idempresa, empresa): "
    f"{dim_company[['idempresa', 'empresa']].duplicated().sum()}"
)

print("-" * 35)

print("Company dimension sample:")
print(
    dim_company[
        ["company_key", "idempresa", "empresa"]
    ].head(10)
)

Duplicated idempresa: 0
Duplicated (idempresa, empresa): 0
-----------------------------------
Company dimension sample:
   company_key idempresa                        empresa
0            1      YSUR  YSUR ENERGÍA ARGENTINA S.R.L.
1            2       YPF                       YPF S.A.
2            3       WIN       WINTERSHALL ENERGIA S.A.
3            4       WDA  WINTERSHALL DEA ARGENTINA S.A
4            5       VST     VISTA ENERGY ARGENTINA SAU
5            6       VOG   Vista Oil & Gas Argentina SA
6            7       VNO                    VENOIL S.A.
7            8       VMI    VACA MUERTA INVERSIONES SAU
8            9       VIS  VISTA OIL & GAS ARGENTINA SAU
9           10       TPT                 TECPETROL S.A.


Step 3. Ensure matching data type for company natural key

In [60]:
fact_production_df["idempresa"] = fact_production_df[
    "idempresa"
].astype(
    dim_company["idempresa"].dtype
)

Step 4. Merge with the company dimension

In [61]:
fact_production_df = fact_production_df.merge(
    dim_company[
        [
            "idempresa",
            "company_key"
        ]
    ],
    on="idempresa",
    how="left"
)

Step 5. Validate company_key lookup

In [62]:
missing_company_keys = (
    fact_production_df["company_key"]
    .isna()
    .sum()
)

production_company_ids = set(
    fact_production_df["idempresa"]
    .dropna()
    .unique()
)

dimension_company_ids = set(
    dim_company["idempresa"]
    .dropna()
    .unique()
)

unmatched_company_ids = sorted(
    production_company_ids
    - dimension_company_ids
)

duplicated_dimension_ids = (
    dim_company["idempresa"]
    .duplicated()
    .sum()
)

print("===================================")
print("COMPANY KEY VALIDATION")
print("===================================")

print(
    f"Missing company_key: "
    f"{missing_company_keys}"
)

print(
    f"Rows: "
    f"{fact_production_df.shape[0]}"
)

print(
    f"Unique production idempresa: "
    f"{len(production_company_ids)}"
)

print(
    f"Unique dimension idempresa: "
    f"{len(dimension_company_ids)}"
)

print(
    f"Unmatched idempresa: "
    f"{len(unmatched_company_ids)}"
)

print(
    f"Duplicated dimension idempresa: "
    f"{duplicated_dimension_ids}"
)

print()
print("Unmatched idempresa:")
print(unmatched_company_ids)

COMPANY KEY VALIDATION
Missing company_key: 0
Rows: 415903
Unique production idempresa: 55
Unique dimension idempresa: 57
Unmatched idempresa: 0
Duplicated dimension idempresa: 0

Unmatched idempresa:
[]


Step 6. Validate company descriptive consistency

In [63]:
company_name_lookup = (
    dim_company
    .set_index("idempresa")["empresa"]
)

company_name_validation = fact_production_df[
    ["idempresa", "empresa"]
].copy()

company_name_validation["dim_empresa"] = (
    company_name_validation["idempresa"]
    .map(company_name_lookup)
)

company_name_mismatches = company_name_validation[
    company_name_validation["empresa"]
    != company_name_validation["dim_empresa"]
]

mismatch_profiles = (
    company_name_mismatches[
        ["idempresa", "empresa", "dim_empresa"]
    ]
    .drop_duplicates()
    .sort_values(["idempresa", "empresa"])
)

print("===================================")
print("COMPANY NAME CONSISTENCY VALIDATION")
print("===================================")

print(
    f"Rows checked: "
    f"{company_name_validation.shape[0]}"
)

print(
    f"Rows with company name mismatch: "
    f"{company_name_mismatches.shape[0]}"
)

print(
    f"Unique idempresa with mismatch: "
    f"{company_name_mismatches['idempresa'].nunique()}"
)

print(
    f"Mismatch profiles: "
    f"{mismatch_profiles.shape[0]}"
)

print()
print("Company name mismatches:")
print(mismatch_profiles)

COMPANY NAME CONSISTENCY VALIDATION
Rows checked: 415903
Rows with company name mismatch: 0
Unique idempresa with mismatch: 0
Mismatch profiles: 0

Company name mismatches:
Empty DataFrame
Columns: [idempresa, empresa, dim_empresa]
Index: []


### 3. Prepare the Fact Production Table

#### Objective

Prepare the final production fact table by retaining only the surrogate keys, measures, and operational attributes defined in the dimensional model, ensuring that the dataset matches the structure of fact_production before loading it into the data warehouse.

#### Source Data

- fact_production_df
- fact_production (target table schema)

#### Transformation

The fact table is prepared by removing all source-system attributes that were only required to perform the dimensional lookups. The resulting dataset preserves the original granularity while matching the target schema of the data warehouse.

#### 3.1 Select Final Columns

Step 1. Select final columns

In [64]:
production_well_geology_lookup = (
    fact_production_df[
        [
            "idpozo",
            "geology_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Rows: "f"{production_well_geology_lookup.shape[0]}")

print(f"Unique wells: "f"{production_well_geology_lookup['idpozo'].nunique()}")

print(f"Duplicated wells: "f"{production_well_geology_lookup['idpozo'].duplicated().sum()}")

print(f"Missing geology_key: "f"{production_well_geology_lookup['geology_key'].isna().sum()}")

production_well_geology_lookup.head()

Rows: 5041
Unique wells: 5041
Duplicated wells: 0
Missing geology_key: 0


,idpozo,geology_key
0,132772,1
1,130300,2
2,130518,3
3,133429,2
4,143846,4


In [65]:
fact_production_df = fact_production_df[
    [
        "time_key",
        "well_key",
        "geology_key",
        "field_area_key",
        "location_key",
        "company_key",
        "prod_pet",
        "prod_gas",
        "prod_agua",
        "iny_agua",
        "tef",
        "tipoextraccion",
        "tipoestado",
        "tipopozo"
    ]
].copy()

#### 3.2 Reorder Columns

In [66]:
fact_production_df = fact_production_df[
    [
        "time_key",
        "well_key",
        "geology_key",
        "field_area_key",
        "location_key",
        "company_key",
        "prod_pet",
        "prod_gas",
        "prod_agua",
        "iny_agua",
        "tef",
        "tipoextraccion",
        "tipoestado",
        "tipopozo"
    ]
]

#### 3.3 Validate Final Structure

Step 2. Validate the final structure

In [67]:
print(f"Rows: {fact_production_df.shape[0]}")
print(f"Columns: {fact_production_df.shape[1]}")

fact_production_df.columns.tolist()

Rows: 415903
Columns: 14


['time_key',
 'well_key',
 'geology_key',
 'field_area_key',
 'location_key',
 'company_key',
 'prod_pet',
 'prod_gas',
 'prod_agua',
 'iny_agua',
 'tef',
 'tipoextraccion',
 'tipoestado',
 'tipopozo']

Validation

The final production fact dataset matches the target schema defined for the data warehouse.

All dimensional lookups have been completed successfully, resulting in a dataset containing only surrogate keys, quantitative measures, and operational attributes.

The final structure preserves the original granularity of the production data and is ready for loading into the fact_production table.

#### 3.4 Load into MySQL

Objective

Load the transformed production fact table into the fact_production table while preserving referential integrity and the original data granularity.

Source Data
- fact_production_df

Transformation

No additional transformations are required. The validated dataset is inserted into the target fact table.

Step 1. Verify the target table is empty

In [68]:
print(f"Rows before load: {get_row_count('fact_production')}")

Rows before load: 415903


Step 2. Prepare INSERT statement

In [69]:
insert_sql = """
INSERT INTO fact_production(
    time_key,
    well_key,
    geology_key,
    field_area_key,
    location_key,
    company_key,
    prod_pet,
    prod_gas,
    prod_agua,
    iny_agua,
    tef,
    tipoextraccion,
    tipoestado,
    tipopozo
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

Step 3. Load data into MySQL

In [ ]:
load_to_table(insert_sql, fact_production_df, "fact_production")

In [71]:
production_geo_check = production_df[
    [
        "formacion",
        "subtipo_recurso"
    ]
].drop_duplicates()

geo_check = production_geo_check.merge(
    dim_geology[
        [
            "formacion",
            "subtipo_recurso",
            "geology_key"
        ]
    ],
    on=[
        "formacion",
        "subtipo_recurso"
    ],
    how="left"
)

print(f"Total combinations: {geo_check.shape[0]}")
print(f"Missing geology_key: {geo_check['geology_key'].isna().sum()}")

geo_check[
    geo_check["geology_key"].isna()
]

Total combinations: 58
Missing geology_key: 0


,formacion,subtipo_recurso,geology_key


#### 3.5 Validate the Load

Objective

Validate that the fact_production table was successfully loaded into the data warehouse, ensuring that the number of records, uniqueness constraints, and referential integrity are preserved after the ETL process.

Source Data

- fact_production_df (validated DataFrame before loading)
- fact_production (loaded MySQL fact table)

Transformation

No transformations are applied. The validation process compares the loaded fact table against the expected structure and data quality requirements.

Step 1. Verify total number of records

In [72]:
print(f"Rows after load: {get_row_count("fact_production")}")

Rows after load: 415903


Step 2. Validate business key uniqueness

In [73]:
result = execute_query(
    """
    SELECT COUNT(*) - COUNT(DISTINCT CONCAT(
        time_key,
        '-',
        well_key
    ))
    FROM fact_production
    """
)

duplicates = result[0][0]

print(f"Duplicated production records: {duplicates}")

Duplicated production records: 0


Step 3. Validate referential integrity

In [74]:
foreign_keys_check = execute_query(
    """
    SELECT
        SUM(time_key IS NULL),
        SUM(well_key IS NULL),
        SUM(geology_key IS NULL),
        SUM(field_area_key IS NULL),
        SUM(location_key IS NULL),
        SUM(company_key IS NULL)
    FROM fact_production
    """
)

foreign_keys_check

[(Decimal('0'),
  Decimal('0'),
  Decimal('0'),
  Decimal('0'),
  Decimal('0'),
  Decimal('0'))]

In [75]:
production_sql = (
    """
    SELECT
        time_key,
        well_key,
        geology_key,
        field_area_key,
        location_key,
        company_key,
        prod_pet,
        prod_gas
    FROM fact_production
    LIMIT 5;
    """
)

table_production = read_query(production_sql)
table_production.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,time_key,well_key,geology_key,field_area_key,location_key,company_key,prod_pet,prod_gas
0,1135,5839,1,47,1,1,15.60,274.01
1,1693,5840,2,48,2,1,1.71,48.13
2,1135,5841,3,46,3,1,14.04,272.42
3,1973,5842,2,47,4,1,1.05,148.47
4,1135,5843,4,40,5,1,73.39,1009.07


# Table fact_fracturing

## Objective

Prepare and load the hydraulic fracturing fact table by integrating fracturing events with the dimensional model, assigning surrogate keys from related dimensions, and preserving the original treatment-level granularity.

## Source Data

- fracture_df
- Dimension tables:
dim_time
dim_well
dim_company
dim_geology
dim_location
dim_field_area

## Transformation

The fracturing dataset is enriched through dimensional lookups to replace natural identifiers with surrogate keys. The final fact table contains only dimension references and hydraulic fracturing measurements required for analytical purposes.

### 2. Transform

#### 2.1 Select Required Columns

|Purpose|Columns|
|-------|-------|
|Time|fecha_data|
|Well|idpozo|
|Company|empresa_informante|
|Geology|formacion_productiva, tipo_reservorio, subtipo_reservorio|
|Field Area|yacimiento|
|Location|cuenca|
|Fracturing Measures|arena_bombeada_nacional_tn, arena_bombeada_importada_tn, agua_inyectada_m3,co2_inyectado_m3, presion_maxima_psi, potencia_equipos_fractura_hp, cantidad_fracturas|
|Business Identifier|id_base_fractura_adjiv|

In [76]:
fact_fracturing_df = fact_fracturing_df[
    [
        "id_base_fractura_adjiv",
        "fecha_data",
        "idpozo",
        "empresa_informante",
        "formacion_productiva",
        "tipo_reservorio",
        "subtipo_reservorio",
        "yacimiento",
        "cuenca",
        "arena_bombeada_nacional_tn",
        "arena_bombeada_importada_tn",
        "agua_inyectada_m3",
        "co2_inyectado_m3",
        "presion_maxima_psi",
        "potencia_equipos_fractura_hp",
        "cantidad_fracturas"
    ]
].copy()

In [77]:
fact_fracturing_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4806 entries, 0 to 4805
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_base_fractura_adjiv        4806 non-null   int64  
 1   fecha_data                    4806 non-null   object 
 2   idpozo                        4806 non-null   int64  
 3   empresa_informante            4806 non-null   object 
 4   formacion_productiva          4806 non-null   object 
 5   tipo_reservorio               4771 non-null   object 
 6   subtipo_reservorio            3832 non-null   object 
 7   yacimiento                    4806 non-null   object 
 8   cuenca                        4806 non-null   object 
 9   arena_bombeada_nacional_tn    4806 non-null   float64
 10  arena_bombeada_importada_tn   4806 non-null   float64
 11  agua_inyectada_m3             4806 non-null   float64
 12  co2_inyectado_m3              4806 non-null   float64
 13  pre

#### 2.2 Lookup time_key

Objective

Assign the corresponding time_key from dim_time to each hydraulic fracturing event, using the fracturing date as the natural temporal reference.

Transformation

The fecha_data attribute from the fracturing dataset is matched against the full_date attribute in dim_time.

The resulting surrogate key:

```text
fecha_data → full_date → time_key
```

will be stored in fact_fracturing_df.

#### Implementation

Step 1. Convert date fields to datetime

In [78]:
fact_fracturing_df["fecha_data"] = pd.to_datetime(
    fact_fracturing_df["fecha_data"]
)

In [79]:
sql = """
SELECT
    time_key,
    full_date
FROM dim_time;
"""

dim_time = read_query(sql)
dim_time.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,time_key,full_date
0,1,2006-01-31
1,2,2006-02-28
2,3,2006-03-31
3,4,2006-04-30
4,5,2006-05-19


In [80]:
dim_time["full_date"] = pd.to_datetime(
    dim_time["full_date"]
)

Step 2. Merge with dim_time

In [81]:
fact_fracturing_df["fracture_date"] = (
    fact_fracturing_df["fecha_data"]
    .dt.normalize()
)

In [82]:
fact_fracturing_df = fact_fracturing_df.merge(
    dim_time,
    left_on="fracture_date",
    right_on="full_date",
    how="left"
)

Step 3. Validation

In [83]:
print(f"Missing time_key: {fact_fracturing_df['time_key'].isna().sum()}")
print(f"Rows: {fact_fracturing_df.shape[0]}")

Missing time_key: 0
Rows: 4806


In [84]:
fact_fracturing_df.head()

,id_base_fractura_adjiv,fecha_data,idpozo,empresa_informante,formacion_productiva,tipo_reservorio,subtipo_reservorio,yacimiento,cuenca,arena_bombeada_nacional_tn,arena_bombeada_importada_tn,agua_inyectada_m3,co2_inyectado_m3,presion_maxima_psi,potencia_equipos_fractura_hp,cantidad_fracturas,fracture_date,time_key,full_date
0,30,2019-06-14 17:13:03.682790,159910,CAPEX S.A.,los molles,NO CONVENCIONAL,SHALE,AGUA DEL CAJON,NEUQUINA,0.000,0.000,2718.20,0.0,10190.0,10897.0,3,2019-06-14,2377,2019-06-14
1,31,2019-06-14 17:14:19.179874,159910,CAPEX S.A.,los molles,NO CONVENCIONAL,SHALE,AGUA DEL CAJON,NEUQUINA,0.000,0.000,600.00,0.0,9250.0,10251.0,1,2019-06-14,2377,2019-06-14
2,37,2019-06-27 13:46:21.149350,159219,YPF S.A.,vaca muerta,NO CONVENCIONAL,SHALE,AGUADA DE LA ARENA,NEUQUINA,3761.370,536.850,25768.30,0.0,15000.0,32000.0,18,2019-06-27,2388,2019-06-27
3,38,2019-06-27 13:46:21.149350,159220,YPF S.A.,vaca muerta,NO CONVENCIONAL,SHALE,AGUADA DE LA ARENA,NEUQUINA,3903.705,558.225,27398.37,0.0,11348.0,32000.0,19,2019-06-27,2388,2019-06-27
4,39,2019-06-27 13:46:21.149350,159221,YPF S.A.,vaca muerta,NO CONVENCIONAL,SHALE,AGUADA DE LA ARENA,NEUQUINA,3949.020,569.925,27157.60,0.0,11076.0,32000.0,19,2019-06-27,2388,2019-06-27


Step 4. Prepare fracturing start and end dates for time lookup

In [85]:
fracture_new_DF = pd.read_csv("../data/raw/datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv")

In [86]:
fracture_new_DF["fecha_inicio_fractura"] = pd.to_datetime(
    fracture_new_DF["fecha_inicio_fractura"]
)

fracture_new_DF["fecha_fin_fractura"] = pd.to_datetime(
    fracture_new_DF["fecha_fin_fractura"]
)

print("Missing fecha_inicio_fractura:", fracture_new_DF["fecha_inicio_fractura"].isna().sum())

print("Missing fecha_fin_fractura:", fracture_new_DF["fecha_fin_fractura"].isna().sum())

print(
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "fecha_inicio_fractura",
            "fecha_fin_fractura"
        ]
    ].head()
)

Missing fecha_inicio_fractura: 0
Missing fecha_fin_fractura: 0
   id_base_fractura_adjiv fecha_inicio_fractura fecha_fin_fractura
0                      30            2019-04-20         2019-04-30
1                      31            2018-11-02         2018-11-03
2                      37            2017-11-19         2017-12-14
3                      38            2017-11-21         2017-12-15
4                      39            2017-11-18         2017-12-15


Step 5. Prepare fracturing start and end dates

5.1. Create normalized start and end dates

In [87]:
fracture_new_DF["start_date"] = (
    pd.to_datetime(
        fracture_new_DF["fecha_inicio_fractura"]
    )
    .dt.normalize()
)

fracture_new_DF["end_date"] = (
    pd.to_datetime(
        fracture_new_DF["fecha_fin_fractura"]
    )
    .dt.normalize()
)

5.2. Validate the prepared dates

In [88]:
print("Unique start dates:", fracture_new_DF["start_date"].nunique())

print("Missing start dates:", fracture_new_DF["start_date"].isna().sum())

print("Unique end dates:", fracture_new_DF["end_date"].nunique())

print("Missing end dates:", fracture_new_DF["end_date"].isna().sum())

Unique start dates: 2729
Missing start dates: 0
Unique end dates: 2651
Missing end dates: 0


In [89]:
print(
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "fecha_inicio_fractura",
            "start_date",
            "fecha_fin_fractura",
            "end_date"
        ]
    ].head()
)

   id_base_fractura_adjiv fecha_inicio_fractura start_date fecha_fin_fractura  \
0                      30            2019-04-20 2019-04-20         2019-04-30   
1                      31            2018-11-02 2018-11-02         2018-11-03   
2                      37            2017-11-19 2017-11-19         2017-12-14   
3                      38            2017-11-21 2017-11-21         2017-12-15   
4                      39            2017-11-18 2017-11-18         2017-12-15   

    end_date  
0 2019-04-30  
1 2018-11-03  
2 2017-12-14  
3 2017-12-15  
4 2017-12-15  


Step 6. Build start and end time lookups

6.1. Prepare lookup of start dates

In [90]:
start_time_lookup = (
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "start_date"
        ]
    ]
    .merge(
        dim_time,
        left_on="start_date",
        right_on="full_date",
        how="left",
        validate="many_to_one"
    )
    [
        [
            "id_base_fractura_adjiv",
            "time_key"
        ]
    ]
    .rename(
        columns={
            "time_key": "start_time_key"
        }
    )
)

6.2. Validate

In [91]:
print("Rows in start_time_lookup:", start_time_lookup.shape[0])

print("Unique fracturing records:", start_time_lookup["id_base_fractura_adjiv"].nunique())

print("Missing start_time_key:", start_time_lookup["start_time_key"].isna().sum())

print("Duplicated fracturing IDs:", start_time_lookup["id_base_fractura_adjiv"].duplicated().sum())

Rows in start_time_lookup: 4806
Unique fracturing records: 4806
Missing start_time_key: 0
Duplicated fracturing IDs: 0


6.3. Build end time lookup

In [92]:
end_time_lookup = (
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "end_date"
        ]
    ]
    .merge(
        dim_time,
        left_on="end_date",
        right_on="full_date",
        how="left",
        validate="many_to_one"
    )
    [
        [
            "id_base_fractura_adjiv",
            "time_key"
        ]
    ]
    .rename(
        columns={
            "time_key": "end_time_key"
        }
    )
)

6.4. Validate

In [93]:
print("Rows in end_time_lookup:", end_time_lookup.shape[0])

print("Unique fracturing records:", end_time_lookup["id_base_fractura_adjiv"].nunique())

print("Missing end_time_key:", end_time_lookup["end_time_key"].isna().sum())

print("Duplicated fracturing IDs:", end_time_lookup["id_base_fractura_adjiv"].duplicated().sum())

Rows in end_time_lookup: 4806
Unique fracturing records: 4806
Missing end_time_key: 0
Duplicated fracturing IDs: 0


Step 7. Add start and end time keys to fact_fracturing_df

7.1. Merge start_time_key

In [94]:
fact_fracturing_df = fact_fracturing_df.merge(
    start_time_lookup,
    on="id_base_fractura_adjiv",
    how="left",
    validate="one_to_one"
)

print("Rows after start_time_key lookup:", fact_fracturing_df.shape[0])

print("Missing start_time_key:", fact_fracturing_df["start_time_key"].isna().sum())

Rows after start_time_key lookup: 4806
Missing start_time_key: 0


7.2. Merge end_time_key

In [95]:
fact_fracturing_df = fact_fracturing_df.merge(
    end_time_lookup,
    on="id_base_fractura_adjiv",
    how="left",
    validate="one_to_one"
)

print("Rows after end_time_key lookup:", fact_fracturing_df.shape[0])

print("Missing end_time_key:", fact_fracturing_df["end_time_key"].isna().sum())

Rows after end_time_key lookup: 4806
Missing end_time_key: 0


Step 8. Validate the complete time-key assignment

In [96]:
print("Rows:", fact_fracturing_df.shape[0])

print("Unique fracturing IDs:", fact_fracturing_df["id_base_fractura_adjiv"].nunique())

print("Missing start_time_key:", fact_fracturing_df["start_time_key"].isna().sum())

print("Missing end_time_key:", fact_fracturing_df["end_time_key"].isna().sum())

print("Duplicated fracturing IDs:", fact_fracturing_df["id_base_fractura_adjiv"].duplicated().sum())

Rows: 4806
Unique fracturing IDs: 4806
Missing start_time_key: 0
Missing end_time_key: 0
Duplicated fracturing IDs: 0


#### 2.3 Lookup well_key

Objective

Assign the corresponding well_key from dim_well to each hydraulic fracturing treatment using the natural well identifier idpozo.

Transformation

The natural key:

```text
idpozo
```

from the fracturing dataset is matched against:

```text
idpozo → dim_well → well_key
```

The resulting surrogate key will be stored in fact_fracturing_df.

Implementation

Step 1. Merge with dim_well

In [97]:
fact_fracturing_df = fact_fracturing_df.merge(
    dim_well,
    on="idpozo",
    how="left"
)

In [98]:
print(f"Missing well_key: {fact_fracturing_df["well_key"].isna().sum()}")
print(f"Rows: {fact_fracturing_df.shape[0]}")

Missing well_key: 0
Rows: 4806


Step 2. Merge validation

In [99]:
duplicate_well_keys = dim_well["idpozo"].duplicated().sum()

print(
    f"Duplicated idpozo in dim_well: "
    f"{duplicate_well_keys}"
)

Duplicated idpozo in dim_well: 0


#### 2.4 Lookup geology_key

Objetivo

Asignar la clave sustituta (geology_key) a cada registro de fact_fracturing utilizando la dimensión dim_geology.

#### Implementation

Step 1. Verify dimension

In [100]:
print("Columns in fact_fracturing_df:")
print(fact_fracturing_df.columns.tolist())

print("\nColumns in dim_field_area:")
print(dim_field_area.columns.tolist())

Columns in fact_fracturing_df:
['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key']

Columns in dim_field_area:
['field_area_key', 'idareapermisoconcesion', 'areapermisoconcesion', 'idareayacimiento', 'areayacimiento', 'yacimiento']


Step 2. Check well overlap between production and fracturing

In [101]:
production_well_ids = set(
    production_df["idpozo"].dropna().unique()
)

fracturing_well_ids = set(
    fact_fracturing_df["idpozo"].dropna().unique()
)

fracturing_wells_in_production = (
    fracturing_well_ids.intersection(production_well_ids)
)

fracturing_wells_only = (
    fracturing_well_ids.difference(production_well_ids)
)

print(f"Fracturing unique wells: " f"{len(fracturing_well_ids)}")

print(f"Fracturing wells also in production: " f"{len(fracturing_wells_in_production)}")

print(f"Fracturing-only wells: " f"{len(fracturing_wells_only)}")

Fracturing unique wells: 4562
Fracturing wells also in production: 3765
Fracturing-only wells: 797


Step 3. Build well-to-geology lookup from production

In [102]:
production_well_geology_lookup.head()

,idpozo,geology_key
0,132772,1
1,130300,2
2,130518,3
3,133429,2
4,143846,4


Step 4. Lookup geology_key for fracturing wells

4.1. That also exist in production

In [103]:
fact_fracturing_geology_lookup = fact_fracturing_df.merge(
    production_well_geology_lookup,
    on="idpozo",
    how="left"
)

print(f"Rows before lookup: " f"{fact_fracturing_df.shape[0]}")

print(f"Rows after lookup:  " f"{fact_fracturing_geology_lookup.shape[0]}")

print(f"Rows preserved: " f"{fact_fracturing_df.shape[0] == fact_fracturing_geology_lookup.shape[0]}")

print(f"Missing geology_key: " f"{fact_fracturing_geology_lookup['geology_key'].isna().sum()}")

Rows before lookup: 4806
Rows after lookup:  4806
Rows preserved: True
Missing geology_key: 969


Step 5. Analyze fracturing records without geology_key

In [104]:
fracturing_missing_geology = (
    fact_fracturing_geology_lookup[
        fact_fracturing_geology_lookup["geology_key"].isna()
    ]
    .copy()
)

print(f"Fracturing records without geology_key: " f"{fracturing_missing_geology.shape[0]}")

print(f"Unique wells without geology_key: " f"{fracturing_missing_geology['idpozo'].nunique()}")

fracturing_missing_geology.head()

Fracturing records without geology_key: 969
Unique wells without geology_key: 797


,id_base_fractura_adjiv,fecha_data,idpozo,empresa_informante,formacion_productiva,tipo_reservorio,subtipo_reservorio,yacimiento,cuenca,arena_bombeada_nacional_tn,...,presion_maxima_psi,potencia_equipos_fractura_hp,cantidad_fracturas,fracture_date,time_key,full_date,start_time_key,end_time_key,well_key,geology_key
717,893,2019-07-26 16:35:14.905733,136243,COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.,springhill,NaN,NaN,LAGUNA DE LOS CAPONES NORTE,AUSTRAL,0.0,...,3322.0,1063.0,1,2019-07-26,2410,2019-07-26,2339,2339,10880,NaN
789,1114,2019-08-06 12:08:47.074373,160771,COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.,ANITA,CONVENCIONAL,NaN,EL PUMA,AUSTRAL,0.0,...,2675.0,707.0,1,2019-08-06,2421,2019-08-06,2281,2281,10881,NaN
842,1176,2019-09-06 15:32:42.764915,8788,CHEVRON ARGENTINA S.R.L.,agrio,CONVENCIONAL,NaN,CURAMCHED,NEUQUINA,0.0,...,5780.0,2433.0,1,2019-09-06,2446,2019-09-06,191,191,10882,NaN
843,1177,2019-09-06 15:32:42.764915,136559,CHEVRON ARGENTINA S.R.L.,huitrín,CONVENCIONAL,NaN,EL TRAPIAL,NEUQUINA,0.0,...,3050.0,1038.0,1,2019-09-06,2446,2019-09-06,326,326,10883,NaN
844,1178,2019-09-06 15:38:19.921644,136362,CHEVRON ARGENTINA S.R.L.,rayoso,CONVENCIONAL,NaN,EL TRAPIAL,NEUQUINA,0.0,...,1550.0,515.0,1,2019-09-06,2446,2019-09-06,308,308,10884,NaN


5.1. Classify missing-geology wells by production availability

In [105]:
fracturing_missing_wells = (
    fracturing_missing_geology["idpozo"]
    .drop_duplicates()
)

fracturing_missing_well_analysis = pd.DataFrame(
    {
        "idpozo": fracturing_missing_wells
    }
)

fracturing_missing_well_analysis["in_production"] = (
    fracturing_missing_well_analysis["idpozo"]
    .isin(production_well_geology_lookup["idpozo"])
)

print(
    "Missing-geology wells also in production:",
    fracturing_missing_well_analysis[
        "in_production"
    ].sum()
)

print(
    "Missing-geology wells exclusive to fracturing:",
    (
        ~fracturing_missing_well_analysis[
            "in_production"
        ]
    ).sum()
)

Missing-geology wells also in production: 0
Missing-geology wells exclusive to fracturing: 797


Step 6. Extract unique geological profiles

6.1. from fracturing-only wells

In [106]:
fracturing_only_wells = (
    fracturing_missing_well_analysis[
        ~fracturing_missing_well_analysis["in_production"]
    ]["idpozo"]
)

fracturing_only_profiles = (
    fact_fracturing_geology_lookup[
        fact_fracturing_geology_lookup["idpozo"].isin(
            fracturing_only_wells
        )
    ][
        [
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Fracturing-only geological profiles: " f"{fracturing_only_profiles.shape[0]}")

Fracturing-only geological profiles: 34


Step 7. Load geological dimension for candidate matching

In [107]:
dim_geology_for_fracturing = read_query("""
    SELECT
        geology_key,
        formacion_productiva,
        tipo_reservorio,
        subtipo_reservorio
    FROM dim_geology
""")

print(f"Dimension geological profiles: " f"{dim_geology_for_fracturing.shape[0]}")

dim_geology_for_fracturing.head()

Dimension geological profiles: 113


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,geology_key,formacion_productiva,tipo_reservorio,subtipo_reservorio
0,1,None,None,None
1,2,None,None,None
2,3,precuyo,NO CONVENCIONAL,TIGHT
3,4,None,None,None
4,5,lajas,NO CONVENCIONAL,TIGHT


7.1. Find candidate geology_keys for each fracturing-only geological profile

In [108]:
fracturing_profile_candidates = (
    fracturing_only_profiles.merge(
        dim_geology_for_fracturing,
        on=[
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ],
        how="left"
    )
)

print(f"Candidate matches: " f"{fracturing_profile_candidates.shape[0]}")

Candidate matches: 34


7.2. Count candidate geology_keys per fracturing profile

In [109]:
fracturing_profile_candidate_summary = (
    fracturing_profile_candidates
    .groupby(
        [
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ],
        dropna=False
    )
    .agg(
        candidate_count=(
            "geology_key",
            lambda x: x.notna().sum()
        ),
        candidate_geology_keys=(
            "geology_key",
            lambda x: sorted(x.dropna().astype(int).unique().tolist())
        )
    )
    .reset_index()
)

print("Candidate count distribution:")

print(
    fracturing_profile_candidate_summary[
        "candidate_count"
    ].value_counts().sort_index()
)

fracturing_profile_candidate_summary.head(10)

Candidate count distribution:
candidate_count
1    34
Name: count, dtype: int64


,formacion_productiva,tipo_reservorio,subtipo_reservorio,candidate_count,candidate_geology_keys
0,ANITA,CONVENCIONAL,NaN,1,[63]
1,agrio,CONVENCIONAL,NaN,1,[65]
2,agrio,NaN,NaN,1,[69]
3,bajo barreal,CONVENCIONAL,NaN,1,[105]
4,castillo,NaN,NaN,1,[107]
5,cañadon seco,CONVENCIONAL,NaN,1,[102]
6,cañadon seco,NaN,NaN,1,[100]
7,comodoro rivadavia,CONVENCIONAL,NaN,1,[86]
8,comodoro rivadavia,NaN,NaN,1,[98]
9,el trebol,CONVENCIONAL,NaN,1,[88]


Step 8. Build geology lookup for fracturing-only profiles

In [110]:
fracturing_only_geology_lookup = (
    fracturing_profile_candidates[
        [
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio",
            "geology_key"
        ]
    ]
    .copy()
)

print(f"Fracturing-only geological profiles: " f"{fracturing_only_geology_lookup.shape[0]}")

print(
    "Duplicated profiles:",
    fracturing_only_geology_lookup.duplicated(
        subset=[
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ).sum()
)

fracturing_only_geology_lookup.head(10)

Fracturing-only geological profiles: 34
Duplicated profiles: 0


,formacion_productiva,tipo_reservorio,subtipo_reservorio,geology_key
0,springhill,NaN,NaN,62
1,ANITA,CONVENCIONAL,NaN,63
2,agrio,CONVENCIONAL,NaN,65
3,huitrín,CONVENCIONAL,NaN,66
4,rayoso,CONVENCIONAL,NaN,67
5,huitrín,NaN,NaN,68
6,agrio,NaN,NaN,69
7,formación improductiva,CONVENCIONAL,NaN,70
8,lotena,CONVENCIONAL,NaN,74
9,lajas,CONVENCIONAL,NaN,75


In [111]:
print("Columns in fracturing_only_geology_lookup:")
print(fracturing_only_geology_lookup.columns.tolist())

print("\nShape:")
print(fracturing_only_geology_lookup.shape)

print("\nDuplicated profiles:")
print(
    fracturing_only_geology_lookup.duplicated(
        subset=[
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ).sum()
)

print("\nMissing geology_key:")
print(fracturing_only_geology_lookup["geology_key"].isna().sum())

Columns in fracturing_only_geology_lookup:
['formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'geology_key']

Shape:
(34, 4)

Duplicated profiles:
0

Missing geology_key:
0


Step 9. Resolve geology_key for fracturing wells

9.1. Using the well-level geology lookup from production

In [112]:
fracturing_well_geology = fact_fracturing_df[
    ["idpozo"]
].drop_duplicates()

fracturing_well_geology = fracturing_well_geology.merge(
    production_well_geology_lookup,
    on="idpozo",
    how="left"
)

print(f"Fracturing unique wells: " f"{fracturing_well_geology['idpozo'].nunique()}")

print(f"Wells with geology_key from production: " 
    f"{fracturing_well_geology['geology_key'].notna().sum()}"
)

print(f"Wells without geology_key from production: " f"{fracturing_well_geology['geology_key'].isna().sum()}"
)

fracturing_well_geology.head()

Fracturing unique wells: 4562
Wells with geology_key from production: 3765
Wells without geology_key from production: 797


,idpozo,geology_key
0,159910,37.0
1,159219,12.0
2,159220,12.0
3,159221,12.0
4,160426,12.0


In [113]:
fracturing_only_wells = fracturing_well_geology[
    fracturing_well_geology["geology_key"].isna()
].copy()

In [114]:
print("Fracturing-only wells:", fracturing_only_wells.shape[0])

print("Missing geology_key:", fracturing_only_wells["geology_key"].isna().sum())

fracturing_only_wells.head()

Fracturing-only wells: 797
Missing geology_key: 797


,idpozo,geology_key
716,136243,NaN
788,160771,NaN
838,8788,NaN
839,136559,NaN
840,136362,NaN


Step 10. Build fracturing-only well profiles

In [115]:
fracturing_only_well_profiles = (
    fact_fracturing_df[
        [
            "idpozo",
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ]
    .drop_duplicates(subset=["idpozo"])
    .merge(
        fracturing_only_wells[
            ["idpozo"]
        ],
        on="idpozo",
        how="inner"
    )
)

In [116]:
print("Fracturing-only wells:", fracturing_only_well_profiles["idpozo"].nunique())

print("Rows:", fracturing_only_well_profiles.shape[0])

print("Duplicated wells:", fracturing_only_well_profiles["idpozo"].duplicated().sum())

Fracturing-only wells: 797
Rows: 797
Duplicated wells: 0


Step 11. Validate geological consistency per fracturing-only well

In [117]:
fracturing_only_profile_consistency = (
    fact_fracturing_df[
        [
            "idpozo",
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ]
    .merge(
        fracturing_only_wells[["idpozo"]],
        on="idpozo",
        how="inner"
    )
    .drop_duplicates()
)

profile_variations_per_well = (
    fracturing_only_profile_consistency
    .groupby("idpozo")
    .size()
)

In [118]:
print("Fracturing-only wells:", profile_variations_per_well.shape[0])

print("Wells with more than one geological profile:", (profile_variations_per_well > 1).sum())

print("Maximum profiles for a single well:", profile_variations_per_well.max())

Fracturing-only wells: 797
Wells with more than one geological profile: 0
Maximum profiles for a single well: 1


Step 12. Lookup geology_key por perfil

In [119]:
fracturing_only_well_geology = (
    fracturing_only_well_profiles[
        [
            "idpozo",
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ]
    .merge(
        fracturing_only_geology_lookup,
        on=[
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ],
        how="left",
        validate="many_to_one"
    )
)

In [120]:
print(f"Rows after geology lookup: " f"{fracturing_only_well_geology.shape[0]}")

print(f"Missing geology_key: " f"{fracturing_only_well_geology['geology_key'].isna().sum()}")

print(f"Unique wells: " f"{fracturing_only_well_geology['idpozo'].nunique()}")

Rows after geology lookup: 797
Missing geology_key: 0
Unique wells: 797


In [121]:
print(
    "Duplicated geological profiles:",
    fracturing_only_geology_lookup.duplicated(
        subset=[
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ).sum()
)

Duplicated geological profiles: 0


Step 13. Validate fracturing-only wells

In [122]:
production_well_ids = set(
    production_well_geology_lookup["idpozo"]
)

fracturing_only_well_ids = set(
    fracturing_only_well_geology["idpozo"]
)

overlap_wells = (
    fracturing_only_well_ids
    .intersection(production_well_ids)
)

print(f"Production wells: " f"{len(production_well_ids)}")

print(f"Fracturing-only wells: " f"{len(fracturing_only_well_ids)}")

print(f"Overlap between production and fracturing-only wells: " f"{len(overlap_wells)}")

Production wells: 5041
Fracturing-only wells: 797
Overlap between production and fracturing-only wells: 0


Step 14. Build complete well → geology lookup

In [123]:
complete_well_geology_lookup = pd.concat(
    [
        production_well_geology_lookup[
            ["idpozo", "geology_key"]
        ],
        fracturing_only_well_geology[
            ["idpozo", "geology_key"]
        ]
    ],
    ignore_index=True
)

print(f"Rows: " f"{complete_well_geology_lookup.shape[0]}")

print(f"Unique wells: " f"{complete_well_geology_lookup['idpozo'].nunique()}")

print(f"Duplicated wells: " f"{complete_well_geology_lookup['idpozo'].duplicated().sum()}")

print(f"Missing geology_key: " f"{complete_well_geology_lookup['geology_key'].isna().sum()}")

complete_well_geology_lookup.head()

Rows: 5838
Unique wells: 5838
Duplicated wells: 0
Missing geology_key: 0


,idpozo,geology_key
0,132772,1
1,130300,2
2,130518,3
3,133429,2
4,143846,4


Step 15. Apply complete well → geology lookup to fracturing

In [124]:
fact_fracturing_geology_df = fact_fracturing_df.merge(
    complete_well_geology_lookup,
    on="idpozo",
    how="left",
    validate="many_to_one"
)

print(f"Rows before lookup: " f"{fact_fracturing_df.shape[0]}")

print(f"Rows after lookup: " f"{fact_fracturing_geology_df.shape[0]}")

print(f"Rows preserved: " f"{fact_fracturing_geology_df.shape[0] == fact_fracturing_df.shape[0]}")

print(f"Missing geology_key: " f"{fact_fracturing_geology_df['geology_key'].isna().sum()}")

print(f"Unique wells: " f"{fact_fracturing_geology_df['idpozo'].nunique()}")

Rows before lookup: 4806
Rows after lookup: 4806
Rows preserved: True
Missing geology_key: 0
Unique wells: 4562


Step 16. Validate well → geology consistency

In [125]:
well_geology_counts = (
    fact_fracturing_geology_df
    .groupby("idpozo")["geology_key"]
    .nunique()
)

print(f"Wells with more than one geology_key: " f"{(well_geology_counts > 1).sum()}")

print(f"Maximum geology_keys for a single well: " f"{well_geology_counts.max()}")

Wells with more than one geology_key: 0
Maximum geology_keys for a single well: 1


Step 17. Build well-to-geology lookup

In [126]:
fracturing_well_geology_lookup = (
    fact_fracturing_geology_df[
        [
            "idpozo",
            "geology_key"
        ]
    ]
    .drop_duplicates()
    .copy()
)

Step 18. Lookup geology_key in fact_fracturing_df

In [127]:
fact_fracturing_df = (
    fact_fracturing_df
    .drop(columns=["geology_key"], errors="ignore")
    .merge(
        fracturing_well_geology_lookup,
        on="idpozo",
        how="left",
        validate="many_to_one"
    )
)

print(f"Rows after geology lookup: " f"{fact_fracturing_df.shape[0]}")

print(f"Missing geology_key: " f"{fact_fracturing_df['geology_key'].isna().sum()}")

print(f"Unique wells: " f"{fact_fracturing_df['idpozo'].nunique()}")

Rows after geology lookup: 4806
Missing geology_key: 0
Unique wells: 4562


#### 2.5 Lookup field_area_key

Implementation

Step 1. Verify dimension

In [128]:
print("Columns in fact_fracturing_df:")
print(fact_fracturing_df.columns.tolist())

print("\nColumns in dim_field_area:")
print(dim_field_area.columns.tolist())

Columns in fact_fracturing_df:
['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key', 'geology_key']

Columns in dim_field_area:
['field_area_key', 'idareapermisoconcesion', 'areapermisoconcesion', 'idareayacimiento', 'areayacimiento', 'yacimiento']


Step 2. Validate field-area lookup keys

In [129]:
field_area_profile_lookup = (
    dim_field_area[
        [
            "areapermisoconcesion",
            "yacimiento",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
)

print(f"Rows in field_area_profile_lookup: " f"{field_area_profile_lookup.shape[0]}")

print(
    f"Duplicated area profiles: "
    f"{field_area_profile_lookup.duplicated(
        subset=[
            "areapermisoconcesion",
            "yacimiento"
        ]
    ).sum()}"
)

print(f"Missing field_area_key: " f"{field_area_profile_lookup['field_area_key'].isna().sum()}")

Rows in field_area_profile_lookup: 292
Duplicated area profiles: 31
Missing field_area_key: 0


Step 3. Investigate ambiguous field-area profiles

In [130]:
ambiguous_field_area_profiles = (
    dim_field_area[
        dim_field_area.duplicated(
            subset=[
                "areapermisoconcesion",
                "yacimiento"
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "areapermisoconcesion",
            "yacimiento"
        ]
    )
)

print(f"Ambiguous field-area profiles: " f"{ambiguous_field_area_profiles.shape[0]}")

ambiguous_field_area_profiles[
    [
        "field_area_key",
        "idareapermisoconcesion",
        "areapermisoconcesion",
        "idareayacimiento",
        "areayacimiento",
        "yacimiento"
    ]
].head()

Ambiguous field-area profiles: 49


,field_area_key,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,yacimiento
29,30,CAO,AGUA DEL CAJON,ELSA,EL SALITRAL,None
38,39,CAO,AGUA DEL CAJON,CAO,AGUA DEL CAJON,None
55,56,APE,AGUADA PICHANA ESTE,APEM,AGUADA PICHANA ESTE MULICHINCO,None
231,232,APE,AGUADA PICHANA ESTE,APVM,AGUADA PICHANA ESTE VACA MUERTA,None
43,44,ANC,ANTICLINAL CAMPAMENTO,ACO,ANTICLINAL CAMPAMENTO OESTE,None


Step 4. Validate fracturing field-area profiles against dim_field_area

In [131]:
fracturing_field_area_profiles = (
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "areapermisoconcesion",
                "areayacimiento"
            ]
        ],
        left_on=[
            "areapermisoconcesion",
            "yacimiento"
        ],
        right_on=[
            "areapermisoconcesion",
            "areayacimiento"
        ],
        how="left"
    )
)

print(
    f"Rows after field-area matching: "
    f"{fracturing_field_area_profiles.shape[0]}"
)

print(
    f"Missing field_area_key: "
    f"{fracturing_field_area_profiles['field_area_key'].isna().sum()}"
)

print(
    f"Unique fracturing records: "
    f"{fracturing_field_area_profiles['id_base_fractura_adjiv'].nunique()}"
)

Rows after field-area matching: 8871
Missing field_area_key: 282
Unique fracturing records: 4806


In [132]:
fracturing_field_area_ambiguous = (
    fracturing_field_area_profiles
    .groupby("id_base_fractura_adjiv")
    .filter(
        lambda x: x["field_area_key"].nunique() > 1
    )
)

print(f"Ambiguous fracturing records: " f"{fracturing_field_area_ambiguous['id_base_fractura_adjiv'].nunique()}")

print(f"Unique wells among ambiguous records: " f"{fracturing_field_area_ambiguous['idpozo'].nunique()}")

Ambiguous fracturing records: 3815
Unique wells among ambiguous records: 3738


In [133]:
fracturing_field_area_ambiguous[
    [
        "id_base_fractura_adjiv",
        "idpozo",
        "areapermisoconcesion",
        "yacimiento",
        "field_area_key"
    ]
].head(10)

,id_base_fractura_adjiv,idpozo,areapermisoconcesion,yacimiento,field_area_key
0,30,159910,AGUA DEL CAJON,AGUA DEL CAJON,39.0
1,30,159910,AGUA DEL CAJON,AGUA DEL CAJON,144.0
2,31,159910,AGUA DEL CAJON,AGUA DEL CAJON,39.0
3,31,159910,AGUA DEL CAJON,AGUA DEL CAJON,144.0
13,46,154744,EL OREJANO,EL OREJANO,82.0
14,46,154744,EL OREJANO,EL OREJANO,133.0
15,47,158750,EL OREJANO,EL OREJANO,82.0
16,47,158750,EL OREJANO,EL OREJANO,133.0
17,48,158751,EL OREJANO,EL OREJANO,82.0
18,48,158751,EL OREJANO,EL OREJANO,133.0


Step 5. Identify ambiguous fracturing field-area matches

In [134]:
fracturing_field_area_counts = (
    fracturing_field_area_profiles
    .groupby("id_base_fractura_adjiv")["field_area_key"]
    .nunique()
)

print(f"Fracturing records with more than one field_area_key: " f"{(fracturing_field_area_counts > 1).sum()}")

print(f"Maximum field_area_keys for a single fracturing record: " f"{fracturing_field_area_counts.max()}")

Fracturing records with more than one field_area_key: 3815
Maximum field_area_keys for a single fracturing record: 4


Step 6. Classify fracturing records by field-area match

In [135]:
fracturing_field_area_classification = (
    fracturing_field_area_profiles
    .groupby("id_base_fractura_adjiv")["field_area_key"]
    .nunique()
    .reset_index(name="field_area_key_count")
)

fracturing_field_area_classification["match_status"] = (
    fracturing_field_area_classification["field_area_key_count"]
    .map({
        0: "unmatched",
        1: "unique",
    })
    .fillna("ambiguous")
)

print(f"Total fracturing records: " f"{fracturing_field_area_classification.shape[0]}")

print(
    f"Unique field-area matches: "
    f"{(
        fracturing_field_area_classification["match_status"] == "unique"
    ).sum()}"
)

print(
    f"Ambiguous field-area matches: "
    f"{(
        fracturing_field_area_classification["match_status"] == "ambiguous"
    ).sum()}"
)

print(
    f"Unmatched field-area records: "
    f"{(
        fracturing_field_area_classification["match_status"] == "unmatched"
    ).sum()}"
)

fracturing_field_area_classification.head(10)

Total fracturing records: 4806
Unique field-area matches: 709
Ambiguous field-area matches: 3815
Unmatched field-area records: 282


,id_base_fractura_adjiv,field_area_key_count,match_status
0,30,2,ambiguous
1,31,2,ambiguous
2,37,1,unique
3,38,1,unique
4,39,1,unique
5,40,1,unique
6,41,1,unique
7,42,1,unique
8,43,1,unique
9,44,1,unique


Step 7. Resolve unique field-area matches

In [136]:
unique_field_area_matches = (
    fracturing_field_area_profiles
    .merge(
        fracturing_field_area_classification[
            [
                "id_base_fractura_adjiv",
                "match_status"
            ]
        ],
        on="id_base_fractura_adjiv",
        how="inner"
    )
    .query("match_status == 'unique'")
    [
        [
            "id_base_fractura_adjiv",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in unique field-area lookup: "
    f"{unique_field_area_matches.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{unique_field_area_matches['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing field_area_key: "
    f"{unique_field_area_matches['field_area_key'].isna().sum()}"
)

unique_field_area_matches.head(10)

Rows in unique field-area lookup: 709
Unique fracturing IDs: 709
Missing field_area_key: 0


,id_base_fractura_adjiv,field_area_key
0,37,197.0
1,38,197.0
2,39,197.0
3,40,197.0
4,41,197.0
5,42,197.0
6,43,197.0
7,44,197.0
8,45,197.0
9,879,100.0


Step 8. Apply unique field-area matches

In [137]:
fact_fracturing_df = (
    fact_fracturing_df
    .drop(columns=["field_area_key"], errors="ignore")
    .merge(
        unique_field_area_matches,
        on="id_base_fractura_adjiv",
        how="left",
        validate="many_to_one"
    )
)

print(
    f"Rows after unique field-area lookup: "
    f"{fact_fracturing_df.shape[0]}"
)

print(
    f"Unique fracturing records: "
    f"{fact_fracturing_df['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing field_area_key: "
    f"{fact_fracturing_df['field_area_key'].isna().sum()}"
)

Rows after unique field-area lookup: 4806
Unique fracturing records: 4806
Missing field_area_key: 4097


Step 9. Identify ambiguous field-area matches

In [138]:
ambiguous_field_area_records = (
    fracturing_field_area_profiles
    .merge(
        fracturing_field_area_classification[
            [
                "id_base_fractura_adjiv",
                "match_status"
            ]
        ],
        on="id_base_fractura_adjiv",
        how="inner"
    )
    .query("match_status == 'ambiguous'")
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in ambiguous field-area records: "
    f"{ambiguous_field_area_records.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{ambiguous_field_area_records['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells: "
    f"{ambiguous_field_area_records['idpozo'].nunique()}"
)

print(
    f"Maximum candidate field_area_keys: "
    f"{ambiguous_field_area_records.groupby('id_base_fractura_adjiv')['field_area_key'].nunique().max()}"
)

ambiguous_field_area_records.head(10)

Rows in ambiguous field-area records: 7880
Unique fracturing IDs: 3815
Unique wells: 3738
Maximum candidate field_area_keys: 4


,id_base_fractura_adjiv,idpozo,areapermisoconcesion,yacimiento,field_area_key,areayacimiento,match_status
0,30,159910,AGUA DEL CAJON,AGUA DEL CAJON,39.0,AGUA DEL CAJON,ambiguous
1,30,159910,AGUA DEL CAJON,AGUA DEL CAJON,144.0,AGUA DEL CAJON,ambiguous
2,31,159910,AGUA DEL CAJON,AGUA DEL CAJON,39.0,AGUA DEL CAJON,ambiguous
3,31,159910,AGUA DEL CAJON,AGUA DEL CAJON,144.0,AGUA DEL CAJON,ambiguous
4,46,154744,EL OREJANO,EL OREJANO,82.0,EL OREJANO,ambiguous
5,46,154744,EL OREJANO,EL OREJANO,133.0,EL OREJANO,ambiguous
6,47,158750,EL OREJANO,EL OREJANO,82.0,EL OREJANO,ambiguous
7,47,158750,EL OREJANO,EL OREJANO,133.0,EL OREJANO,ambiguous
8,48,158751,EL OREJANO,EL OREJANO,82.0,EL OREJANO,ambiguous
9,48,158751,EL OREJANO,EL OREJANO,133.0,EL OREJANO,ambiguous


Step 10. Analyze candidate field-area keys

In [139]:
ambiguous_field_area_counts = (
    ambiguous_field_area_records
    .groupby("id_base_fractura_adjiv")["field_area_key"]
    .nunique()
    .value_counts()
    .sort_index()
)

print("Candidate field_area_keys per fracturing record:")
print(ambiguous_field_area_counts)

print("-" * 35)

print(
    f"Fracturing IDs with 2 candidate keys: "
    f"{ambiguous_field_area_counts.get(2, 0)}"
)

print(
    f"Fracturing IDs with 3 candidate keys: "
    f"{ambiguous_field_area_counts.get(3, 0)}"
)

print(
    f"Fracturing IDs with 4 candidate keys: "
    f"{ambiguous_field_area_counts.get(4, 0)}"
)

Candidate field_area_keys per fracturing record:
field_area_key
2    3690
4     125
Name: count, dtype: int64
-----------------------------------
Fracturing IDs with 2 candidate keys: 3690
Fracturing IDs with 3 candidate keys: 0
Fracturing IDs with 4 candidate keys: 125


Step 11. Inspect ambiguous field-area candidates

In [140]:
ambiguous_field_area_candidates = (
    ambiguous_field_area_records[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento",
            "field_area_key",
            "areayacimiento"
        ]
    ]
    .sort_values(
        [
            "areapermisoconcesion",
            "yacimiento",
            "id_base_fractura_adjiv",
            "field_area_key"
        ]
    )
    .reset_index(drop=True)
)

print(
    f"Rows in ambiguous candidates: "
    f"{ambiguous_field_area_candidates.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{ambiguous_field_area_candidates['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique candidate field_area_keys: "
    f"{ambiguous_field_area_candidates['field_area_key'].nunique()}"
)

ambiguous_field_area_candidates.head(10)

Rows in ambiguous candidates: 7880
Unique fracturing IDs: 3815
Unique candidate field_area_keys: 154


,id_base_fractura_adjiv,idpozo,areapermisoconcesion,yacimiento,field_area_key,areayacimiento
0,30,159910,AGUA DEL CAJON,AGUA DEL CAJON,39.0,AGUA DEL CAJON
1,30,159910,AGUA DEL CAJON,AGUA DEL CAJON,144.0,AGUA DEL CAJON
2,31,159910,AGUA DEL CAJON,AGUA DEL CAJON,39.0,AGUA DEL CAJON
3,31,159910,AGUA DEL CAJON,AGUA DEL CAJON,144.0,AGUA DEL CAJON
4,1156,160202,AGUA DEL CAJON,AGUA DEL CAJON,39.0,AGUA DEL CAJON
5,1156,160202,AGUA DEL CAJON,AGUA DEL CAJON,144.0,AGUA DEL CAJON
6,1164,160202,AGUA DEL CAJON,AGUA DEL CAJON,39.0,AGUA DEL CAJON
7,1164,160202,AGUA DEL CAJON,AGUA DEL CAJON,144.0,AGUA DEL CAJON
8,2873,152937,AGUA DEL CAJON,AGUA DEL CAJON,39.0,AGUA DEL CAJON
9,2873,152937,AGUA DEL CAJON,AGUA DEL CAJON,144.0,AGUA DEL CAJON


Step 12. Match ambiguous wells against production history

In [141]:
ambiguous_wells = (
    ambiguous_field_area_records[
        ["idpozo"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

production_ambiguous_well_area_history = (
    production_df[
        production_df["idpozo"].isin(
            ambiguous_wells["idpozo"]
        )
    ][
        [
            "idpozo",
            "idareapermisoconcesion",
            "areapermisoconcesion",
            "idareayacimiento",
            "areayacimiento"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["idpozo", "idareapermisoconcesion", "idareayacimiento"]
    )
    .reset_index(drop=True)
)

print(
    f"Rows in production history for ambiguous wells: "
    f"{production_ambiguous_well_area_history.shape[0]}"
)

print(
    f"Unique wells in production history: "
    f"{production_ambiguous_well_area_history['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{production_ambiguous_well_area_history['idpozo'].isna().sum()}"
)

production_ambiguous_well_area_history.head(10)

Rows in production history for ambiguous wells: 3486
Unique wells in production history: 3486
Missing idpozo: 0


,idpozo,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento
0,11191,CIEC,CAMPO INDIO ESTE - EL CERRITO,INDI,CAMPO INDIO
1,73672,CIEC,CAMPO INDIO ESTE - EL CERRITO,INDI,CAMPO INDIO
2,129725,CIEC,CAMPO INDIO ESTE - EL CERRITO,INDI,CAMPO INDIO
3,130158,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO
4,130518,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO
5,131344,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO
6,131431,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO
7,132487,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO
8,132491,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO
9,132492,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO


Step 13. Match ambiguous wells with candidate field-area keys

In [142]:
ambiguous_field_area_candidates = (
    ambiguous_field_area_records[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
)

ambiguous_candidates_with_production = (
    ambiguous_field_area_candidates
    .merge(
        production_ambiguous_well_area_history[
            [
                "idpozo",
                "idareapermisoconcesion",
                "areapermisoconcesion",
                "idareayacimiento",
                "areayacimiento"
            ]
        ],
        on="idpozo",
        how="left",
        suffixes=(
            "_fracturing",
            "_production"
        )
    )
    .reset_index(drop=True)
)

print(
    f"Rows after matching ambiguous candidates with production: "
    f"{ambiguous_candidates_with_production.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{ambiguous_candidates_with_production['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells: "
    f"{ambiguous_candidates_with_production['idpozo'].nunique()}"
)

print(
    f"Rows with production area information: "
    f"{ambiguous_candidates_with_production['idareapermisoconcesion'].notna().sum()}"
)

ambiguous_candidates_with_production.head(10)

Rows after matching ambiguous candidates with production: 7880
Unique fracturing IDs: 3815
Unique wells: 3738
Rows with production area information: 7348


,id_base_fractura_adjiv,idpozo,areapermisoconcesion_fracturing,yacimiento,field_area_key,idareapermisoconcesion,areapermisoconcesion_production,idareayacimiento,areayacimiento
0,30,159910,AGUA DEL CAJON,AGUA DEL CAJON,39.0,CAO,AGUA DEL CAJON,CAO,AGUA DEL CAJON
1,30,159910,AGUA DEL CAJON,AGUA DEL CAJON,144.0,CAO,AGUA DEL CAJON,CAO,AGUA DEL CAJON
2,31,159910,AGUA DEL CAJON,AGUA DEL CAJON,39.0,CAO,AGUA DEL CAJON,CAO,AGUA DEL CAJON
3,31,159910,AGUA DEL CAJON,AGUA DEL CAJON,144.0,CAO,AGUA DEL CAJON,CAO,AGUA DEL CAJON
4,46,154744,EL OREJANO,EL OREJANO,82.0,ELOR,EL OREJANO,ELOR,EL OREJANO
5,46,154744,EL OREJANO,EL OREJANO,133.0,ELOR,EL OREJANO,ELOR,EL OREJANO
6,47,158750,EL OREJANO,EL OREJANO,82.0,ELOR,EL OREJANO,ELOR,EL OREJANO
7,47,158750,EL OREJANO,EL OREJANO,133.0,ELOR,EL OREJANO,ELOR,EL OREJANO
8,48,158751,EL OREJANO,EL OREJANO,82.0,ELOR,EL OREJANO,ELOR,EL OREJANO
9,48,158751,EL OREJANO,EL OREJANO,133.0,ELOR,EL OREJANO,ELOR,EL OREJANO


Step 14. Validate ambiguous field-area candidates using production history

In [143]:
ambiguous_candidates_supported_by_production = (
    ambiguous_candidates_with_production[
        ambiguous_candidates_with_production[
            "idareapermisoconcesion"
        ].notna()
    ]
    .loc[
        lambda df:
        (
            df["idareayacimiento"].astype("string")
            ==
            df["field_area_key"]
            .map(
                dim_field_area
                .set_index("field_area_key")["idareayacimiento"]
            )
            .astype("string")
        )
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows supported by production history: "
    f"{ambiguous_candidates_supported_by_production.shape[0]}"
)

print(
    f"Fracturing IDs supported by production: "
    f"{ambiguous_candidates_supported_by_production['id_base_fractura_adjiv'].nunique()}"
)

production_supported_counts = (
    ambiguous_candidates_supported_by_production
    .groupby("id_base_fractura_adjiv")["field_area_key"]
    .nunique()
)

print(
    f"Fracturing IDs with 1 supported field_area_key: "
    f"{(production_supported_counts == 1).sum()}"
)

print(
    f"Fracturing IDs with 2 supported field_area_keys: "
    f"{(production_supported_counts == 2).sum()}"
)

print(
    f"Fracturing IDs with more than 2 supported field_area_keys: "
    f"{(production_supported_counts > 2).sum()}"
)

print(
    f"Maximum supported field_area_keys: "
    f"{production_supported_counts.max()}"
)

Rows supported by production history: 7102
Fracturing IDs supported by production: 3551
Fracturing IDs with 1 supported field_area_key: 0
Fracturing IDs with 2 supported field_area_keys: 3551
Fracturing IDs with more than 2 supported field_area_keys: 0
Maximum supported field_area_keys: 2


Step 15. Characterize remaining field-area candidate pairs

In [144]:
candidate_pairs = (
    ambiguous_candidates_supported_by_production[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion_fracturing",
            "yacimiento",
            "field_area_key",
            "idareapermisoconcesion",
            "areapermisoconcesion_production",
            "idareayacimiento",
            "areayacimiento"
        ]
    ]
    .drop_duplicates()
)

15.1. Build one record per fracturing ID with its two keys

In [145]:
candidate_pairs_by_fracturing = (
    candidate_pairs
    .groupby("id_base_fractura_adjiv")
    .agg(
        field_area_keys=(
            "field_area_key",
            lambda x: tuple(sorted(x))
        )
    )
    .reset_index()
)

# Keep only IDs with exactly two candidates
candidate_pairs_by_fracturing = (
    candidate_pairs_by_fracturing[
        candidate_pairs_by_fracturing["field_area_keys"].map(len) == 2
    ]
    .copy()
)

candidate_pairs_by_fracturing[
    ["field_area_key_1", "field_area_key_2"]
] = pd.DataFrame(
    candidate_pairs_by_fracturing["field_area_keys"].tolist(),
    index=candidate_pairs_by_fracturing.index
)

candidate_pairs_by_fracturing = (
    candidate_pairs_by_fracturing
    .drop(columns="field_area_keys")
)

15.2. Count how often each pair of keys occurs

In [146]:
field_area_pair_counts = (
    candidate_pairs_by_fracturing
    .groupby(
        [
            "field_area_key_1",
            "field_area_key_2"
        ]
    )
    .size()
    .reset_index(
        name="fracturing_id_count"
    )
    .sort_values(
        "fracturing_id_count",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    f"Unique field_area_key pairs: "
    f"{field_area_pair_counts.shape[0]}"
)

Unique field_area_key pairs: 77


Step 16. Characterize field-area candidate pairs

In [147]:
field_area_characterization = (
    candidate_pairs_by_fracturing
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "areapermisoconcesion",
                "yacimiento",
                "idareapermisoconcesion",
                "areayacimiento",
                "idareayacimiento"
            ]
        ].rename(
            columns={
                "field_area_key": "field_area_key_1",
                "areapermisoconcesion": "areapermisoconcesion_1",
                "yacimiento": "yacimiento_1",
                "idareapermisoconcesion": "idareapermisoconcesion_1",
                "areayacimiento": "areayacimiento_1",
                "idareayacimiento": "idareayacimiento_1"
            }
        ),
        on="field_area_key_1",
        how="left"
    )
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "areapermisoconcesion",
                "yacimiento",
                "idareapermisoconcesion",
                "areayacimiento",
                "idareayacimiento"
            ]
        ].rename(
            columns={
                "field_area_key": "field_area_key_2",
                "areapermisoconcesion": "areapermisoconcesion_2",
                "yacimiento": "yacimiento_2",
                "idareapermisoconcesion": "idareapermisoconcesion_2",
                "areayacimiento": "areayacimiento_2",
                "idareayacimiento": "idareayacimiento_2"
            }
        ),
        on="field_area_key_2",
        how="left"
    )
    .merge(
        field_area_pair_counts,
        on=[
            "field_area_key_1",
            "field_area_key_2"
        ],
        how="left"
    )
    .sort_values(
        "fracturing_id_count",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    f"Fracturing IDs characterized: "
    f"{field_area_characterization['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique field_area_key pairs characterized: "
    f"{field_area_characterization[
        ['field_area_key_1', 'field_area_key_2']
    ].drop_duplicates().shape[0]}"
)

field_area_characterization.head(10)

Fracturing IDs characterized: 3551
Unique field_area_key pairs characterized: 77


,id_base_fractura_adjiv,field_area_key_1,field_area_key_2,areapermisoconcesion_1,yacimiento_1,idareapermisoconcesion_1,areayacimiento_1,idareayacimiento_1,areapermisoconcesion_2,yacimiento_2,idareapermisoconcesion_2,areayacimiento_2,idareayacimiento_2,fracturing_id_count
0,2811,36.0,76.0,LOMA CAMPANA,None,LCAM,LOMA CAMPANA-LLL,LCLL,LOMA CAMPANA,LOMA CAMPANA-LLL,LCAM,LOMA CAMPANA-LLL,LCLL,611
1,2812,36.0,76.0,LOMA CAMPANA,None,LCAM,LOMA CAMPANA-LLL,LCLL,LOMA CAMPANA,LOMA CAMPANA-LLL,LCAM,LOMA CAMPANA-LLL,LCLL,611
2,2813,36.0,76.0,LOMA CAMPANA,None,LCAM,LOMA CAMPANA-LLL,LCLL,LOMA CAMPANA,LOMA CAMPANA-LLL,LCAM,LOMA CAMPANA-LLL,LCLL,611
3,2814,36.0,76.0,LOMA CAMPANA,None,LCAM,LOMA CAMPANA-LLL,LCLL,LOMA CAMPANA,LOMA CAMPANA-LLL,LCAM,LOMA CAMPANA-LLL,LCLL,611
4,2815,36.0,76.0,LOMA CAMPANA,None,LCAM,LOMA CAMPANA-LLL,LCLL,LOMA CAMPANA,LOMA CAMPANA-LLL,LCAM,LOMA CAMPANA-LLL,LCLL,611
5,2816,36.0,76.0,LOMA CAMPANA,None,LCAM,LOMA CAMPANA-LLL,LCLL,LOMA CAMPANA,LOMA CAMPANA-LLL,LCAM,LOMA CAMPANA-LLL,LCLL,611
6,2994,36.0,76.0,LOMA CAMPANA,None,LCAM,LOMA CAMPANA-LLL,LCLL,LOMA CAMPANA,LOMA CAMPANA-LLL,LCAM,LOMA CAMPANA-LLL,LCLL,611
7,2995,36.0,76.0,LOMA CAMPANA,None,LCAM,LOMA CAMPANA-LLL,LCLL,LOMA CAMPANA,LOMA CAMPANA-LLL,LCAM,LOMA CAMPANA-LLL,LCLL,611
8,2996,36.0,76.0,LOMA CAMPANA,None,LCAM,LOMA CAMPANA-LLL,LCLL,LOMA CAMPANA,LOMA CAMPANA-LLL,LCAM,LOMA CAMPANA-LLL,LCLL,611
9,2997,36.0,76.0,LOMA CAMPANA,None,LCAM,LOMA CAMPANA-LLL,LCLL,LOMA CAMPANA,LOMA CAMPANA-LLL,LCAM,LOMA CAMPANA-LLL,LCLL,611


Step 17. Resolve remaining field-area ambiguity using fracturing yacimiento

In [148]:
yacimiento_resolution = (
    ambiguous_candidates_supported_by_production[
        [
            "id_base_fractura_adjiv",
            "field_area_key",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
    .copy()
)

# Map each candidate field_area_key to its yacimiento
field_area_yacimiento_lookup = (
    dim_field_area[
        [
            "field_area_key",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
)

yacimiento_resolution = (
    yacimiento_resolution
    .merge(
        field_area_yacimiento_lookup,
        on="field_area_key",
        how="left",
        suffixes=(
            "_fracturing",
            "_field_area"
        )
    )
)

# Normalize text values for comparison
yacimiento_resolution["yacimiento_fracturing_norm"] = (
    yacimiento_resolution["yacimiento_fracturing"]
    .astype("string")
    .str.strip()
    .str.upper()
)

yacimiento_resolution["yacimiento_field_area_norm"] = (
    yacimiento_resolution["yacimiento_field_area"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identify candidate keys whose yacimiento agrees with fracturing
yacimiento_resolution["yacimiento_match"] = (
    yacimiento_resolution["yacimiento_fracturing_norm"]
    .notna()
    &
    yacimiento_resolution["yacimiento_field_area_norm"].notna()
    &
    (
        yacimiento_resolution["yacimiento_fracturing_norm"]
        ==
        yacimiento_resolution["yacimiento_field_area_norm"]
    )
)

# Count how many candidate keys match the fracturing yacimiento
yacimiento_match_counts = (
    yacimiento_resolution
    .groupby("id_base_fractura_adjiv")["yacimiento_match"]
    .sum()
)

print(
    f"Fracturing IDs analyzed: "
    f"{yacimiento_match_counts.shape[0]}"
)

print(
    f"Fracturing IDs with 1 yacimiento-matching candidate: "
    f"{(yacimiento_match_counts == 1).sum()}"
)

print(
    f"Fracturing IDs with 2 yacimiento-matching candidates: "
    f"{(yacimiento_match_counts == 2).sum()}"
)

print(
    f"Fracturing IDs with 0 yacimiento-matching candidates: "
    f"{(yacimiento_match_counts == 0).sum()}"
)

yacimiento_resolution.head(10)

Fracturing IDs analyzed: 3551
Fracturing IDs with 1 yacimiento-matching candidate: 3551
Fracturing IDs with 2 yacimiento-matching candidates: 0
Fracturing IDs with 0 yacimiento-matching candidates: 0


,id_base_fractura_adjiv,field_area_key,yacimiento_fracturing,yacimiento_field_area,yacimiento_fracturing_norm,yacimiento_field_area_norm,yacimiento_match
0,30,39.0,AGUA DEL CAJON,None,AGUA DEL CAJON,<NA>,False
1,30,144.0,AGUA DEL CAJON,AGUA DEL CAJON,AGUA DEL CAJON,AGUA DEL CAJON,True
2,31,39.0,AGUA DEL CAJON,None,AGUA DEL CAJON,<NA>,False
3,31,144.0,AGUA DEL CAJON,AGUA DEL CAJON,AGUA DEL CAJON,AGUA DEL CAJON,True
4,46,82.0,EL OREJANO,None,EL OREJANO,<NA>,False
5,46,133.0,EL OREJANO,EL OREJANO,EL OREJANO,EL OREJANO,True
6,47,82.0,EL OREJANO,None,EL OREJANO,<NA>,False
7,47,133.0,EL OREJANO,EL OREJANO,EL OREJANO,EL OREJANO,True
8,48,82.0,EL OREJANO,None,EL OREJANO,<NA>,False
9,48,133.0,EL OREJANO,EL OREJANO,EL OREJANO,EL OREJANO,True


Step 18. Create and validate resolved field-area lookup

In [149]:
resolved_field_area_lookup = (
    yacimiento_resolution[
        yacimiento_resolution["yacimiento_match"]
    ][
        [
            "id_base_fractura_adjiv",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Fracturing IDs resolved: "
    f"{resolved_field_area_lookup['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique field_area_keys resolved: "
    f"{resolved_field_area_lookup['field_area_key'].nunique()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{resolved_field_area_lookup['id_base_fractura_adjiv'].duplicated().sum()}"
)

print(
    f"Rows in resolved lookup: "
    f"{resolved_field_area_lookup.shape[0]}"
)

print(
    f"Missing field_area_keys: "
    f"{resolved_field_area_lookup['field_area_key'].isna().sum()}"
)

resolved_field_area_lookup.head(10)

Fracturing IDs resolved: 3551
Unique field_area_keys resolved: 77
Duplicated fracturing IDs: 0
Rows in resolved lookup: 3551
Missing field_area_keys: 0


,id_base_fractura_adjiv,field_area_key
0,30,144.0
1,31,144.0
2,46,133.0
3,47,133.0
4,48,133.0
5,49,133.0
6,50,133.0
7,51,133.0
8,52,133.0
9,53,133.0


Step 19. Validate global field-area resolution

In [150]:
resolved_field_area_ids = (
    pd.concat(
        [
            unique_field_area_matches[
                ["id_base_fractura_adjiv", "field_area_key"]
            ],
            resolved_field_area_lookup[
                ["id_base_fractura_adjiv", "field_area_key"]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

In [151]:
total_fracturing_ids = (
    fact_fracturing_df["id_base_fractura_adjiv"].nunique()
)

resolved_fracturing_ids = (
    resolved_field_area_ids["id_base_fractura_adjiv"].nunique()
)

unresolved_fracturing_ids = (
    total_fracturing_ids - resolved_fracturing_ids
)

resolution_rate = (
    resolved_fracturing_ids / total_fracturing_ids * 100
)

duplicated_resolved_ids = (
    resolved_field_area_ids["id_base_fractura_adjiv"]
    .duplicated()
    .sum()
)

missing_resolved_keys = (
    resolved_field_area_ids["field_area_key"]
    .isna()
    .sum()
)

print(
    f"Total fracturing IDs: "
    f"{total_fracturing_ids}"
)

print(
    f"Resolved fracturing IDs: "
    f"{resolved_fracturing_ids}"
)

print(
    f"Unresolved fracturing IDs: "
    f"{unresolved_fracturing_ids}"
)

print(
    f"Resolution rate: "
    f"{resolution_rate:.2f}%"
)

print(
    f"Duplicated fracturing IDs in resolved lookup: "
    f"{duplicated_resolved_ids}"
)

print(
    f"Missing field_area_keys in resolved lookup: "
    f"{missing_resolved_keys}"
)

Total fracturing IDs: 4806
Resolved fracturing IDs: 4260
Unresolved fracturing IDs: 546
Resolution rate: 88.64%
Duplicated fracturing IDs in resolved lookup: 0
Missing field_area_keys in resolved lookup: 0


Step 20. Identify unresolved fracturing records

In [152]:
unresolved_fracturing = (
    fact_fracturing_df[
        ~fact_fracturing_df["id_base_fractura_adjiv"].isin(
            resolved_field_area_ids["id_base_fractura_adjiv"]
        )
    ]
    [
        [
            "id_base_fractura_adjiv",
            "idpozo"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Unresolved fracturing IDs: "
    f"{unresolved_fracturing['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells among unresolved IDs: "
    f"{unresolved_fracturing['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{unresolved_fracturing['idpozo'].isna().sum()}"
)

unresolved_fracturing.head(10)

Unresolved fracturing IDs: 546
Unique wells among unresolved IDs: 463
Missing idpozo: 0


,id_base_fractura_adjiv,idpozo
0,893,136243
1,1177,136559
2,1178,136362
3,1179,135306
4,1180,144101
5,1181,136367
6,1182,143921
7,1183,137176
8,1184,137586
9,1185,144203


Step 21. Classify unresolved fracturing records by match status

In [153]:
unresolved_fracturing_classification = (
    unresolved_fracturing[
        ["id_base_fractura_adjiv"]
    ]
    .drop_duplicates()
    .merge(
        fracturing_field_area_classification[
            [
                "id_base_fractura_adjiv",
                "field_area_key_count",
                "match_status"
            ]
        ],
        on="id_base_fractura_adjiv",
        how="left",
        validate="one_to_one"
    )
    .reset_index(drop=True)
)

print(
    f"Total unresolved fracturing IDs: "
    f"{unresolved_fracturing_classification['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unmatched fracturing IDs: "
    f"{(
        unresolved_fracturing_classification["match_status"]
        == "unmatched"
    ).sum()}"
)

print(
    f"Ambiguous fracturing IDs: "
    f"{(
        unresolved_fracturing_classification["match_status"]
        == "ambiguous"
    ).sum()}"
)

print(
    f"Missing match status: "
    f"{unresolved_fracturing_classification['match_status'].isna().sum()}"
)

unresolved_fracturing_classification.head(10)

Total unresolved fracturing IDs: 546
Unmatched fracturing IDs: 282
Ambiguous fracturing IDs: 264
Missing match status: 0


,id_base_fractura_adjiv,field_area_key_count,match_status
0,893,0,unmatched
1,1177,2,ambiguous
2,1178,2,ambiguous
3,1179,2,ambiguous
4,1180,2,ambiguous
5,1181,2,ambiguous
6,1182,2,ambiguous
7,1183,2,ambiguous
8,1184,2,ambiguous
9,1185,2,ambiguous


Step 22. Investigate unmatched fracturing records

In [154]:
unmatched_fracturing_records = (
    unresolved_fracturing_classification[
        unresolved_fracturing_classification["match_status"] == "unmatched"
    ][
        ["id_base_fractura_adjiv"]
    ]
    .merge(
        fracture_new_DF[
            [
                "id_base_fractura_adjiv",
                "idpozo",
                "areapermisoconcesion",
                "yacimiento",
                "cuenca",
                "formacion_productiva",
                "tipo_reservorio",
                "subtipo_reservorio"
            ]
        ],
        on="id_base_fractura_adjiv",
        how="left",
        validate="one_to_one"
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Unmatched fracturing IDs: "
    f"{unmatched_fracturing_records['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells: "
    f"{unmatched_fracturing_records['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{unmatched_fracturing_records['idpozo'].isna().sum()}"
)

print(
    f"Missing areapermisoconcesion: "
    f"{unmatched_fracturing_records['areapermisoconcesion'].isna().sum()}"
)

print(
    f"Missing yacimiento: "
    f"{unmatched_fracturing_records['yacimiento'].isna().sum()}"
)

print(
    f"Missing cuenca: "
    f"{unmatched_fracturing_records['cuenca'].isna().sum()}"
)

unmatched_fracturing_records.head(10)

Unmatched fracturing IDs: 282
Unique wells: 211
Missing idpozo: 0
Missing areapermisoconcesion: 0
Missing yacimiento: 0
Missing cuenca: 0


,id_base_fractura_adjiv,idpozo,areapermisoconcesion,yacimiento,cuenca,formacion_productiva,tipo_reservorio,subtipo_reservorio
0,893,136243,SANTA CRUZ I - FRACCION C,LAGUNA DE LOS CAPONES NORTE,AUSTRAL,springhill,NaN,NaN
1,1860,158603,CN-V,CN-V,NEUQUINA,grupo neuquén,CONVENCIONAL,NaN
2,1902,160305,AGUA SALADA,LOMA AZUL,NEUQUINA,precuyo,CONVENCIONAL,NaN
3,1907,135444,AGUA SALADA,LOMA AZUL,NEUQUINA,los molles,CONVENCIONAL,NaN
4,1910,129310,AGUA SALADA,LOMA AZUL,NEUQUINA,los molles,CONVENCIONAL,NaN
5,1911,155092,AGUA SALADA,LOMA AZUL,NEUQUINA,los molles,CONVENCIONAL,NaN
6,1916,160580,AGUA SALADA,PALENQUE DE LA CHINA,NEUQUINA,los molles,CONVENCIONAL,NaN
7,1922,458,ESTANCIA LA MARIPOSA,ESTANCIA LA MARIPOSA,GOLFO SAN JORGE,comodoro rivadavia,CONVENCIONAL,NaN
8,1927,8308,PUESTO QUIROGA,PUESTO QUIROGA,GOLFO SAN JORGE,mina el carmen,CONVENCIONAL,NaN
9,1933,134474,LOS BASTOS,LOS BASTOS SUR,NEUQUINA,loma montosa,CONVENCIONAL,NaN


Step 23. Diagnose unmatched field-area components

In [155]:
unmatched_component_analysis = (
    unmatched_fracturing_records[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
)

23.1. Match by concession area only

In [156]:
area_matches = (
    unmatched_component_analysis[
        ["areapermisoconcesion"]
    ]
    .drop_duplicates()
    .merge(
        dim_field_area[
            [
                "areapermisoconcesion",
                "field_area_key"
            ]
        ].drop_duplicates(),
        on="areapermisoconcesion",
        how="left"
    )
)

area_match_keys = (
    area_matches
    .groupby("areapermisoconcesion")["field_area_key"]
    .nunique()
)

23.2. Match by yacimiento only

In [157]:
yacimiento_matches = (
    unmatched_component_analysis[
        ["yacimiento"]
    ]
    .drop_duplicates()
    .merge(
        dim_field_area[
            [
                "areayacimiento",
                "field_area_key"
            ]
        ].drop_duplicates(),
        left_on="yacimiento",
        right_on="areayacimiento",
        how="left"
    )
)

yacimiento_match_keys = (
    yacimiento_matches
    .groupby("yacimiento")["field_area_key"]
    .nunique()
)

23.3. Classify each unmatched fracturing ID

In [158]:
unmatched_component_analysis["area_match_count"] = (
    unmatched_component_analysis["areapermisoconcesion"]
    .map(area_match_keys)
    .fillna(0)
)

unmatched_component_analysis["yacimiento_match_count"] = (
    unmatched_component_analysis["yacimiento"]
    .map(yacimiento_match_keys)
    .fillna(0)
)

unmatched_component_analysis["component_match_status"] = (
    unmatched_component_analysis.apply(
        lambda row: (
            "area_and_yacimiento"
            if row["area_match_count"] > 0
            and row["yacimiento_match_count"] > 0
            else
            "area_only"
            if row["area_match_count"] > 0
            else
            "yacimiento_only"
            if row["yacimiento_match_count"] > 0
            else
            "no_component_match"
        ),
        axis=1
    )
)

23.4. Summary

In [159]:
print(
    f"Unmatched fracturing IDs analyzed: "
    f"{unmatched_component_analysis['id_base_fractura_adjiv'].nunique()}"
)

print("-" * 35)

print("Component match classification:")

print(
    unmatched_component_analysis[
        "component_match_status"
    ].value_counts()
)

print("-" * 35)

print(
    f"IDs with area match: "
    f"{(unmatched_component_analysis['area_match_count'] > 0).sum()}"
)

print(
    f"IDs with yacimiento match: "
    f"{(unmatched_component_analysis['yacimiento_match_count'] > 0).sum()}"
)

print(
    f"IDs with no component match: "
    f"{(
        (unmatched_component_analysis['area_match_count'] == 0)
        &
        (unmatched_component_analysis['yacimiento_match_count'] == 0)
    ).sum()}"
)

unmatched_component_analysis.head(10)

Unmatched fracturing IDs analyzed: 282
-----------------------------------
Component match classification:
component_match_status
no_component_match    270
area_only              12
Name: count, dtype: int64
-----------------------------------
IDs with area match: 12
IDs with yacimiento match: 0
IDs with no component match: 270


,id_base_fractura_adjiv,idpozo,areapermisoconcesion,yacimiento,area_match_count,yacimiento_match_count,component_match_status
0,893,136243,SANTA CRUZ I - FRACCION C,LAGUNA DE LOS CAPONES NORTE,0,0,no_component_match
1,1860,158603,CN-V,CN-V,0,0,no_component_match
2,1902,160305,AGUA SALADA,LOMA AZUL,0,0,no_component_match
3,1907,135444,AGUA SALADA,LOMA AZUL,0,0,no_component_match
4,1910,129310,AGUA SALADA,LOMA AZUL,0,0,no_component_match
5,1911,155092,AGUA SALADA,LOMA AZUL,0,0,no_component_match
6,1916,160580,AGUA SALADA,PALENQUE DE LA CHINA,0,0,no_component_match
7,1922,458,ESTANCIA LA MARIPOSA,ESTANCIA LA MARIPOSA,0,0,no_component_match
8,1927,8308,PUESTO QUIROGA,PUESTO QUIROGA,0,0,no_component_match
9,1933,134474,LOS BASTOS,LOS BASTOS SUR,0,0,no_component_match


Step 24. Investigate unmatched fracturing IDs through production history

In [160]:
unmatched_wells = (
    unmatched_fracturing_records[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

unmatched_production_history = (
    production_df[
        production_df["idpozo"].isin(
            unmatched_wells["idpozo"]
        )
    ][
        [
            "idpozo",
            "idareapermisoconcesion",
            "areapermisoconcesion",
            "idareayacimiento",
            "areayacimiento"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "idpozo",
            "idareapermisoconcesion",
            "idareayacimiento"
        ]
    )
    .reset_index(drop=True)
)

print(
    f"Unmatched fracturing IDs: "
    f"{unmatched_wells['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells among unmatched IDs: "
    f"{unmatched_wells['idpozo'].nunique()}"
)

print(
    f"Wells with production history: "
    f"{unmatched_production_history['idpozo'].nunique()}"
)

print(
    f"Production history rows: "
    f"{unmatched_production_history.shape[0]}"
)

print(
    f"Missing idpozo in production history: "
    f"{unmatched_production_history['idpozo'].isna().sum()}"
)

unmatched_production_history.head(10)

Unmatched fracturing IDs: 282
Unique wells among unmatched IDs: 211
Wells with production history: 0
Production history rows: 0
Missing idpozo in production history: 0


,idpozo,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento


Step 25. Investigate unresolved ambiguous field-area assignments

In [161]:
unresolved_ambiguous_ids = (
    fracturing_field_area_classification[
        fracturing_field_area_classification["match_status"] == "ambiguous"
    ][
        ["id_base_fractura_adjiv"]
    ]
)

25.1. Keep only ambiguous IDs that are still unresolved

In [162]:
resolved_ids = (
    resolved_field_area_lookup[
        ["id_base_fractura_adjiv"]
    ]
    .drop_duplicates()
)

unresolved_ambiguous_ids = (
    unresolved_ambiguous_ids[
        ~unresolved_ambiguous_ids["id_base_fractura_adjiv"].isin(
            resolved_ids["id_base_fractura_adjiv"]
        )
    ]
    .drop_duplicates()
)

25.2. Retrieve candidate field-area assignments

In [163]:
unresolved_ambiguous_records = (
    ambiguous_field_area_records[
        ambiguous_field_area_records[
            "id_base_fractura_adjiv"
        ].isin(
            unresolved_ambiguous_ids["id_base_fractura_adjiv"]
        )
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento",
            "field_area_key",
            "areayacimiento"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "id_base_fractura_adjiv",
            "field_area_key"
        ]
    )
    .reset_index(drop=True)
)

25.3. Count candidate field_area_keys per fracturing ID

In [164]:
unresolved_ambiguous_counts = (
    unresolved_ambiguous_records
    .groupby("id_base_fractura_adjiv")["field_area_key"]
    .nunique()
)

25.4. Summary

In [165]:
print(
    f"Unresolved ambiguous fracturing IDs: "
    f"{unresolved_ambiguous_counts.shape[0]}"
)

print(
    f"IDs with exactly 2 candidate field_area_keys: "
    f"{(unresolved_ambiguous_counts == 2).sum()}"
)

print(
    f"IDs with more than 2 candidate field_area_keys: "
    f"{(unresolved_ambiguous_counts > 2).sum()}"
)

print(
    f"Maximum candidate field_area_keys: "
    f"{unresolved_ambiguous_counts.max()}"
)

print(
    f"Unique wells among unresolved ambiguous IDs: "
    f"{unresolved_ambiguous_records['idpozo'].nunique()}"
)

print("-" * 35)

print("Candidate field_area_key frequency:")

print(
    unresolved_ambiguous_records[
        "field_area_key"
    ]
    .value_counts()
    .sort_index()
)

print("-" * 35)

unresolved_ambiguous_records.head(10)

Unresolved ambiguous fracturing IDs: 264
IDs with exactly 2 candidate field_area_keys: 262
IDs with more than 2 candidate field_area_keys: 2
Maximum candidate field_area_keys: 4
Unique wells among unresolved ambiguous IDs: 252
-----------------------------------
Candidate field_area_key frequency:
field_area_key
4.0        1
12.0       2
13.0       3
24.0       2
30.0      47
39.0      10
40.0       1
52.0       1
55.0       8
56.0       8
72.0      47
108.0    175
117.0     11
120.0      1
126.0     11
143.0      3
144.0     10
175.0      1
189.0      1
196.0      1
206.0      1
208.0      2
228.0      2
233.0    175
234.0      1
240.0      2
253.0      2
259.0      1
268.0      1
270.0      1
Name: count, dtype: int64
-----------------------------------


,id_base_fractura_adjiv,idpozo,areapermisoconcesion,yacimiento,field_area_key,areayacimiento
0,1177,136559,EL TRAPIAL ESTE,EL TRAPIAL,108.0,EL TRAPIAL
1,1177,136559,EL TRAPIAL ESTE,EL TRAPIAL,233.0,EL TRAPIAL
2,1178,136362,EL TRAPIAL ESTE,EL TRAPIAL,108.0,EL TRAPIAL
3,1178,136362,EL TRAPIAL ESTE,EL TRAPIAL,233.0,EL TRAPIAL
4,1179,135306,EL TRAPIAL ESTE,EL TRAPIAL,108.0,EL TRAPIAL
5,1179,135306,EL TRAPIAL ESTE,EL TRAPIAL,233.0,EL TRAPIAL
6,1180,144101,EL TRAPIAL ESTE,EL TRAPIAL,108.0,EL TRAPIAL
7,1180,144101,EL TRAPIAL ESTE,EL TRAPIAL,233.0,EL TRAPIAL
8,1181,136367,EL TRAPIAL ESTE,EL TRAPIAL,108.0,EL TRAPIAL
9,1181,136367,EL TRAPIAL ESTE,EL TRAPIAL,233.0,EL TRAPIAL


Step 26. Characterize unresolved ambiguous field-area candidates

In [166]:
unresolved_candidate_keys = (
    unresolved_ambiguous_records[
        ["field_area_key"]
    ]
    .drop_duplicates()
)

unresolved_candidate_profiles = (
    unresolved_candidate_keys
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "idareapermisoconcesion",
                "areapermisoconcesion",
                "idareayacimiento",
                "areayacimiento",
                "yacimiento"
            ]
        ],
        on="field_area_key",
        how="left"
    )
    .sort_values("field_area_key")
    .reset_index(drop=True)
)

print(
    f"Unique candidate field_area_keys: "
    f"{unresolved_candidate_profiles['field_area_key'].nunique()}"
)

print(
    f"Missing idareapermisoconcesion: "
    f"{unresolved_candidate_profiles['idareapermisoconcesion'].isna().sum()}"
)

print(
    f"Missing areapermisoconcesion: "
    f"{unresolved_candidate_profiles['areapermisoconcesion'].isna().sum()}"
)

print(
    f"Missing idareayacimiento: "
    f"{unresolved_candidate_profiles['idareayacimiento'].isna().sum()}"
)

print(
    f"Missing areayacimiento: "
    f"{unresolved_candidate_profiles['areayacimiento'].isna().sum()}"
)

print()

unresolved_candidate_profiles

Unique candidate field_area_keys: 30
Missing idareapermisoconcesion: 0
Missing areapermisoconcesion: 0
Missing idareayacimiento: 0
Missing areayacimiento: 0



,field_area_key,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,yacimiento
0,4.0,FOR,FORTIN DE PIEDRA,FOR,FORTIN DE PIEDRA,None
1,12.0,CIEC,CAMPO INDIO ESTE - EL CERRITO,INDI,CAMPO INDIO,CAMPO INDIO
2,13.0,PUSE,PUNTA SENILLOSA,PSNC,PUNTA SENILLOSA NC,None
3,24.0,CIEC,CAMPO INDIO ESTE - EL CERRITO,INDI,CAMPO INDIO,None
4,30.0,CAO,AGUA DEL CAJON,ELSA,EL SALITRAL,None
5,39.0,CAO,AGUA DEL CAJON,CAO,AGUA DEL CAJON,None
6,40.0,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,None
7,52.0,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,ESTACION FERNANDEZ ORO
8,55.0,APE,AGUADA PICHANA ESTE,APEM,AGUADA PICHANA ESTE MULICHINCO,AGUADA PICHANA ESTE MULICHINCO
9,56.0,APE,AGUADA PICHANA ESTE,APEM,AGUADA PICHANA ESTE MULICHINCO,None


Step 27. Identify equivalent field-area keys among ambiguous candidates

In [167]:
step26_candidates = (
    unresolved_ambiguous_records[
        unresolved_ambiguous_records["field_area_key"].notna()
    ][
        [
            "id_base_fractura_adjiv",
            "field_area_key",
            "areapermisoconcesion",
            "areayacimiento",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["id_base_fractura_adjiv", "field_area_key"]
    )
    .reset_index(drop=True)
)

print(
    "Ambiguous fracturing IDs analyzed:",
    step26_candidates["id_base_fractura_adjiv"].nunique()
)

print(
    "Unique candidate field_area_keys:",
    step26_candidates["field_area_key"].nunique()
)

print(
    "Unique logical area-yacimiento combinations:",
    step26_candidates[
        ["areapermisoconcesion", "areayacimiento"]
    ].drop_duplicates().shape[0]
)

print("-" * 35)

field_area_equivalence = (
    step26_candidates[
        [
            "field_area_key",
            "areapermisoconcesion",
            "areayacimiento",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "areapermisoconcesion",
            "areayacimiento",
            "field_area_key"
        ]
    )
    .reset_index(drop=True)
)

print("Field-area key equivalence candidates:")

display(field_area_equivalence)

Ambiguous fracturing IDs analyzed: 264
Unique candidate field_area_keys: 30
Unique logical area-yacimiento combinations: 14
-----------------------------------
Field-area key equivalence candidates:


,field_area_key,areapermisoconcesion,areayacimiento,yacimiento
0,39.0,AGUA DEL CAJON,AGUA DEL CAJON,AGUA DEL CAJON
1,144.0,AGUA DEL CAJON,AGUA DEL CAJON,AGUA DEL CAJON
2,30.0,AGUA DEL CAJON,EL SALITRAL,EL SALITRAL
3,72.0,AGUA DEL CAJON,EL SALITRAL,EL SALITRAL
4,55.0,AGUADA PICHANA ESTE,AGUADA PICHANA ESTE MULICHINCO,AGUADA PICHANA ESTE MULICHINCO
5,56.0,AGUADA PICHANA ESTE,AGUADA PICHANA ESTE MULICHINCO,AGUADA PICHANA ESTE MULICHINCO
6,120.0,AGUADA PICHANA OESTE,AGUADA PICHANA OESTE,AGUADA PICHANA OESTE
7,196.0,AGUADA PICHANA OESTE,AGUADA PICHANA OESTE,AGUADA PICHANA OESTE
8,234.0,BAJADA DEL PALO OESTE,BAJADA DEL PALO OESTE,BAJADA DEL PALO OESTE
9,268.0,BAJADA DEL PALO OESTE,BAJADA DEL PALO OESTE,BAJADA DEL PALO OESTE


Step 28. Determine whether ambiguous keys are logically equivalent

28.1. Create a logical combination for each candidate

In [168]:
step27_candidates = (
    step26_candidates
    .copy()
)

step27_candidates["logical_area_yacimiento"] = (
    step27_candidates["areapermisoconcesion"].fillna("").astype(str).str.strip()
    + " | "
    + step27_candidates["areayacimiento"].fillna("").astype(str).str.strip()
)

28.2. Count distinct logical combinations for each fracturing ID

In [169]:
step27_classification = (
    step27_candidates
    .groupby("id_base_fractura_adjiv")
    .agg(
        candidate_field_area_keys=(
            "field_area_key",
            "nunique"
        ),
        logical_combinations=(
            "logical_area_yacimiento",
            "nunique"
        )
    )
    .reset_index()
)

28.3. Classify the ambiguity

In [170]:
step27_classification["logical_match_status"] = (
    step27_classification["logical_combinations"]
    .map(
        lambda x:
            "equivalent_candidates"
            if x == 1
            else "different_candidates"
            if x > 1
            else "no_candidate"
    )
)

28.4. Summary

In [171]:
print(
    "Ambiguous fracturing IDs analyzed:",
    step27_classification["id_base_fractura_adjiv"].nunique()
)

print("-" * 35)

print("Logical match classification:")

print(
    step27_classification[
        "logical_match_status"
    ].value_counts()
)

print("-" * 35)

print(
    "IDs with equivalent candidates:",
    (
        step27_classification["logical_match_status"]
        == "equivalent_candidates"
    ).sum()
)

print(
    "IDs with different candidates:",
    (
        step27_classification["logical_match_status"]
        == "different_candidates"
    ).sum()
)

print(
    "IDs with no candidate:",
    (
        step27_classification["logical_match_status"]
        == "no_candidate"
    ).sum()
)

print("-" * 35)

print("Classification detail:")

display(
    step27_classification
    .sort_values(
        [
            "logical_match_status",
            "id_base_fractura_adjiv"
        ]
    )
    .reset_index(drop=True)
)

Ambiguous fracturing IDs analyzed: 264
-----------------------------------
Logical match classification:
logical_match_status
equivalent_candidates    264
Name: count, dtype: int64
-----------------------------------
IDs with equivalent candidates: 264
IDs with different candidates: 0
IDs with no candidate: 0
-----------------------------------
Classification detail:


,id_base_fractura_adjiv,candidate_field_area_keys,logical_combinations,logical_match_status
0,1177,2,1,equivalent_candidates
1,1178,2,1,equivalent_candidates
2,1179,2,1,equivalent_candidates
3,1180,2,1,equivalent_candidates
4,1181,2,1,equivalent_candidates
...,...,...,...,...
259,3807,4,1,equivalent_candidates
260,3809,2,1,equivalent_candidates
261,4096,2,1,equivalent_candidates
262,4334,2,1,equivalent_candidates


Step 29. Investigate canonical field-area key among equivalent candidates

In [172]:
step29_candidates = (
    step26_candidates[
        step26_candidates["field_area_key"].notna()
    ][
        [
            "field_area_key",
            "areapermisoconcesion",
            "areayacimiento",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "areapermisoconcesion",
            "areayacimiento",
            "field_area_key"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Unique candidate field_area_keys:",
    step29_candidates["field_area_key"].nunique()
)

print(
    "Unique logical area-yacimiento combinations:",
    step29_candidates[
        [
            "areapermisoconcesion",
            "areayacimiento",
            "yacimiento"
        ]
    ].drop_duplicates().shape[0]
)

Unique candidate field_area_keys: 30
Unique logical area-yacimiento combinations: 14


29.1. Count candidate keys per logical combination

In [173]:
step29_summary = (
    step29_candidates
    .groupby(
        [
            "areapermisoconcesion",
            "areayacimiento",
            "yacimiento"
        ],
        dropna=False
    )
    .agg(
        candidate_field_area_keys=("field_area_key", "nunique"),
        candidate_keys=("field_area_key", list)
    )
    .reset_index()
)

print("-" * 35)
print("Logical combinations with candidate keys:")
print(
    step29_summary[
        "candidate_field_area_keys"
    ].value_counts().sort_index()
)

print("-" * 35)
print("Logical combinations with more than one candidate:")
print(
    step29_summary[
        step29_summary["candidate_field_area_keys"] > 1
    ]
)

print("-" * 35)
print("Full candidate mapping:")
print(step29_candidates)

-----------------------------------
Logical combinations with candidate keys:
candidate_field_area_keys
2    13
4     1
Name: count, dtype: int64
-----------------------------------
Logical combinations with more than one candidate:
              areapermisoconcesion                  areayacimiento  \
0                   AGUA DEL CAJON                  AGUA DEL CAJON   
1                   AGUA DEL CAJON                     EL SALITRAL   
2            AGUADA PICHANA ESTE    AGUADA PICHANA ESTE MULICHINCO   
3            AGUADA PICHANA OESTE             AGUADA PICHANA OESTE   
4            BAJADA DEL PALO OESTE           BAJADA DEL PALO OESTE   
5   CAMPO INDIO ESTE - EL CERRITO                      CAMPO INDIO   
6   CAMPO INDIO ESTE - EL CERRITO                 EL CERRITO OESTE   
7                     CAÑADON SECO                    CAÑADON SECO   
8                  EL TRAPIAL ESTE                      EL TRAPIAL   
9           ESTACION FERNANDEZ ORO          ESTACION FERNANDEZ ORO 

Step 30. Search for additional well-level evidence

30.1. Wells associated with unresolved ambiguous fracturing IDs

In [174]:
step30_wells = (
    unresolved_ambiguous_records[
        unresolved_ambiguous_records["field_area_key"].notna()
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo"
        ]
    ]
    .drop_duplicates()
)

30.2. Resolved fracturing IDs with their corresponding wells

In [175]:
step30_resolved_wells = (
    fact_fracturing_df[
        fact_fracturing_df["id_base_fractura_adjiv"].isin(
            resolved_field_area_ids["id_base_fractura_adjiv"]
        )
    ][
        [
            "idpozo",
            "id_base_fractura_adjiv"
        ]
    ]
    .drop_duplicates()
)

30.3. Associate each well with the field_area_keys already resolved

In [176]:
step30_well_evidence = (
    step30_resolved_wells
    .merge(
        resolved_field_area_ids[
            [
                "id_base_fractura_adjiv",
                "field_area_key"
            ]
        ],
        on="id_base_fractura_adjiv",
        how="inner"
    )
    [
        [
            "idpozo",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
)

30.4. Keep only wells from the unresolved ambiguous cases

In [177]:
step30_well_evidence = (
    step30_well_evidence
    .merge(
        step30_wells,
        on="idpozo",
        how="inner"
    )
    .drop_duplicates()
)

30.5. Number of resolved field_area_keys observed for each unresolved well

In [178]:
step30_summary = (
    step30_well_evidence
    .groupby("idpozo")["field_area_key"]
    .nunique()
    .reset_index(name="resolved_field_area_key_count")
)

print(
    "Unresolved ambiguous fracturing IDs:",
    step30_wells["id_base_fractura_adjiv"].nunique()
)

print(
    "Unique wells among unresolved ambiguous IDs:",
    step30_wells["idpozo"].nunique()
)

print(
    "Wells with resolved field_area_key evidence:",
    step30_summary["idpozo"].nunique()
)

print(
    "Wells without resolved field_area_key evidence:",
    step30_wells["idpozo"].nunique()
    - step30_summary["idpozo"].nunique()
)

print("-" * 35)
print("Resolved field_area_key evidence per well:")
print(
    step30_summary[
        "resolved_field_area_key_count"
    ].value_counts().sort_index()
)

print("-" * 35)
print("Well-level evidence:")
print(
    step30_well_evidence.sort_values(
        ["idpozo", "field_area_key"]
    )
)

Unresolved ambiguous fracturing IDs: 264
Unique wells among unresolved ambiguous IDs: 252
Wells with resolved field_area_key evidence: 0
Wells without resolved field_area_key evidence: 252
-----------------------------------
Resolved field_area_key evidence per well:
Series([], Name: count, dtype: int64)
-----------------------------------
Well-level evidence:
Empty DataFrame
Columns: [idpozo, field_area_key, id_base_fractura_adjiv]
Index: []


Step 31. Final validation of field_area_key resolution

31.1. Total fracturing IDs

In [179]:
total_fracturing_ids = (
    fact_fracturing_df["id_base_fractura_adjiv"]
    .nunique()
)

31.2. Resolved fracturing IDs

In [180]:
resolved_fracturing_ids_count = (
    resolved_field_area_ids["id_base_fractura_adjiv"]
    .nunique()
)

31.3. Rebuild unresolved IDs

In [181]:
all_fracturing_ids = set(
    fact_fracturing_df["id_base_fractura_adjiv"]
    .dropna()
    .unique()
)

resolved_id_set = set(
    resolved_field_area_ids["id_base_fractura_adjiv"]
    .dropna()
    .unique()
)

unresolved_id_set = (
    all_fracturing_ids - resolved_id_set
)

31.4. Rebuild unresolved records

In [182]:
step31_unresolved = (
    fact_fracturing_df[
        fact_fracturing_df["id_base_fractura_adjiv"].isin(
            unresolved_id_set
        )
    ]
    .copy()
)

31.5. Classify unmatched IDs

In [183]:
step31_unmatched_ids = (
    set(
        step31_unresolved[
            step31_unresolved["id_base_fractura_adjiv"].isin(
                unresolved_id_set
            )
        ]["id_base_fractura_adjiv"]
        .unique()
    )
)

31.6. Use the classification from Step 21

In [184]:
# Rebuild match status using candidate count
# from unresolved_ambiguous_records when available

step31_ambiguous_ids = set(
    unresolved_ambiguous_records[
        "id_base_fractura_adjiv"
    ]
    .dropna()
    .unique()
)

step31_ambiguous_ids = (
    step31_ambiguous_ids & unresolved_id_set
)

step31_unmatched_ids = (
    unresolved_id_set - step31_ambiguous_ids
)

31.7. Final counts

In [185]:
unresolved_fracturing_ids_count = len(
    unresolved_id_set
)

unmatched_ids_count = len(
    step31_unmatched_ids
)

ambiguous_ids_count = len(
    step31_ambiguous_ids
)

duplicated_resolved_ids = (
    resolved_field_area_ids[
        "id_base_fractura_adjiv"
    ]
    .duplicated()
    .sum()
)

missing_resolved_keys = (
    resolved_field_area_ids[
        "field_area_key"
    ]
    .isna()
    .sum()
)

31.8. Print final validation

In [186]:
print("===================================")
print("FINAL FIELD_AREA_KEY VALIDATION")
print("===================================")

print(
    "Total fracturing IDs:",
    total_fracturing_ids
)

print(
    "Resolved fracturing IDs:",
    resolved_fracturing_ids_count
)

print(
    "Unresolved fracturing IDs:",
    unresolved_fracturing_ids_count
)

print("-" * 35)

print(
    "Unmatched fracturing IDs:",
    unmatched_ids_count
)

print(
    "Ambiguous fracturing IDs:",
    ambiguous_ids_count
)

print("-" * 35)

print(
    "Resolved + unresolved:",
    resolved_fracturing_ids_count
    + unresolved_fracturing_ids_count
)

print(
    "Unmatched + ambiguous:",
    unmatched_ids_count
    + ambiguous_ids_count
)

print("-" * 35)

print(
    "Duplicated fracturing IDs in resolved lookup:",
    duplicated_resolved_ids
)

print(
    "Missing field_area_keys in resolved lookup:",
    missing_resolved_keys
)

print("-" * 35)

print(
    "Resolution rate:",
    f"{resolved_fracturing_ids_count / total_fracturing_ids * 100:.2f}%"
)

print(
    "Unresolved rate:",
    f"{unresolved_fracturing_ids_count / total_fracturing_ids * 100:.2f}%"
)

FINAL FIELD_AREA_KEY VALIDATION
Total fracturing IDs: 4806
Resolved fracturing IDs: 4260
Unresolved fracturing IDs: 546
-----------------------------------
Unmatched fracturing IDs: 282
Ambiguous fracturing IDs: 264
-----------------------------------
Resolved + unresolved: 4806
Unmatched + ambiguous: 546
-----------------------------------
Duplicated fracturing IDs in resolved lookup: 0
Missing field_area_keys in resolved lookup: 0
-----------------------------------
Resolution rate: 88.64%
Unresolved rate: 11.36%


#### 2.6 Lookup location_key

Step 1. Check source columns for dim_location

In [187]:
location_source_columns = [
    "idpozo",
    "cuenca",
    "provincia",
    "coordenadax",
    "coordenaday"
]

missing_columns = [
    col
    for col in location_source_columns
    if col not in production_df.columns
]

print("Required columns:")
print(location_source_columns)

print("\nMissing columns:")
print(missing_columns)

Required columns:
['idpozo', 'cuenca', 'provincia', 'coordenadax', 'coordenaday']

Missing columns:
[]


Step 2. Audit location source data

In [188]:
print("Rows in production_df:", production_df.shape[0])

print("\nData types:")
print(
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ].dtypes
)

print("\nMissing values:")
print(
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ].isna().sum()
)

Rows in production_df: 415903

Data types:
idpozo           int64
cuenca          object
provincia       object
coordenadax    float64
coordenaday    float64
dtype: object

Missing values:
idpozo         0
cuenca         0
provincia      0
coordenadax    0
coordenaday    0
dtype: int64


Step 3. Check geographic consistency by well

In [189]:
location_profile_counts = (
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .groupby("idpozo")
    .size()
    .reset_index(name="location_profiles")
)

print(
    f"Unique wells: "
    f"{location_profile_counts['idpozo'].nunique()}"
)

print(
    f"Wells with exactly one location profile: "
    f"{
        (
            location_profile_counts["location_profiles"] == 1
        ).sum()
    }"
)

print(
    f"Wells with more than one location profile: "
    f"{
        (
            location_profile_counts["location_profiles"] > 1
        ).sum()
    }"
)

print(
    f"Maximum location profiles for a single well: "
    f"{location_profile_counts['location_profiles'].max()}"
)

Unique wells: 5041
Wells with exactly one location profile: 5041
Wells with more than one location profile: 0
Maximum location profiles for a single well: 1


Step 4. Count unique geographic profiles

In [190]:
location_source = (
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
)

unique_location_profiles = (
    location_source[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
)

print(
    f"Rows in location source: "
    f"{location_source.shape[0]}"
)

print(
    f"Unique wells: "
    f"{location_source['idpozo'].nunique()}"
)

print(
    f"Unique location profiles: "
    f"{unique_location_profiles.shape[0]}"
)

Rows in location source: 5041
Unique wells: 5041
Unique location profiles: 4845


Step 5. Check wells sharing the same location

In [191]:
location_well_counts = (
    location_source
    .groupby(
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    )
    .agg(
        unique_wells=("idpozo", "nunique")
    )
    .reset_index()
)

print(
    f"Unique location profiles: "
    f"{location_well_counts.shape[0]}"
)

print(
    f"Locations shared by multiple wells: "
    f"{
        (
            location_well_counts["unique_wells"] > 1
        ).sum()
    }"
)

print(
    f"Maximum wells sharing one location: "
    f"{location_well_counts['unique_wells'].max()}"
)

Unique location profiles: 4845
Locations shared by multiple wells: 179
Maximum wells sharing one location: 4


In [192]:
shared_locations = (
    location_well_counts[
        location_well_counts["unique_wells"] > 1
    ]
    .sort_values(
        "unique_wells",
        ascending=False
    )
)

print(
    shared_locations.head(10).to_string(index=False)
)

  cuenca provincia  coordenadax  coordenaday  unique_wells
NEUQUINA   Neuquén   -69.345480   -37.470910             4
NEUQUINA   Mendoza   -69.712860   -35.260990             3
NEUQUINA   Mendoza   -69.720910   -35.259870             3
NEUQUINA   Neuquén   -69.255971   -38.860779             3
NEUQUINA   Mendoza   -69.718340   -35.259280             3
NEUQUINA   Neuquén   -68.226071   -38.923867             3
NEUQUINA   Neuquén   -68.208912   -38.715831             3
NEUQUINA   Neuquén   -68.896696   -38.753551             3
NEUQUINA   Neuquén   -68.199241   -38.935584             3
NEUQUINA   Neuquén   -68.208070   -38.922870             3


Step 6. Build unique location lookup

In [193]:
location_lookup = (
    production_df[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in location lookup: "
    f"{location_lookup.shape[0]}"
)

print(
    f"Duplicated location profiles: "
    f"{location_lookup.duplicated().sum()}"
)

print(
    f"Missing cuenca: "
    f"{location_lookup['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{location_lookup['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{location_lookup['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{location_lookup['coordenaday'].isna().sum()}"
)

Rows in location lookup: 4845
Duplicated location profiles: 0
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0


Step 7. Compare fracturing wells against dim_well

In [194]:
fracturing_well_keys = (
    fact_fracturing_df["well_key"]
    .dropna()
    .unique()
)

dim_well_keys = (
    dim_well["well_key"]
    .dropna()
    .unique()
)

fracturing_well_keys_set = set(fracturing_well_keys)
dim_well_keys_set = set(dim_well_keys)

common_well_keys = (
    fracturing_well_keys_set
    & dim_well_keys_set
)

fracturing_only_well_keys = (
    fracturing_well_keys_set
    - dim_well_keys_set
)

print(
    f"Fracturing unique well_keys: "
    f"{len(fracturing_well_keys_set)}"
)

print(
    f"dim_well unique well_keys: "
    f"{len(dim_well_keys_set)}"
)

print(
    f"Common well_keys: "
    f"{len(common_well_keys)}"
)

print(
    f"Fracturing well_keys NOT in dim_well: "
    f"{len(fracturing_only_well_keys)}"
)

Fracturing unique well_keys: 4562
dim_well unique well_keys: 5838
Common well_keys: 4562
Fracturing well_keys NOT in dim_well: 0


Step 8. Check fracturing wells against production location source

In [195]:
fracturing_well_ids = (
    fact_fracturing_df["idpozo"]
    .dropna()
    .unique()
)

production_well_ids = (
    production_df["idpozo"]
    .dropna()
    .unique()
)

fracturing_well_ids_set = set(fracturing_well_ids)
production_well_ids_set = set(production_well_ids)

common_well_ids = (
    fracturing_well_ids_set
    & production_well_ids_set
)

fracturing_only_well_ids = (
    fracturing_well_ids_set
    - production_well_ids_set
)

print(
    f"Fracturing unique idpozo: "
    f"{len(fracturing_well_ids_set)}"
)

print(
    f"Production unique idpozo: "
    f"{len(production_well_ids_set)}"
)

print(
    f"Common idpozo: "
    f"{len(common_well_ids)}"
)

print(
    f"Fracturing idpozo NOT in production: "
    f"{len(fracturing_only_well_ids)}"
)

Fracturing unique idpozo: 4562
Production unique idpozo: 5041
Common idpozo: 3765
Fracturing idpozo NOT in production: 797


Step 9. Inspect fracturing-only wells

In [196]:
fracturing_only_well_ids_df = (
    fact_fracturing_df[
        fact_fracturing_df["idpozo"].isin(fracturing_only_well_ids)
    ][
        [
            "idpozo",
            "cuenca"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in fracturing-only well profiles: "
    f"{fracturing_only_well_ids_df.shape[0]}"
)

print(
    f"Unique wells: "
    f"{fracturing_only_well_ids_df['idpozo'].nunique()}"
)

print(
    f"Missing cuenca: "
    f"{fracturing_only_well_ids_df['cuenca'].isna().sum()}"
)

print(
    f"Unique cuencas: "
    f"{fracturing_only_well_ids_df['cuenca'].nunique()}"
)

fracturing_only_well_ids_df.head(10)

Rows in fracturing-only well profiles: 797
Unique wells: 797
Missing cuenca: 0
Unique cuencas: 3


,idpozo,cuenca
0,136243,AUSTRAL
1,160771,AUSTRAL
2,8788,NEUQUINA
3,136559,NEUQUINA
4,136362,NEUQUINA
5,135306,NEUQUINA
6,144101,NEUQUINA
7,136367,NEUQUINA
8,143921,NEUQUINA
9,137176,NEUQUINA


Step 10. Compare fracturing wells against production wells

In [197]:
production_wells = set(
    production_df["idpozo"].dropna().unique()
)

fracturing_wells = set(
    fact_fracturing_df["idpozo"].dropna().unique()
)

common_wells = fracturing_wells & production_wells
fracturing_only_wells = fracturing_wells - production_wells

print(
    f"Fracturing unique idpozo: {len(fracturing_wells)}"
)

print(
    f"Production unique idpozo: {len(production_wells)}"
)

print(
    f"Common idpozo: {len(common_wells)}"
)

print(
    f"Fracturing idpozo NOT in production: "
    f"{len(fracturing_only_wells)}"
)

Fracturing unique idpozo: 4562
Production unique idpozo: 5041
Common idpozo: 3765
Fracturing idpozo NOT in production: 797


Step 11. Validate location profiles for common fracturing/production wells

In [198]:
common_well_location_profiles = (
    production_df[
        production_df["idpozo"].isin(common_wells)
    ][
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
)

profiles_per_well = (
    common_well_location_profiles
    .groupby("idpozo")
    .size()
)

print(
    f"Common wells: "
    f"{len(common_wells)}"
)

print(
    f"Wells with exactly one location profile: "
    f"{(profiles_per_well == 1).sum()}"
)

print(
    f"Wells with more than one location profile: "
    f"{(profiles_per_well > 1).sum()}"
)

print(
    f"Maximum profiles for one well: "
    f"{profiles_per_well.max()}"
)

print(
    f"Missing provincia: "
    f"{common_well_location_profiles['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{common_well_location_profiles['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{common_well_location_profiles['coordenaday'].isna().sum()}"
)

Common wells: 3765
Wells with exactly one location profile: 3765
Wells with more than one location profile: 0
Maximum profiles for one well: 1
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0


Step 12. Validate basin consistency between production and fracturing

In [199]:
production_common = (
    production_df[
        production_df["idpozo"].isin(common_wells)
    ][
        ["idpozo", "cuenca"]
    ]
    .drop_duplicates()
)

fracturing_common = (
    fact_fracturing_df[
        fact_fracturing_df["idpozo"].isin(common_wells)
    ][
        ["idpozo", "cuenca"]
    ]
    .drop_duplicates()
)

cuenca_comparison = (
    fracturing_common
    .merge(
        production_common,
        on="idpozo",
        how="left",
        suffixes=("_fracturing", "_production")
    )
)

cuenca_comparison["cuenca_match"] = (
    cuenca_comparison["cuenca_fracturing"]
    == cuenca_comparison["cuenca_production"]
)

print(
    f"Rows in cuenca comparison: "
    f"{cuenca_comparison.shape[0]}"
)

print(
    f"Matching cuenca records: "
    f"{cuenca_comparison['cuenca_match'].sum()}"
)

print(
    f"Non-matching cuenca records: "
    f"{(~cuenca_comparison['cuenca_match']).sum()}"
)

print(
    f"Missing production cuenca: "
    f"{cuenca_comparison['cuenca_production'].isna().sum()}"
)

Rows in cuenca comparison: 3765
Matching cuenca records: 3765
Non-matching cuenca records: 0
Missing production cuenca: 0


Step 13. Load dim_location from MySQL

In [200]:
dim_location_check = pd.read_sql(
    """
    SELECT
        location_key,
        cuenca,
        provincia,
        coordenadax,
        coordenaday
    FROM dim_location
    """,
    conn
)

print(
    f"Rows in dim_location: "
    f"{dim_location_check.shape[0]}"
)

print(
    f"Unique location_key: "
    f"{dim_location_check['location_key'].nunique()}"
)

print(
    f"Missing cuenca: "
    f"{dim_location_check['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{dim_location_check['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{dim_location_check['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{dim_location_check['coordenaday'].isna().sum()}"
)

Rows in dim_location: 4845
Unique location_key: 4845
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 1
Missing coordenaday: 1


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1467741168.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_location_check = pd.read_sql(


Step 14. Compare production location profiles with dim_location

In [201]:
production_profiles = (
    production_df[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_profiles = (
    dim_location_check[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

comparison = (
    production_profiles
    .merge(
        dim_profiles,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="left",
        indicator=True
    )
)

print(
    f"Unique production profiles: "
    f"{production_profiles.shape[0]}"
)

print(
    f"Unique dim_location profiles: "
    f"{dim_profiles.shape[0]}"
)

print(
    f"Exact matches: "
    f"{(comparison['_merge'] == 'both').sum()}"
)

print(
    f"Profiles missing from dim_location: "
    f"{(comparison['_merge'] == 'left_only').sum()}"
)

Unique production profiles: 4845
Unique dim_location profiles: 4845
Exact matches: 4489
Profiles missing from dim_location: 356


Step 15. Check coordinate precision differences

In [202]:
missing_profiles = (
    comparison[
        comparison["_merge"] == "left_only"
    ][
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .reset_index(drop=True)
)

print(
    f"Missing profiles to investigate: "
    f"{missing_profiles.shape[0]}"
)

print("-" * 35)
print(missing_profiles.head(10))

Missing profiles to investigate: 356
-----------------------------------
     cuenca  provincia  coordenadax  coordenaday
0  NEUQUINA  Rio Negro   -67.835484   -39.013561
1  NEUQUINA  Rio Negro   -67.847106   -39.010672
2  NEUQUINA    Neuquén   -68.869028   -38.298751
3  NEUQUINA    Neuquén   -68.631429   -38.357228
4  NEUQUINA    Neuquén   -68.869028   -38.298751
5  NEUQUINA    Neuquén   -68.911033   -38.405911
6  NEUQUINA    Neuquén   -68.798926   -38.302823
7  NEUQUINA    Neuquén   -68.267933   -38.709843
8  NEUQUINA    Neuquén   -68.938997   -38.744930
9  NEUQUINA    Neuquén   -69.363491   -38.573512


Step 16. Trace missing location profiles back to wells

In [203]:
missing_profile_wells = (
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .merge(
        missing_profiles,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="inner"
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Missing location profiles: "
    f"{missing_profiles.shape[0]}"
)

print(
    f"Wells associated with missing profiles: "
    f"{missing_profile_wells['idpozo'].nunique()}"
)

print(
    f"Rows in missing profile/well analysis: "
    f"{missing_profile_wells.shape[0]}"
)

print("-" * 35)
print(
    missing_profile_wells.head(10)
)

Missing location profiles: 356
Wells associated with missing profiles: 377
Rows in missing profile/well analysis: 377
-----------------------------------
   idpozo    cuenca  provincia  coordenadax  coordenaday
0  155580  NEUQUINA  Rio Negro   -67.835484   -39.013561
1  155581  NEUQUINA  Rio Negro   -67.847106   -39.010672
2  135335  NEUQUINA    Neuquén   -68.869028   -38.298751
3  135484  NEUQUINA    Neuquén   -68.631429   -38.357228
4  135480  NEUQUINA    Neuquén   -68.869028   -38.298751
5  134528  NEUQUINA    Neuquén   -68.911033   -38.405911
6  135147  NEUQUINA    Neuquén   -68.798926   -38.302823
7   72232  NEUQUINA    Neuquén   -68.267933   -38.709843
8  131600  NEUQUINA    Neuquén   -68.938997   -38.744930
9  115725  NEUQUINA    Neuquén   -69.363491   -38.573512


Step 17. Compare location profiles in both directions

In [204]:
location_profile_comparison_full = (
    production_profiles
    .merge(
        dim_profiles,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="outer",
        indicator=True
    )
)

production_only_profiles = (
    location_profile_comparison_full[
        location_profile_comparison_full["_merge"] == "left_only"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

dim_only_profiles = (
    location_profile_comparison_full[
        location_profile_comparison_full["_merge"] == "right_only"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

common_profiles = (
    location_profile_comparison_full[
        location_profile_comparison_full["_merge"] == "both"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

print(
    f"Production-only profiles: "
    f"{production_only_profiles.shape[0]}"
)

print(
    f"dim_location-only profiles: "
    f"{dim_only_profiles.shape[0]}"
)

print(
    f"Common profiles: "
    f"{common_profiles.shape[0]}"
)

print(
    f"Total production profiles: "
    f"{production_profiles.shape[0]}"
)

print(
    f"Total dim_location profiles: "
    f"{dim_profiles.shape[0]}"
)

print("-" * 35)
print("First production-only profiles:")
print(production_only_profiles.head(10))

print("-" * 35)
print("First dim_location-only profiles:")
print(dim_only_profiles.head(10))

Production-only profiles: 356
dim_location-only profiles: 356
Common profiles: 4489
Total production profiles: 4845
Total dim_location profiles: 4845
-----------------------------------
First production-only profiles:
            cuenca   provincia  coordenadax  coordenaday
0          AUSTRAL  Santa Cruz   -71.158111   -50.535583
1          AUSTRAL  Santa Cruz   -71.139333   -50.652806
2          AUSTRAL  Santa Cruz   -70.671833   -50.769833
3          AUSTRAL  Santa Cruz   -70.660528   -50.801167
4  GOLFO SAN JORGE      Chubut   -67.927537   -45.884326
5  GOLFO SAN JORGE  Santa Cruz   -67.514736   -46.563062
6  GOLFO SAN JORGE  Santa Cruz   -67.499236   -46.564269
7  GOLFO SAN JORGE  Santa Cruz   -67.490557   -46.563390
8  GOLFO SAN JORGE  Santa Cruz   -67.466650   -46.546775
9  GOLFO SAN JORGE  Santa Cruz   -67.457883   -46.544774
-----------------------------------
First dim_location-only profiles:
            cuenca   provincia  coordenadax  coordenaday
0          AUSTRAL  Santa Cr

Step 18. Check floating-point precision differences

In [205]:
test_coordinate = (
    production_only_profiles
    .iloc[0]
)

print("Production profile:")
print(test_coordinate)

print("-" * 35)
print("Matching coordinate candidates in dim_location:")

candidate = dim_profiles[
    (
        dim_profiles["cuenca"] == test_coordinate["cuenca"]
    )
    &
    (
        dim_profiles["provincia"] == test_coordinate["provincia"]
    )
    &
    (
        dim_profiles["coordenadax"].round(8)
        == round(test_coordinate["coordenadax"], 8)
    )
    &
    (
        dim_profiles["coordenaday"].round(8)
        == round(test_coordinate["coordenaday"], 8)
    )
]

print(candidate)

Production profile:
cuenca            AUSTRAL
provincia      Santa Cruz
coordenadax    -71.158111
coordenaday    -50.535583
Name: 0, dtype: object
-----------------------------------
Matching coordinate candidates in dim_location:
       cuenca   provincia  coordenadax  coordenaday
3916  AUSTRAL  Santa Cruz   -71.158111   -50.535583


Step 19. Robust location profile comparison

In [206]:
production_profiles_rounded = production_profiles.copy()
dim_profiles_rounded = dim_profiles.copy()

for col in ["coordenadax", "coordenaday"]:
    production_profiles_rounded[col] = (
        production_profiles_rounded[col].round(8)
    )
    
    dim_profiles_rounded[col] = (
        dim_profiles_rounded[col].round(8)
    )

robust_location_comparison = (
    production_profiles_rounded
    .merge(
        dim_profiles_rounded,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="outer",
        indicator=True
    )
)

production_only_robust = (
    robust_location_comparison[
        robust_location_comparison["_merge"] == "left_only"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

dim_only_robust = (
    robust_location_comparison[
        robust_location_comparison["_merge"] == "right_only"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

common_robust = (
    robust_location_comparison[
        robust_location_comparison["_merge"] == "both"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

print(
    f"Production profiles: "
    f"{production_profiles.shape[0]}"
)

print(
    f"dim_location profiles: "
    f"{dim_profiles.shape[0]}"
)

print(
    f"Common profiles after rounding: "
    f"{common_robust.shape[0]}"
)

print(
    f"Production-only after rounding: "
    f"{production_only_robust.shape[0]}"
)

print(
    f"dim_location-only after rounding: "
    f"{dim_only_robust.shape[0]}"
)

print("-" * 35)
print("Production-only profiles:")
print(production_only_robust.head(10))

Production profiles: 4845
dim_location profiles: 4845
Common profiles after rounding: 4845
Production-only after rounding: 0
dim_location-only after rounding: 4
-----------------------------------
Production-only profiles:
Empty DataFrame
Columns: [cuenca, provincia, coordenadax, coordenaday]
Index: []


Step 20. Identify wells behind the 3 missing location profiles

In [207]:
missing_location_wells = (
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .merge(
        production_only_robust,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="inner"
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows: {missing_location_wells.shape[0]}"
)

print(
    f"Unique wells: "
    f"{missing_location_wells['idpozo'].nunique()}"
)

print("-" * 35)
print(missing_location_wells)

Rows: 0
Unique wells: 0
-----------------------------------
Empty DataFrame
Columns: [idpozo, cuenca, provincia, coordenadax, coordenaday]
Index: []


Step 21. Identify wells behind the 3 truly missing profiles

In [208]:
production_for_location_check = production_df.copy()

production_for_location_check["coordenadax"] = (
    production_for_location_check["coordenadax"].round(8)
)

production_for_location_check["coordenaday"] = (
    production_for_location_check["coordenaday"].round(8)
)

missing_location_wells = (
    production_for_location_check[
        production_for_location_check[
            [
                "cuenca",
                "provincia",
                "coordenadax",
                "coordenaday"
            ]
        ].apply(tuple, axis=1).isin(
            production_only_robust[
                [
                    "cuenca",
                    "provincia",
                    "coordenadax",
                    "coordenaday"
                ]
            ].apply(tuple, axis=1)
        )
    ]
    [
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows: {missing_location_wells.shape[0]}"
)

print(
    f"Unique wells: "
    f"{missing_location_wells['idpozo'].nunique()}"
)

print("-" * 35)
print(missing_location_wells)

Rows: 0
Unique wells: 0
-----------------------------------
Empty DataFrame
Columns: [idpozo, cuenca, provincia, coordenadax, coordenaday]
Index: []


Step 22. Check whether the 3 missing production wells

In [209]:
missing_location_well_ids = set(
    missing_location_wells["idpozo"]
)

fracturing_well_check = (
    fact_fracturing_df[
        fact_fracturing_df["idpozo"].isin(
            missing_location_well_ids
        )
    ]
    [
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "cuenca"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Production wells checked: "
    f"{len(missing_location_well_ids)}"
)

print(
    f"Wells found in fracturing: "
    f"{fracturing_well_check['idpozo'].nunique()}"
)

print(
    f"Fracturing records found: "
    f"{fracturing_well_check.shape[0]}"
)

print("-" * 35)
print(fracturing_well_check)

Production wells checked: 0
Wells found in fracturing: 0
Fracturing records found: 0
-----------------------------------
Empty DataFrame
Columns: [id_base_fractura_adjiv, idpozo, cuenca]
Index: []


Step 23. Final verification of the 3 missing location profiles

In [210]:
profiles_to_add_check = production_only_robust.copy()

print(
    f"Profiles to verify: "
    f"{profiles_to_add_check.shape[0]}"
)

print("-" * 35)
print(profiles_to_add_check)

Profiles to verify: 0
-----------------------------------
Empty DataFrame
Columns: [cuenca, provincia, coordenadax, coordenaday]
Index: []


In [211]:
# Check against dim_location using rounded coordinates

dim_location_check = dim_profiles.copy()

dim_location_check["coordenadax"] = (
    dim_location_check["coordenadax"].round(8)
)

dim_location_check["coordenaday"] = (
    dim_location_check["coordenaday"].round(8)
)

profiles_to_add_verification = (
    profiles_to_add_check
    .merge(
        dim_location_check,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="left",
        indicator=True
    )
)

print(
    f"Profiles found in dim_location: "
    f"{
        (
            profiles_to_add_verification["_merge"]
            == "both"
        ).sum()
    }"
)

print(
    f"Profiles not found in dim_location: "
    f"{
        (
            profiles_to_add_verification["_merge"]
            == "left_only"
        ).sum()
    }"
)

print("-" * 35)
print(profiles_to_add_verification)

Profiles found in dim_location: 0
Profiles not found in dim_location: 0
-----------------------------------
Empty DataFrame
Columns: [cuenca, provincia, coordenadax, coordenaday, _merge]
Index: []


Step 24. Prepare missing location profiles for dimension insertion

In [212]:
new_location_profiles = (
    missing_location_wells[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows to insert: "
    f"{new_location_profiles.shape[0]}"
)

print(
    f"Duplicated profiles: "
    f"{new_location_profiles.duplicated().sum()}"
)

print(
    f"Missing cuenca: "
    f"{new_location_profiles['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{new_location_profiles['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{new_location_profiles['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{new_location_profiles['coordenaday'].isna().sum()}"
)

print("-" * 35)
print(new_location_profiles)

Rows to insert: 0
Duplicated profiles: 0
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0
-----------------------------------
Empty DataFrame
Columns: [cuenca, provincia, coordenadax, coordenaday]
Index: []


Step 25. Insert missing profiles into dim_location

In [213]:
insert_sql = """
INSERT INTO dim_location (
    cuenca,
    provincia,
    coordenadax,
    coordenaday
)
VALUES (%s, %s, %s, %s)
"""

for _, row in new_location_profiles.iterrows():
    cursor.execute(
        insert_sql,
        (
            row["cuenca"],
            row["provincia"],
            row["coordenadax"],
            row["coordenaday"]
        )
    )

conn.commit()

print("Inserted profiles:", len(new_location_profiles))

Inserted profiles: 0


Step 26. Validate dim_location after insertion

In [214]:
dim_location_validation = pd.read_sql(
    """
    SELECT
        location_key,
        cuenca,
        provincia,
        coordenadax,
        coordenaday
    FROM dim_location
    """,
    conn
)

dim_location_profiles = (
    dim_location_validation[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
)

print(
    f"Rows in dim_location: "
    f"{dim_location_validation.shape[0]}"
)

print(
    f"Unique location profiles: "
    f"{dim_location_profiles.shape[0]}"
)

print(
    f"Duplicated location profiles: "
    f"{dim_location_profiles.duplicated().sum()}"
)

print(
    f"Missing cuenca: "
    f"{dim_location_validation['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{dim_location_validation['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{dim_location_validation['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{dim_location_validation['coordenaday'].isna().sum()}"
)

Rows in dim_location: 4845
Unique location profiles: 4845
Duplicated location profiles: 0
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 1
Missing coordenaday: 1


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1060550342.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_location_validation = pd.read_sql(


Step 27. Final production vs dim_location validation

In [215]:
production_location_profiles = (
    production_df[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .copy()
)

dim_location_profiles = (
    dim_location_validation[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .copy()
)

for df in [
    production_location_profiles,
    dim_location_profiles
]:
    df["coordenadax"] = df["coordenadax"].round(8)
    df["coordenaday"] = df["coordenaday"].round(8)

production_profiles_set = set(
    production_location_profiles.itertuples(
        index=False,
        name=None
    )
)

dim_location_profiles_set = set(
    dim_location_profiles.itertuples(
        index=False,
        name=None
    )
)

common_profiles = (
    production_profiles_set
    & dim_location_profiles_set
)

production_only_profiles = (
    production_profiles_set
    - dim_location_profiles_set
)

dim_location_only_profiles = (
    dim_location_profiles_set
    - production_profiles_set
)

print(
    f"Production profiles: "
    f"{len(production_profiles_set)}"
)

print(
    f"dim_location profiles: "
    f"{len(dim_location_profiles_set)}"
)

print(
    f"Common profiles: "
    f"{len(common_profiles)}"
)

print(
    f"Production-only profiles: "
    f"{len(production_only_profiles)}"
)

print(
    f"dim_location-only profiles: "
    f"{len(dim_location_only_profiles)}"
)

Production profiles: 4841
dim_location profiles: 4845
Common profiles: 4841
Production-only profiles: 0
dim_location-only profiles: 4


Step 28. Build production well-location lookup

In [216]:
production_well_location_lookup = (
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in production well-location lookup: "
    f"{production_well_location_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{production_well_location_lookup['idpozo'].nunique()}"
)

print(
    f"Duplicated wells: "
    f"{production_well_location_lookup['idpozo'].duplicated().sum()}"
)

print(
    f"Missing provincia: "
    f"{production_well_location_lookup['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{production_well_location_lookup['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{production_well_location_lookup['coordenaday'].isna().sum()}"
)

Rows in production well-location lookup: 5041
Unique wells: 5041
Duplicated wells: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0


Step 29. Classify fracturing wells by production coverage

In [217]:
fracturing_well_ids = set(
    fact_fracturing_df["idpozo"]
    .dropna()
    .unique()
)

production_well_ids = set(
    production_well_location_lookup["idpozo"]
    .dropna()
    .unique()
)

fracturing_wells_in_production = (
    fracturing_well_ids
    & production_well_ids
)

fracturing_wells_only = (
    fracturing_well_ids
    - production_well_ids
)

print(
    f"Fracturing unique wells: "
    f"{len(fracturing_well_ids)}"
)

print(
    f"Fracturing wells also in production: "
    f"{len(fracturing_wells_in_production)}"
)

print(
    f"Fracturing-only wells: "
    f"{len(fracturing_wells_only)}"
)

Fracturing unique wells: 4562
Fracturing wells also in production: 3765
Fracturing-only wells: 797


Step 30. Match shared fracturing wells with production locations

In [218]:
shared_well_location_lookup = (
    fact_fracturing_df[
        ["idpozo"]
    ]
    .drop_duplicates()
    .merge(
        production_well_location_lookup,
        on="idpozo",
        how="inner"
    )
)

print(
    f"Rows in shared well-location lookup: "
    f"{shared_well_location_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{shared_well_location_lookup['idpozo'].nunique()}"
)

print(
    f"Missing cuenca: "
    f"{shared_well_location_lookup['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{shared_well_location_lookup['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{shared_well_location_lookup['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{shared_well_location_lookup['coordenaday'].isna().sum()}"
)

print("-" * 35)
print(shared_well_location_lookup.head())

Rows in shared well-location lookup: 3765
Unique wells: 3765
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0
-----------------------------------
   idpozo    cuenca provincia  coordenadax  coordenaday
0  159910  NEUQUINA   Neuquén   -68.403889   -38.873611
1  159219  NEUQUINA   Neuquén   -69.159188   -38.221866
2  159220  NEUQUINA   Neuquén   -69.159074   -38.221864
3  159221  NEUQUINA   Neuquén   -69.158959   -38.221862
4  160426  NEUQUINA   Neuquén   -69.157819   -38.223443


Step 31. Lookup location_key for shared fracturing wells

In [219]:
shared_well_location_lookup_rounded = (
    shared_well_location_lookup.copy()
)

dim_location_lookup = (
    dim_location_validation[
        [
            "location_key",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .copy()
)

shared_well_location_lookup_rounded["coordenadax"] = (
    shared_well_location_lookup_rounded["coordenadax"].round(8)
)

shared_well_location_lookup_rounded["coordenaday"] = (
    shared_well_location_lookup_rounded["coordenaday"].round(8)
)

dim_location_lookup["coordenadax"] = (
    dim_location_lookup["coordenadax"].round(8)
)

dim_location_lookup["coordenaday"] = (
    dim_location_lookup["coordenaday"].round(8)
)

shared_location_lookup = (
    shared_well_location_lookup_rounded
    .merge(
        dim_location_lookup,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="left",
        indicator=True
    )
)

print(
    f"Rows after location lookup: "
    f"{shared_location_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{shared_location_lookup['idpozo'].nunique()}"
)

print(
    f"Missing location_key: "
    f"{shared_location_lookup['location_key'].isna().sum()}"
)

print(
    f"Matched profiles: "
    f"{
        (
            shared_location_lookup["_merge"] == "both"
        ).sum()
    }"
)

print(
    f"Unmatched profiles: "
    f"{
        (
            shared_location_lookup["_merge"] == "left_only"
        ).sum()
    }"
)

print(
    f"Duplicated wells: "
    f"{shared_location_lookup['idpozo'].duplicated().sum()}"
)

Rows after location lookup: 3765
Unique wells: 3765
Missing location_key: 0
Matched profiles: 3765
Unmatched profiles: 0
Duplicated wells: 0


Step 32. Inspect fracturing-only wells

In [220]:
fracturing_only_well_profiles = (
    fact_fracturing_df[
        fact_fracturing_df["idpozo"].isin(
            fracturing_wells_only
        )
    ][
        [
            "idpozo",
            "cuenca"
        ]
    ]
    .drop_duplicates()
    .sort_values("idpozo")
    .reset_index(drop=True)
)

print(
    f"Rows in fracturing-only well profiles: "
    f"{fracturing_only_well_profiles.shape[0]}"
)

print(
    f"Unique wells: "
    f"{fracturing_only_well_profiles['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{fracturing_only_well_profiles['idpozo'].isna().sum()}"
)

print(
    f"Missing cuenca: "
    f"{fracturing_only_well_profiles['cuenca'].isna().sum()}"
)

print(
    f"Unique cuencas: "
    f"{fracturing_only_well_profiles['cuenca'].nunique()}"
)

print("-" * 35)
print(
    fracturing_only_well_profiles["cuenca"]
    .value_counts(dropna=False)
)

print("-" * 35)
print(
    fracturing_only_well_profiles.head(20)
)

Rows in fracturing-only well profiles: 797
Unique wells: 797
Missing idpozo: 0
Missing cuenca: 0
Unique cuencas: 3
-----------------------------------
cuenca
GOLFO SAN JORGE    462
NEUQUINA           324
AUSTRAL             11
Name: count, dtype: int64
-----------------------------------
    idpozo           cuenca
0      458  GOLFO SAN JORGE
1      460  GOLFO SAN JORGE
2     5984  GOLFO SAN JORGE
3     5990  GOLFO SAN JORGE
4     5993  GOLFO SAN JORGE
5     5994  GOLFO SAN JORGE
6     6002  GOLFO SAN JORGE
7     6007  GOLFO SAN JORGE
8     6013  GOLFO SAN JORGE
9     6019  GOLFO SAN JORGE
10    6031  GOLFO SAN JORGE
11    6036  GOLFO SAN JORGE
12    6040  GOLFO SAN JORGE
13    6050  GOLFO SAN JORGE
14    6051  GOLFO SAN JORGE
15    6066  GOLFO SAN JORGE
16    6068  GOLFO SAN JORGE
17    6090  GOLFO SAN JORGE
18    6128  GOLFO SAN JORGE
19    6140  GOLFO SAN JORGE


Step 33. Verify fracturing-only wells against production source

In [221]:
fracturing_only_production_check = (
    fracturing_only_well_profiles[
        ["idpozo"]
    ]
    .merge(
        production_df[
            [
                "idpozo",
                "cuenca",
                "provincia",
                "coordenadax",
                "coordenaday"
            ]
        ]
        .drop_duplicates("idpozo"),
        on="idpozo",
        how="left",
        indicator=True
    )
)

print(
    f"Rows checked: "
    f"{fracturing_only_production_check.shape[0]}"
)

print(
    f"Wells found in production: "
    f"{
        (
            fracturing_only_production_check["_merge"]
            == "both"
        ).sum()
    }"
)

print(
    f"Wells NOT found in production: "
    f"{
        (
            fracturing_only_production_check["_merge"]
            == "left_only"
        ).sum()
    }"
)

Rows checked: 797
Wells found in production: 0
Wells NOT found in production: 797


Step 34. Final well-location_key lookup for shared wells

In [222]:
shared_well_location_key_lookup = (
    shared_location_lookup[
        [
            "idpozo",
            "location_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in well-location_key lookup: "
    f"{shared_well_location_key_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{shared_well_location_key_lookup['idpozo'].nunique()}"
)

print(
    f"Unique location_keys: "
    f"{shared_well_location_key_lookup['location_key'].nunique()}"
)

print(
    f"Missing location_key: "
    f"{shared_well_location_key_lookup['location_key'].isna().sum()}"
)

print(
    f"Duplicated wells: "
    f"{shared_well_location_key_lookup['idpozo'].duplicated().sum()}"
)

print("-" * 35)
print(
    shared_well_location_key_lookup.head()
)

Rows in well-location_key lookup: 3765
Unique wells: 3765
Unique location_keys: 3731
Missing location_key: 0
Duplicated wells: 0
-----------------------------------
   idpozo  location_key
0  159910          4684
1  159219           862
2  159220           844
3  159221           408
4  160426          1169


Step 35. Apply location_key lookup to fracturing records

In [223]:
fact_fracturing_location_lookup = (
    fact_fracturing_df[
        [
            "id_base_fractura_adjiv",
            "idpozo"
        ]
    ]
    .merge(
        shared_well_location_key_lookup,
        on="idpozo",
        how="left"
    )
)

print(
    f"Rows after location lookup: "
    f"{fact_fracturing_location_lookup.shape[0]}"
)

print(
    f"Unique fracturing records: "
    f"{fact_fracturing_location_lookup['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing location_key: "
    f"{fact_fracturing_location_lookup['location_key'].isna().sum()}"
)

print(
    f"Resolved location_key: "
    f"{fact_fracturing_location_lookup['location_key'].notna().sum()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{fact_fracturing_location_lookup['id_base_fractura_adjiv'].duplicated().sum()}"
)

Rows after location lookup: 4806
Unique fracturing records: 4806
Missing location_key: 969
Resolved location_key: 3837
Duplicated fracturing IDs: 0


Step 36. Validate resolved fracturing wells

In [224]:
resolved_fracturing_location = (
    fact_fracturing_location_lookup[
        fact_fracturing_location_lookup["location_key"].notna()
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "location_key"
        ]
    ]
    .drop_duplicates()
)

resolved_well_ids = set(
    resolved_fracturing_location["idpozo"]
)

shared_well_ids = set(
    fracturing_wells_in_production
)

fracturing_only_ids = set(
    fracturing_wells_only
)

print(
    f"Resolved fracturing records: "
    f"{resolved_fracturing_location.shape[0]}"
)

print(
    f"Unique resolved wells: "
    f"{len(resolved_well_ids)}"
)

print(
    f"Resolved wells expected from production: "
    f"{len(resolved_well_ids & shared_well_ids)}"
)

print(
    f"Resolved wells classified as fracturing-only: "
    f"{len(resolved_well_ids & fracturing_only_ids)}"
)

Resolved fracturing records: 3837
Unique resolved wells: 3765
Resolved wells expected from production: 3765
Resolved wells classified as fracturing-only: 0


Step 37. Validate unresolved location records

In [225]:
unresolved_fracturing_location = (
    fact_fracturing_location_lookup[
        fact_fracturing_location_lookup["location_key"].isna()
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

unresolved_well_ids = set(
    unresolved_fracturing_location["idpozo"]
)

print(
    f"Unresolved fracturing records: "
    f"{unresolved_fracturing_location.shape[0]}"
)

print(
    f"Unique unresolved wells: "
    f"{len(unresolved_well_ids)}"
)

print(
    f"Unresolved wells classified as fracturing-only: "
    f"{len(unresolved_well_ids & fracturing_only_ids)}"
)

print(
    f"Unresolved wells classified as shared: "
    f"{len(unresolved_well_ids & shared_well_ids)}"
)

print(
    f"Missing idpozo: "
    f"{unresolved_fracturing_location['idpozo'].isna().sum()}"
)

Unresolved fracturing records: 969
Unique unresolved wells: 797
Unresolved wells classified as fracturing-only: 797
Unresolved wells classified as shared: 0
Missing idpozo: 0


Step 38. Add location_key to fact_fracturing_df

In [226]:
fact_fracturing_location_df = (
    fact_fracturing_df
    .drop(columns=["location_key"], errors="ignore")
    .merge(
        fact_fracturing_location_lookup[
            [
                "id_base_fractura_adjiv",
                "location_key"
            ]
        ],
        on="id_base_fractura_adjiv",
        how="left"
    )
)

print(
    f"Rows after final location lookup: "
    f"{fact_fracturing_location_df.shape[0]}"
)

print(
    f"Unique fracturing records: "
    f"{fact_fracturing_location_df['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing location_key: "
    f"{fact_fracturing_location_df['location_key'].isna().sum()}"
)

print(
    f"Resolved location_key: "
    f"{fact_fracturing_location_df['location_key'].notna().sum()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{fact_fracturing_location_df['id_base_fractura_adjiv'].duplicated().sum()}"
)

Rows after final location lookup: 4806
Unique fracturing records: 4806
Missing location_key: 969
Resolved location_key: 3837
Duplicated fracturing IDs: 0


Step 39. Validate final fact structure

In [227]:
print("Original columns:")
print(fact_fracturing_df.columns.tolist())

print("-" * 35)
print("Final columns:")
print(fact_fracturing_location_df.columns.tolist())

print("-" * 35)
print(
    "Original shape:",
    fact_fracturing_df.shape
)

print(
    "Final shape:",
    fact_fracturing_location_df.shape
)

Original columns:
['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key', 'geology_key', 'field_area_key']
-----------------------------------
Final columns:
['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key', 'geology_key', 'field_area_key', 'location

Step 40. Validate location_key referential integrity

In [228]:
dim_location_keys = set(
    dim_location["location_key"]
)

fact_fracturing_location_keys = set(
    fact_fracturing_location_df[
        "location_key"
    ].dropna()
)

invalid_location_keys = (
    fact_fracturing_location_keys
    - dim_location_keys
)

print(
    f"Location keys in fracturing fact: "
    f"{len(fact_fracturing_location_keys)}"
)

print(
    f"Location keys existing in dim_location: "
    f"{len(fact_fracturing_location_keys & dim_location_keys)}"
)

print(
    f"Invalid location keys: "
    f"{len(invalid_location_keys)}"
)

Location keys in fracturing fact: 3731
Location keys existing in dim_location: 3731
Invalid location keys: 0


Step 41. Identify invalid location_keys

In [229]:
invalid_location_keys_df = (
    fact_fracturing_location_df[
        fact_fracturing_location_df["location_key"].isin(
            invalid_location_keys
        )
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "cuenca",
            "location_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Invalid location keys: "
    f"{len(invalid_location_keys)}"
)

print(
    f"Fracturing records affected: "
    f"{invalid_location_keys_df.shape[0]}"
)

print(
    f"Unique wells affected: "
    f"{invalid_location_keys_df['idpozo'].nunique()}"
)

print("-" * 35)

print(invalid_location_keys_df)

Invalid location keys: 0
Fracturing records affected: 0
Unique wells affected: 0
-----------------------------------
Empty DataFrame
Columns: [id_base_fractura_adjiv, idpozo, cuenca, location_key]
Index: []


Step 42. Check the three manually added location profile

In [230]:
dim_location[
    dim_location["location_key"].isin(
        [4842, 4843, 4844]
    )
]

,location_key,cuenca,provincia,coordenadax,coordenaday
4841,4842,NEUQUINA,Neuquén,-69.228209,-38.561615
4842,4843,NEUQUINA,Neuquén,-69.024305,-38.484603
4843,4844,NEUQUINA,Neuquén,-68.469571,-38.415174


In [231]:
print(
    "Maximum location_key in dim_location:",
    dim_location["location_key"].max()
)

print(
    "Rows in dim_location:",
    dim_location.shape[0]
)

Maximum location_key in dim_location: 4857
Rows in dim_location: 4845


Step 43. Recover the three missing location profiles

In [232]:
profiles_to_recover = (
    production_location_profiles[
        production_location_profiles[
            [
                "cuenca",
                "provincia",
                "coordenadax",
                "coordenaday"
            ]
        ].apply(tuple, axis=1).isin(
            [
                ("NEUQUINA", "Neuquén", -69.228209, -38.561615),
                ("NEUQUINA", "Neuquén", -69.024305, -38.484603),
                ("NEUQUINA", "Neuquén", -68.469571, -38.415174)
            ]
        )
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Profiles recovered: "
    f"{profiles_to_recover.shape[0]}"
)

print(profiles_to_recover)

Profiles recovered: 0
Empty DataFrame
Columns: [cuenca, provincia, coordenadax, coordenaday]
Index: []


In [233]:
print(dim_location.columns.tolist())
print(dim_location.shape)

['location_key', 'cuenca', 'provincia', 'coordenadax', 'coordenaday']
(4845, 5)


Step 44. Prepare missing location profiles for insertion

In [234]:
new_location_profiles = profiles_to_recover[
    [
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday"
    ]
].copy()

print(
    f"Rows to insert: {new_location_profiles.shape[0]}"
)

print(
    f"Duplicated profiles: "
    f"{new_location_profiles.duplicated().sum()}"
)

print(
    f"Missing cuenca: "
    f"{new_location_profiles['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{new_location_profiles['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{new_location_profiles['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{new_location_profiles['coordenaday'].isna().sum()}"
)

print("-" * 35)

print(new_location_profiles)

Rows to insert: 0
Duplicated profiles: 0
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0
-----------------------------------
Empty DataFrame
Columns: [cuenca, provincia, coordenadax, coordenaday]
Index: []


Step 45. Insert missing location profiles into MySQL

In [235]:
insert_sql = """
INSERT INTO dim_location (
    cuenca,
    provincia,
    coordenadax,
    coordenaday
)
VALUES (%s, %s, %s, %s)
"""

cursor = conn.cursor()

for _, row in new_location_profiles.iterrows():
    cursor.execute(
        insert_sql,
        (
            row["cuenca"],
            row["provincia"],
            row["coordenadax"],
            row["coordenaday"]
        )
    )

conn.commit()

cursor.close()

print(
    f"Inserted profiles: "
    f"{new_location_profiles.shape[0]}"
)

Inserted profiles: 0


Step 46. Refresh dim_location from MySQL

In [236]:
dim_location = pd.read_sql(
    """
    SELECT
        location_key,
        cuenca,
        provincia,
        coordenadax,
        coordenaday
    FROM dim_location
    """,
    conn
)

print(
    f"Rows in dim_location: "
    f"{dim_location.shape[0]}"
)

print(
    f"Maximum location_key: "
    f"{dim_location['location_key'].max()}"
)

Rows in dim_location: 4845
Maximum location_key: 4857


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\2453009554.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_location = pd.read_sql(


Step 47. Identify duplicated location profiles

In [237]:
duplicated_location_profiles = (
    dim_location[
        dim_location.duplicated(
            subset=[
                "cuenca",
                "provincia",
                "coordenadax",
                "coordenaday"
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday",
            "location_key"
        ]
    )
    .reset_index(drop=True)
)

print(
    f"Duplicated location rows: "
    f"{duplicated_location_profiles.shape[0]}"
)

print(
    f"Duplicated location profiles: "
    f"{duplicated_location_profiles[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ].drop_duplicates().shape[0]}"
)

print("-" * 35)

print(duplicated_location_profiles)

Duplicated location rows: 0
Duplicated location profiles: 0
-----------------------------------
Empty DataFrame
Columns: [location_key, cuenca, provincia, coordenadax, coordenaday]
Index: []


Step 48. Validate dim_location after cleanup

In [238]:
dim_location = pd.read_sql(
    """
    SELECT
        location_key,
        cuenca,
        provincia,
        coordenadax,
        coordenaday
    FROM dim_location
    """,
    conn
)

location_profile_columns = [
    "cuenca",
    "provincia",
    "coordenadax",
    "coordenaday"
]

print(
    f"Rows in dim_location: "
    f"{dim_location.shape[0]}"
)

print(
    f"Unique location_keys: "
    f"{dim_location['location_key'].nunique()}"
)

print(
    f"Duplicated location profiles: "
    f"{dim_location.duplicated(
        subset=location_profile_columns
    ).sum()}"
)

print(
    f"Missing cuenca: "
    f"{dim_location['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{dim_location['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{dim_location['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{dim_location['coordenaday'].isna().sum()}"
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\583726840.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_location = pd.read_sql(


Rows in dim_location: 4845
Unique location_keys: 4845
Duplicated location profiles: 0
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 1
Missing coordenaday: 1


Step 49. Validate location_key integrity in fact_fracturing_location_df

In [239]:
valid_location_keys = set(
    dim_location["location_key"].astype(int)
)

fact_location_keys = set(
    fact_fracturing_location_df["location_key"]
    .dropna()
    .astype(int)
)

invalid_location_keys = (
    fact_location_keys - valid_location_keys
)

# Primero filtramos los registros que tienen location_key
# y recién después convertimos a int.
fact_with_location = fact_fracturing_location_df[
    fact_fracturing_location_df["location_key"].notna()
].copy()

fact_with_location["location_key"] = (
    fact_with_location["location_key"].astype(int)
)

invalid_records = (
    fact_with_location[
        fact_with_location["location_key"].isin(
            invalid_location_keys
        )
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "cuenca",
            "location_key"
        ]
    ]
    .drop_duplicates()
)

print(
    f"Location keys in fracturing fact: "
    f"{len(fact_location_keys)}"
)

print(
    f"Location keys existing in dim_location: "
    f"{len(fact_location_keys & valid_location_keys)}"
)

print(
    f"Invalid location keys: "
    f"{len(invalid_location_keys)}"
)

print(
    f"Fracturing records affected: "
    f"{invalid_records.shape[0]}"
)

print(
    f"Unique wells affected: "
    f"{invalid_records['idpozo'].nunique()}"
)

print("-" * 35)

if invalid_records.shape[0] > 0:
    print(invalid_records)

Location keys in fracturing fact: 3731
Location keys existing in dim_location: 3731
Invalid location keys: 0
Fracturing records affected: 0
Unique wells affected: 0
-----------------------------------


Step 50. Final validation of fracturing location coverage

In [240]:
# Fracturing wells shared with production
shared_well_ids = set(
    fracturing_wells_in_production
)

# Exclusive fracking wells
fracturing_only_well_ids_set = set(
    fracturing_only_well_ids
)

# Records with location_key
resolved_location_records = (
    fact_fracturing_location_df[
        fact_fracturing_location_df["location_key"].notna()
    ]
)

# Records without location_key
unresolved_location_records = (
    fact_fracturing_location_df[
        fact_fracturing_location_df["location_key"].isna()
    ]
)

# Wells solved
resolved_wells = set(
    resolved_location_records["idpozo"]
)

# Unsolved Wells
unresolved_wells = set(
    unresolved_location_records["idpozo"]
)

print(
    f"Resolved fracturing records: "
    f"{resolved_location_records.shape[0]}"
)

print(
    f"Resolved unique wells: "
    f"{len(resolved_wells)}"
)

print(
    f"Expected shared wells: "
    f"{len(shared_well_ids)}"
)

print(
    f"Resolved wells not shared with production: "
    f"{len(resolved_wells - shared_well_ids)}"
)

print("-" * 35)

print(
    f"Unresolved fracturing records: "
    f"{unresolved_location_records.shape[0]}"
)

print(
    f"Unresolved unique wells: "
    f"{len(unresolved_wells)}"
)

print(
    f"Expected fracturing-only wells: "
    f"{len(fracturing_only_well_ids_set)}"
)

print(
    f"Unresolved wells not fracturing-only: "
    f"{len(unresolved_wells - fracturing_only_well_ids_set)}"
)

print("-" * 35)

print(
    f"Shared wells without location_key: "
    f"{len(shared_well_ids & unresolved_wells)}"
)

print(
    f"Fracturing-only wells with location_key: "
    f"{len(fracturing_only_well_ids_set & resolved_wells)}"
)

Resolved fracturing records: 3837
Resolved unique wells: 3765
Expected shared wells: 3765
Resolved wells not shared with production: 0
-----------------------------------
Unresolved fracturing records: 969
Unresolved unique wells: 797
Expected fracturing-only wells: 797
Unresolved wells not fracturing-only: 0
-----------------------------------
Shared wells without location_key: 0
Fracturing-only wells with location_key: 0


Step 51. Diagnostic of duplicated and incomplete location profiles

In [241]:
# Reload dim_location from MySQL
dim_location = pd.read_sql(
    """
    SELECT
        location_key,
        cuenca,
        provincia,
        coordenadax,
        coordenaday
    FROM dim_location
    """,
    conn
)

location_profile_columns = [
    "cuenca",
    "provincia",
    "coordenadax",
    "coordenaday"
]

# Identify duplicated location profiles
duplicated_location_profiles = (
    dim_location[
        dim_location.duplicated(
            subset=location_profile_columns,
            keep=False
        )
    ]
    .sort_values(
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday",
            "location_key"
        ]
    )
    .reset_index(drop=True)
)

print(
    f"Duplicated location rows: "
    f"{duplicated_location_profiles.shape[0]}"
)

print(
    f"Duplicated location profiles: "
    f"{duplicated_location_profiles[location_profile_columns]
        .drop_duplicates()
        .shape[0]}"
)

print("-" * 35)
print("Duplicated location profiles:")
print(duplicated_location_profiles)

# Identify all location_keys involved in duplicated profiles
duplicated_location_keys = (
    duplicated_location_profiles["location_key"]
    .dropna()
    .astype(int)
    .tolist()
)

print("-" * 35)
print(
    f"Location_keys involved in duplicated profiles: "
    f"{duplicated_location_keys}"
)

# Check whether duplicated location_keys are used by fact_fracturing
used_duplicate_keys = (
    set(
        fact_fracturing_location_df["location_key"]
        .dropna()
        .astype(int)
    )
    & set(duplicated_location_keys)
)

print("-" * 35)
print(
    f"Duplicated location_keys used by fact_fracturing: "
    f"{sorted(used_duplicate_keys)}"
)

# Identify incomplete location profiles
incomplete_location_profiles = dim_location[
    dim_location[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ].isna().any(axis=1)
].copy()

print("-" * 35)
print(
    f"Incomplete location profiles: "
    f"{incomplete_location_profiles.shape[0]}"
)

print("-" * 35)
print("Incomplete location profiles:")
print(incomplete_location_profiles)

Duplicated location rows: 0
Duplicated location profiles: 0
-----------------------------------
Duplicated location profiles:
Empty DataFrame
Columns: [location_key, cuenca, provincia, coordenadax, coordenaday]
Index: []
-----------------------------------
Location_keys involved in duplicated profiles: []
-----------------------------------
Duplicated location_keys used by fact_fracturing: []
-----------------------------------
Incomplete location profiles: 1
-----------------------------------
Incomplete location profiles:
      location_key   cuenca provincia  coordenadax  coordenaday
4844          4857  UNKNOWN   UNKNOWN          NaN          NaN


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\709543862.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_location = pd.read_sql(


#### 2.7 Lookup company_key

Objective

To obtain the substitute company_key by associating each fracturing treatment with the corresponding operating company.

Source Data

- fact_fracturing_df
- dim_company


Step 1. Verify company dimension

In [242]:
print(f"Rows in dim_company: {dim_company.shape[0]}")
print("-" * 35)

print(
    dim_company[
        ["company_key", "idempresa", "empresa"]
    ].head(10)
)

Rows in dim_company: 57
-----------------------------------
   company_key idempresa                        empresa
0            1      YSUR  YSUR ENERGÍA ARGENTINA S.R.L.
1            2       YPF                       YPF S.A.
2            3       WIN       WINTERSHALL ENERGIA S.A.
3            4       WDA  WINTERSHALL DEA ARGENTINA S.A
4            5       VST     VISTA ENERGY ARGENTINA SAU
5            6       VOG   Vista Oil & Gas Argentina SA
6            7       VNO                    VENOIL S.A.
7            8       VMI    VACA MUERTA INVERSIONES SAU
8            9       VIS  VISTA OIL & GAS ARGENTINA SAU
9           10       TPT                 TECPETROL S.A.


Step 2. Validate company name lookup key

In [243]:
duplicated_company_names = dim_company[
    dim_company["empresa"].duplicated(keep=False)
].sort_values("empresa")

print("===================================")
print("COMPANY NAME LOOKUP KEY VALIDATION")
print("===================================")

print(
    f"Duplicated company names in dim_company: "
    f"{duplicated_company_names.shape[0]}"
)

print(
    f"Unique duplicated company names: "
    f"{duplicated_company_names['empresa'].nunique()}"
)

print("-" * 35)
print("Duplicated company names:")
print(
    duplicated_company_names[
        ["company_key", "idempresa", "empresa"]
    ]
)

COMPANY NAME LOOKUP KEY VALIDATION
Duplicated company names in dim_company: 0
Unique duplicated company names: 0
-----------------------------------
Duplicated company names:
Empty DataFrame
Columns: [company_key, idempresa, empresa]
Index: []


Step 3. Merge with dim_company

In [244]:
fact_fracturing_df = fact_fracturing_df.merge(
    dim_company[
        [
            "company_key",
            "empresa"
        ]
    ],
    left_on="empresa_informante",
    right_on="empresa",
    how="left"
)

In [245]:
print(f"Missing company_key: {fact_fracturing_df['company_key'].isna().sum()}")
print(f"Rows: {fact_fracturing_df.shape[0]}")

Missing company_key: 0
Rows: 4806


Step 4. Validate company lookup coverage

In [246]:
fracturing_company_names = set(
    fact_fracturing_df["empresa_informante"]
    .dropna()
    .unique()
)

dimension_company_names = set(
    dim_company["empresa"]
    .dropna()
    .unique()
)

unmatched_company_names = sorted(
    fracturing_company_names
    - dimension_company_names
)

print("===================================")
print("COMPANY LOOKUP COVERAGE")
print("===================================")

print(
    f"Unique fracturing companies: "
    f"{len(fracturing_company_names)}"
)

print(
    f"Unique dimension companies: "
    f"{len(dimension_company_names)}"
)

print(
    f"Unmatched company names: "
    f"{len(unmatched_company_names)}"
)

print("-" * 35)
print("Unmatched company names:")
print(unmatched_company_names)

COMPANY LOOKUP COVERAGE
Unique fracturing companies: 23
Unique dimension companies: 57
Unmatched company names: 0
-----------------------------------
Unmatched company names:
[]


Step 5. Validate company descriptive consistency

In [247]:
company_name_lookup = (
    dim_company
    .set_index("empresa")["company_key"]
)

company_name_validation = fact_fracturing_df[
    ["empresa_informante", "empresa", "company_key"]
].copy()

company_name_validation["lookup_company_key"] = (
    company_name_validation["empresa_informante"]
    .map(company_name_lookup)
)

company_name_mismatches = company_name_validation[
    company_name_validation["company_key"]
    != company_name_validation["lookup_company_key"]
]

mismatch_profiles = (
    company_name_mismatches[
        ["empresa_informante", "empresa", "company_key", "lookup_company_key"]
    ]
    .drop_duplicates()
    .sort_values("empresa_informante")
)

print("===================================")
print("COMPANY DESCRIPTIVE CONSISTENCY")
print("===================================")

print(
    f"Rows checked: "
    f"{company_name_validation.shape[0]}"
)

print(
    f"Rows with mismatch: "
    f"{company_name_mismatches.shape[0]}"
)

print(
    f"Unique empresa_informante with mismatch: "
    f"{company_name_mismatches['empresa_informante'].nunique()}"
)

print(
    f"Mismatch profiles: "
    f"{mismatch_profiles.shape[0]}"
)

print("-" * 35)
print("Company mismatches:")
print(mismatch_profiles)

COMPANY DESCRIPTIVE CONSISTENCY
Rows checked: 4806
Rows with mismatch: 0
Unique empresa_informante with mismatch: 0
Mismatch profiles: 0
-----------------------------------
Company mismatches:
Empty DataFrame
Columns: [empresa_informante, empresa, company_key, lookup_company_key]
Index: []


### 3. Prepare the Fact Fracturing Table

#### Objective

Prepare the final fracturing fact table by retaining only the surrogate keys and quantitative measures defined in the dimensional model, ensuring that the dataset matches the structure of fact_fracturing before loading it into the data warehouse.

#### Source Data

- fact_fracturing_location_df
- fact_fracturing (target table schema)

#### Transformation

The fact table is prepared by removing all source-system attributes that were only required to perform the dimensional lookups. The resulting dataset preserves the original granularity of the fracturing data while matching the target schema of the data warehouse.

#### 3.1 Select Final Columns

In [248]:
fact_fracturing_load_df = (
    fact_fracturing_df
    .drop(columns=["location_key"], errors="ignore")
    .merge(
        fact_fracturing_location_lookup[
            [
                "id_base_fractura_adjiv",
                "location_key"
            ]
        ],
        on="id_base_fractura_adjiv",
        how="left",
        validate="one_to_one"
    )
)

In [249]:
fact_fracturing_load_df = fact_fracturing_load_df[
    [
        "id_base_fractura_adjiv",
        "start_time_key",
        "end_time_key",
        "well_key",
        "geology_key",
        "field_area_key",
        "location_key",
        "company_key",
        "arena_bombeada_nacional_tn",
        "arena_bombeada_importada_tn",
        "agua_inyectada_m3",
        "co2_inyectado_m3",
        "presion_maxima_psi",
        "potencia_equipos_fractura_hp",
        "cantidad_fracturas",
    ]
].copy()

In [250]:
print("Rows:", len(fact_fracturing_load_df))
print("Columns:", len(fact_fracturing_load_df.columns))
print("Final columns:")
print(fact_fracturing_load_df.columns.tolist())

Rows: 4806
Columns: 15
Final columns:
['id_base_fractura_adjiv', 'start_time_key', 'end_time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas']


#### 3.2 Validate Final Structure

In [251]:
print(f"Rows: {fact_fracturing_load_df.shape[0]}")

print(
    f"Unique fracturing IDs: "
    f"{fact_fracturing_load_df['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{fact_fracturing_load_df['id_base_fractura_adjiv'].duplicated().sum()}"
)

print(
    f"Columns: "
    f"{fact_fracturing_load_df.shape[1]}"
)

print("-" * 35)

print("Final columns:")
print(fact_fracturing_load_df.columns.tolist())

Rows: 4806
Unique fracturing IDs: 4806
Duplicated fracturing IDs: 0
Columns: 15
-----------------------------------
Final columns:
['id_base_fractura_adjiv', 'start_time_key', 'end_time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas']


Step 3.2.1. Validate start_time_key & end_time_key

In [252]:
print(
    f"Missing start_time_key: "
    f"{fact_fracturing_load_df['start_time_key'].isna().sum()}"
)

print(
    f"Missing end_time_key: "
    f"{fact_fracturing_load_df['end_time_key'].isna().sum()}"
)

Missing start_time_key: 0
Missing end_time_key: 0


Step 3.2.2. Validate well_key

In [253]:
print(
    f"Missing well_key: "
    f"{fact_fracturing_load_df['well_key'].isna().sum()}"
)

Missing well_key: 0


Step 3.2.3. Validate geology_key

In [254]:
print(
    f"Missing geology_key: "
    f"{fact_fracturing_load_df['geology_key'].isna().sum()}"
)

Missing geology_key: 0


Step 3.2.4. Validate field_area_key

In [255]:
print(
    f"Missing field_area_key: "
    f"{fact_fracturing_load_df['field_area_key'].isna().sum()}"
)

Missing field_area_key: 4097


Step 3.2.5. Validate location_key

In [256]:
print(
    f"Missing location_key: "
    f"{fact_fracturing_load_df['location_key'].isna().sum()}"
)

Missing location_key: 969


Step 3.2.6. Validate company_key

In [257]:
print(
    f"Missing company_key: "
    f"{fact_fracturing_load_df['company_key'].isna().sum()}"
)

Missing company_key: 0


#### 3.3 Load into MySQL

3.3.1. Validate Load Dataset

In [258]:
print("DataFrame prepared for MySQL load:")
print(f"Rows: {len(fact_fracturing_load_df)}")
print(f"Columns: {len(fact_fracturing_load_df.columns)}")

print("\nColumns:")
print(fact_fracturing_load_df.columns.tolist())

DataFrame prepared for MySQL load:
Rows: 4806
Columns: 15

Columns:
['id_base_fractura_adjiv', 'start_time_key', 'end_time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas']


3.3.2. Prepare INSERT Statement

In [266]:
alter_table = """
ALTER TABLE fact_fracturing
    MODIFY field_area_key INT NULL,
    MODIFY location_key INT NULL;
"""
execute_query(alter_table)

[]

In [267]:
execute_query("DESCRIBE fact_fracturing;")

[('fracturing_key', 'int', 'NO', 'PRI', None, 'auto_increment'),
 ('id_base_fractura_adjiv', 'int', 'NO', 'UNI', None, ''),
 ('start_time_key', 'int', 'NO', 'MUL', None, ''),
 ('end_time_key', 'int', 'NO', 'MUL', None, ''),
 ('well_key', 'int', 'NO', 'MUL', None, ''),
 ('geology_key', 'int', 'NO', 'MUL', None, ''),
 ('field_area_key', 'int', 'YES', 'MUL', None, ''),
 ('location_key', 'int', 'YES', 'MUL', None, ''),
 ('company_key', 'int', 'NO', 'MUL', None, ''),
 ('arena_bombeada_nacional_tn', 'decimal(15,2)', 'YES', '', None, ''),
 ('arena_bombeada_importada_tn', 'decimal(15,2)', 'YES', '', None, ''),
 ('agua_inyectada_m3', 'decimal(15,2)', 'YES', '', None, ''),
 ('co2_inyectado_m3', 'decimal(15,2)', 'YES', '', None, ''),
 ('presion_maxima_psi', 'decimal(10,2)', 'YES', '', None, ''),
 ('potencia_equipos_fractura_hp', 'decimal(10,2)', 'YES', '', None, ''),
 ('cantidad_fracturas', 'smallint', 'YES', '', None, '')]

3.3.3. Load fact_fracturing into MySQL

In [269]:
insert_sql = """
INSERT INTO fact_fracturing(
    id_base_fractura_adjiv,
    start_time_key,
    end_time_key,
    well_key,
    geology_key,
    field_area_key,
    location_key,
    company_key,
    arena_bombeada_nacional_tn,
    arena_bombeada_importada_tn,
    agua_inyectada_m3,
    co2_inyectado_m3,
    presion_maxima_psi,
    potencia_equipos_fractura_hp,
    cantidad_fracturas
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s
)
"""

In [ ]:
load_to_table(insert_sql, fact_fracturing_load_df, "fact_fracturing")

✓ 4806 records inserted into 'fact_fracturing'.


3.3.4. Validate MySQL Row Count

In [273]:
print(
    "Rows in fact_fracturing:",
    get_row_count("fact_fracturing")
)

Rows in fact_fracturing: 4806


3.3.5. Validate Fracturing ID Uniqueness

In [ ]:
sql_count = """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id_base_fractura_adjiv) AS unique_fracturing_ids
    FROM fact_fracturing
"""

fracturing_id_validation = read_query(sql_count)

print(fracturing_id_validation)

   total_rows  unique_fracturing_ids
0        4806                   4806


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


3.3.6. Validate Duplicate Fracturing IDs

In [ ]:
sql_duplicate  = """
    SELECT
        id_base_fractura_adjiv,
        COUNT(*) AS occurrences
    FROM fact_fracturing
    GROUP BY id_base_fractura_adjiv
    HAVING COUNT(*) > 1
"""

duplicate_ids = read_query(sql_duplicate)

print("Duplicated fracturing IDs:")
print(duplicate_ids)

Duplicated fracturing IDs:
Empty DataFrame
Columns: [id_base_fractura_adjiv, occurrences]
Index: []


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


3.3.7. Validate NULL Foreign Keys

In [277]:
sql_fk_null = """
    SELECT
        SUM(start_time_key IS NULL) AS missing_start_time_key,
        SUM(end_time_key IS NULL) AS missing_end_time_key,
        SUM(well_key IS NULL) AS missing_well_key,
        SUM(geology_key IS NULL) AS missing_geology_key,
        SUM(field_area_key IS NULL) AS missing_field_area_key,
        SUM(location_key IS NULL) AS missing_location_key,
        SUM(company_key IS NULL) AS missing_company_key
    FROM fact_fracturing
"""

fk_null_validation = read_query(sql_fk_null)

print(fk_null_validation)

   missing_start_time_key  missing_end_time_key  missing_well_key  \
0                     0.0                   0.0               0.0   

   missing_geology_key  missing_field_area_key  missing_location_key  \
0                  0.0                  4097.0                 969.0   

   missing_company_key  
0                  0.0  


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


3.3.8. Validate Foreign Key Integrity

3.3.8.1. Field Area

In [278]:
sql_invalid_field = """
    SELECT COUNT(*) AS invalid_field_area_keys
    FROM fact_fracturing f
    LEFT JOIN dim_field_area d
        ON f.field_area_key = d.field_area_key
    WHERE f.field_area_key IS NOT NULL
      AND d.field_area_key IS NULL
"""

invalid_field_area_keys = read_query(sql_invalid_field)

print(invalid_field_area_keys)

   invalid_field_area_keys
0                        0


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


3.3.8.2. Location

In [279]:
sql_invalid_location = """
    SELECT COUNT(*) AS invalid_location_keys
    FROM fact_fracturing f
    LEFT JOIN dim_location d
        ON f.location_key = d.location_key
    WHERE f.location_key IS NOT NULL
      AND d.location_key IS NULL
"""

invalid_location_keys = read_query(sql_invalid_location)

print(invalid_location_keys)

   invalid_location_keys
0                      0


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


3.3.9. Validate Remaining Foreign Keys

In [280]:
sql_fk_validation = """
    SELECT
        SUM(
            f.start_time_key IS NOT NULL
            AND st.time_key IS NULL
        ) AS invalid_start_time_keys,

        SUM(
            f.end_time_key IS NOT NULL
            AND et.time_key IS NULL
        ) AS invalid_end_time_keys,

        SUM(
            f.well_key IS NOT NULL
            AND w.well_key IS NULL
        ) AS invalid_well_keys,

        SUM(
            f.geology_key IS NOT NULL
            AND g.geology_key IS NULL
        ) AS invalid_geology_keys,

        SUM(
            f.company_key IS NOT NULL
            AND c.company_key IS NULL
        ) AS invalid_company_keys

    FROM fact_fracturing f

    LEFT JOIN dim_time st
        ON f.start_time_key = st.time_key

    LEFT JOIN dim_time et
        ON f.end_time_key = et.time_key

    LEFT JOIN dim_well w
        ON f.well_key = w.well_key

    LEFT JOIN dim_geology g
        ON f.geology_key = g.geology_key

    LEFT JOIN dim_company c
        ON f.company_key = c.company_key
"""

invalid_fk_validation = read_query(sql_fk_validation)

print(invalid_fk_validation)

   invalid_start_time_keys  invalid_end_time_keys  invalid_well_keys  \
0                      0.0                    0.0                0.0   

   invalid_geology_keys  invalid_company_keys  
0                   0.0                   0.0  


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


3.3.10. Validate Measures

In [281]:
sql_measure_validation = """
    SELECT
        COUNT(*) AS total_rows,

        SUM(arena_bombeada_nacional_tn IS NULL)
            AS null_arena_nacional,

        SUM(arena_bombeada_importada_tn IS NULL)
            AS null_arena_importada,

        SUM(agua_inyectada_m3 IS NULL)
            AS null_agua_inyectada,

        SUM(co2_inyectado_m3 IS NULL)
            AS null_co2_inyectado,

        SUM(presion_maxima_psi IS NULL)
            AS null_presion_maxima,

        SUM(potencia_equipos_fractura_hp IS NULL)
            AS null_potencia_equipos,

        SUM(cantidad_fracturas IS NULL)
            AS null_cantidad_fracturas

    FROM fact_fracturing
"""

measure_validation = read_query(sql_measure_validation)

print(measure_validation)

   total_rows  null_arena_nacional  null_arena_importada  null_agua_inyectada  \
0        4806                  0.0                   0.0                  0.0   

   null_co2_inyectado  null_presion_maxima  null_potencia_equipos  \
0                 0.0                  0.0                   51.0   

   null_cantidad_fracturas  
0                      0.0  


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


3.4 Final Post-load validation

In [282]:
sql_final_validation = """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id_base_fractura_adjiv)
            AS unique_fracturing_ids,

        SUM(start_time_key IS NULL)
            AS missing_start_time_key,

        SUM(end_time_key IS NULL)
            AS missing_end_time_key,

        SUM(well_key IS NULL)
            AS missing_well_key,

        SUM(geology_key IS NULL)
            AS missing_geology_key,

        SUM(field_area_key IS NULL)
            AS missing_field_area_key,

        SUM(location_key IS NULL)
            AS missing_location_key,

        SUM(company_key IS NULL)
            AS missing_company_key

    FROM fact_fracturing
"""

final_fact_fracturing_validation = read_query(sql_final_validation)

print(final_fact_fracturing_validation)

   total_rows  unique_fracturing_ids  missing_start_time_key  \
0        4806                   4806                     0.0   

   missing_end_time_key  missing_well_key  missing_geology_key  \
0                   0.0               0.0                  0.0   

   missing_field_area_key  missing_location_key  missing_company_key  
0                  4097.0                 969.0                  0.0  


C:\Users\Admin\AppData\Local\Temp\ipykernel_12020\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


## Notebook conclusion

The fact table loading process was completed successfully.

The `fact_production` and `fact_fracturing` tables were populated in MySQL after preparing the transformed datasets and validating their structure before loading.

For `fact_fracturing`, a total of **4,806 records** were loaded, corresponding to **4,806 unique fracturing events**, with **no duplicated fracturing IDs**.

The foreign-key validations confirmed that all mandatory relationships were successfully resolved. No invalid references were found for `start_time_key`, `end_time_key`, `well_key`, `geology_key`, or `company_key`. The optional `field_area_key` and `location_key` relationships contain unresolved values that correspond to limitations already identified during the dimension-matching process. These values were therefore preserved as `NULL` rather than assigning unsupported relationships.

The measures in `fact_fracturing` were also successfully loaded. All measures were populated except for `potencia_equipos_fractura_hp`, which contains **51 NULL values** corresponding to missing information in the source data.

Overall, the fact-loading stage produced structurally consistent fact tables while preserving the limitations and missing information present in the original datasets. The resulting tables are ready to be used in the analytical stage of the project.
